In [1]:
import sys
import pandas as pd
import re
from pathlib import Path
import matplotlib.pyplot as plt
from io import StringIO
import seaborn as sns
from scipy.stats import chi2_contingency
import numpy as np
from statsmodels.stats.contingency_tables import Table2x2
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
print(sys.executable)

/Users/ahthini/Desktop/DissProject/my_env/bin/python


### **T2.3 - Feature engineering**

#### **T2.3.1 - Admission-level feature engineering**



**Load cleaned admission-level dataset**

The cleaned admission-level dataset from T2.2.1 is loaded as the starting point for feature engineering. This dataset contains one row per psychiatric admission and includes demographic, admission, prior utilisation, ICU, care pathway, DRG severity, diagnosis, medication, laboratory, and ICU vital-sign summary features.



In [2]:
#set output folder and load cleaned admission-level dataset from T2.2.1
output_path = Path('/Users/ahthini/Desktop/DissProject/outputs')
input_file = output_path / 't2_2_cleaned_dataset.csv'
df = pd.read_csv(input_file)
#convert timestamps to datetime format for chronology-based features
for col in ['admittime', 'dischtime']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print('Loaded cleaned admission-level dataset shape:')
print(df.shape)
print()
print('Duplicate subject_id + hadm_id rows:')
print(df.duplicated(subset=['subject_id', 'hadm_id']).sum())
print()
print('Readmission outcome distribution:')
print(df['readmitted_30d'].value_counts())

Loaded cleaned admission-level dataset shape:
(238491, 913)

Duplicate subject_id + hadm_id rows:
0

Readmission outcome distribution:
readmitted_30d
0    191133
1     47358
Name: count, dtype: int64


**Create previous psychiatric admission features**

Admissions were sorted chronologically within each patient. A cumulative count was used to represent the number of previous psychiatric admissions before the current admission, and the time since the previous psychiatric discharge was calculated where available.



In [3]:
#sort admissions chronologically for each patient
#the cohort contains psychiatric index admissions, so cumulative count represents previous psychiatric admissions in this cohort
df = df.sort_values(['subject_id', 'admittime']).copy()
df['previous_psych_admissions'] = df.groupby('subject_id').cumcount()
df['has_previous_psych_admission'] = (df['previous_psych_admissions'] > 0).astype(int)

#calculate time since previous psychiatric admission using the previous discharge time
df['previous_dischtime'] = df.groupby('subject_id')['dischtime'].shift(1)
df['days_since_previous_psych_admission'] = ((df['admittime'] - df['previous_dischtime']).dt.total_seconds() / (60 * 60 * 24))

df.loc[df['previous_psych_admissions'] == 0, 'days_since_previous_psych_admission'] = np.nan

#create filled version for modelling/statistical summaries
df['days_since_previous_psych_admission_filled'] = (df['days_since_previous_psych_admission'].fillna(0).clip(lower=0))
print('Previous psychiatric admissions summary:')
print(df['previous_psych_admissions'].describe())
print()
print('Negative raw days since previous psychiatric admission:')
print((df['days_since_previous_psych_admission'] < 0).sum())

Previous psychiatric admissions summary:
count    238491.000000
mean          3.490983
std           9.645282
min           0.000000
25%           0.000000
50%           1.000000
75%           3.000000
max         236.000000
Name: previous_psych_admissions, dtype: float64

Negative raw days since previous psychiatric admission:
33


**Create previous admission recency features**

Time since previous psychiatric admission was grouped into clinically interpretable intervals. Binary indicators were also created for recent psychiatric admission within 30, 90, and 365 days.



In [4]:
#categorise time since previous psychiatric admission for interpretation
df['time_since_previous_psych_category'] = pd.cut(df['days_since_previous_psych_admission_filled'],
    bins=[-0.01, 30, 90, 180, 365, np.inf],
    labels=['0-30 days', '31-90 days', '91-180 days', '181-365 days', '365+ days'],
    include_lowest=True)

df['time_since_previous_psych_category'] = df['time_since_previous_psych_category'].cat.add_categories(
    ['No previous psychiatric admission'])
df.loc[df['previous_psych_admissions'] == 0, 'time_since_previous_psych_category'] = 'No previous psychiatric admission'

#recent psychiatric admission indicators
df['recent_psych_admission_30d'] = ((df['previous_psych_admissions'] > 0)
    & (df['days_since_previous_psych_admission_filled'] <= 30)).astype(int)

df['recent_psych_admission_90d'] = ((df['previous_psych_admissions'] > 0)
    & (df['days_since_previous_psych_admission_filled'] <= 90)).astype(int)

df['recent_psych_admission_365d'] = ((df['previous_psych_admissions'] > 0)
    & (df['days_since_previous_psych_admission_filled'] <= 365)).astype(int)

print('Time since previous psychiatric admission category distribution:')
print(df['time_since_previous_psych_category'].value_counts(dropna=False))
print()
print('Recent psychiatric admission indicators:')
print(df[['recent_psych_admission_30d', 'recent_psych_admission_90d', 'recent_psych_admission_365d']].sum())

Time since previous psychiatric admission category distribution:
time_since_previous_psych_category
No previous psychiatric admission    107926
0-30 days                             47674
365+ days                             30039
31-90 days                            23135
91-180 days                           14902
181-365 days                          14815
Name: count, dtype: int64

Recent psychiatric admission indicators:
recent_psych_admission_30d      47674
recent_psych_admission_90d      70809
recent_psych_admission_365d    100526
dtype: int64


**Create prior hospital utilisation features**

The prior hospital utilisation counts created in T2.1 were converted into binary indicators to represent any previous hospital admission, previous non-psychiatric admission, and multiple previous admissions.



In [5]:
#prior hospital utilisation indicators from T2.1 extraction
df['has_previous_hospital_admission'] = (df['previous_total_admissions'] > 0).astype(int)
df['has_previous_nonpsych_admission'] = (df['previous_nonpsych_admissions'] > 0).astype(int)
df['has_multiple_previous_hospital_admissions'] = (df['previous_total_admissions'] >= 2).astype(int)

print('Prior hospital utilisation summary:')
print(df[['previous_total_admissions', 'previous_psych_admissions_from_all_hosp', 'previous_nonpsych_admissions']].describe())
print()
print('Prior hospital utilisation indicators:')
print(df[['has_previous_hospital_admission', 'has_previous_nonpsych_admission', 'has_multiple_previous_hospital_admissions']].sum())

Prior hospital utilisation summary:
       previous_total_admissions  previous_psych_admissions_from_all_hosp  \
count              238491.000000                            238491.000000   
mean                    4.547174                                 3.491448   
std                    10.563868                                 9.647174   
min                     0.000000                                 0.000000   
25%                     0.000000                                 0.000000   
50%                     1.000000                                 1.000000   
75%                     4.000000                                 3.000000   
max                   237.000000                               237.000000   

       previous_nonpsych_admissions  
count                 238491.000000  
mean                       1.055725  
std                        2.812220  
min                        0.000000  
25%                        0.000000  
50%                        0.000000  
75% 

**Create demographic and admission category features**

Continuous age and length of stay variables were retained for modelling, while grouped versions were created for interpretation and exploratory analysis. Admission timing features were also derived from the admission timestamp.



In [6]:
#create age groups for interpretation
df['age_group'] = pd.cut(df['anchor_age'], bins=[17, 29, 39, 49, 59, 69, 79, 91],
    labels=['18-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80+'], include_lowest=True)

#create hospital length of stay categories for interpretation
df['hospital_los_category'] = pd.cut(df['hospital_los_days'], bins=[-0.01, 1, 3, 7, np.inf],
    labels=['0-1 days', '1-3 days', '3-7 days', '7+ days'], include_lowest=True)

#create simple admission timing features
df['admission_year'] = df['admittime'].dt.year
df['admission_month'] = df['admittime'].dt.month
df['admission_dayofweek'] = df['admittime'].dt.dayofweek
df['weekend_admission'] = df['admission_dayofweek'].isin([5, 6]).astype(int)
df["admission_hour"] = df["admittime"].dt.hour
df["night_admission"] = ((df["admission_hour"] >= 22) | (df["admission_hour"] <= 6)).astype(int)

#create discharge timing features; these are useful for pathway interpretation in the full-record model
df["discharge_hour"] = df["dischtime"].dt.hour
df["discharge_month"] = df["dischtime"].dt.month
df["discharge_dayofweek"] = df["dischtime"].dt.dayofweek
df["weekend_discharge"] = df["discharge_dayofweek"].isin([5, 6]).astype(int)
df["friday_discharge"] = (df["discharge_dayofweek"] == 4).astype(int)
df["night_discharge"] = ((df["discharge_hour"] >= 22) | (df["discharge_hour"] <= 6)).astype(int)
season_map = {12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer", 9: "autumn", 10: "autumn", 11: "autumn"}
df["admission_season"] = df["admission_month"].map(season_map).fillna("Unknown")
df["discharge_season"] = df["discharge_month"].map(season_map).fillna("Unknown")

print("Admission hour summary:")
print(df["admission_hour"].describe())
print()
print("Night admission indicator:")
print(df["night_admission"].value_counts())
print("\nNight admission readmission rates:")
print((df.groupby("night_admission")["readmitted_30d"].mean() * 100).round(2))
print('Age group distribution:')
print(df['age_group'].value_counts(dropna=False).sort_index())
print()
print('Hospital LOS category distribution:')
print(df['hospital_los_category'].value_counts(dropna=False).sort_index())
print()
print('Weekend admission indicator:')
print(df['weekend_admission'].value_counts())
print()
print("\nWeekend admission readmission rates:")
print((df.groupby("weekend_admission")["readmitted_30d"].mean() * 100).round(2))

Admission hour summary:
count    238491.000000
mean         13.146798
std           7.455925
min           0.000000
25%           6.000000
50%          15.000000
75%          19.000000
max          23.000000
Name: admission_hour, dtype: float64

Night admission indicator:
night_admission
0    149384
1     89107
Name: count, dtype: int64

Night admission readmission rates:
night_admission
0    20.00
1    19.63
Name: readmitted_30d, dtype: float64
Age group distribution:
age_group
18-29    30700
30-39    25907
40-49    36256
50-59    48511
60-69    42536
70-79    28381
80+      26200
Name: count, dtype: int64

Hospital LOS category distribution:
hospital_los_category
0-1 days    53962
1-3 days    64843
3-7 days    68731
7+ days     50955
Name: count, dtype: int64

Weekend admission indicator:
weekend_admission
0    169471
1     69020
Name: count, dtype: int64


Weekend admission readmission rates:
weekend_admission
0    19.76
1    20.11
Name: readmitted_30d, dtype: float64


**Create grouped administrative pathway features**

Administrative variables such as discharge destination, admission source, and insurance contain many detailed categories. Grouped versions were created to improve interpretability in exploratory analysis and modelling. A separate discharge-to-death-or-hospice indicator was also created to support later sensitivity analysis, since these discharge outcomes may represent a different readmission-risk context.



In [7]:
#create grouped discharge destination feature
def group_discharge_location(value):
    value = str(value).upper()
    if value == "HOME":
        return "Home"
    elif "HOME HEALTH" in value:
        return "Home health care"
    elif any(term in value for term in ["SKILLED", "REHAB", "LONG TERM", "CHRONIC"]):
        return "Facility / rehab / long-term care"
    elif "PSYCH" in value:
        return "Psych facility"
    elif "HOSPICE" in value or "DIED" in value:
        return "Hospice / died"
    elif value in ["UNKNOWN", "NAN", ""] or "NOT AVAILABLE" in value:
        return "Unknown"
    else:
        return "Other"

df["discharge_location_grouped"] = df["discharge_location"].apply(group_discharge_location)

#create sensitivity-analysis indicator for death/hospice discharge destinations
df["discharged_to_death_or_hospice"] = (df["discharge_location_grouped"] == "Hospice / died").astype(int)

#create grouped admission source feature
def group_admission_location(value):
    value = str(value).upper()
    if "EMERGENCY" in value:
        return "Emergency room"
    elif "TRANSFER FROM HOSPITAL" in value:
        return "Transfer from hospital"
    elif "PSYCH" in value:
        return "Internal transfer involving psych"
    elif "PHYSICIAN" in value or "CLINIC" in value:
        return "Physician / clinic referral"
    elif "SKILLED" in value:
        return "Transfer from skilled nursing facility"
    elif value in ["UNKNOWN", "NAN", ""] or "NOT AVAILABLE" in value:
        return "Unknown"
    else:
        return "Other"

df["admission_location_grouped"] = df["admission_location"].apply(group_admission_location)

#create grouped insurance feature
def group_insurance(value):
    value = str(value).upper()
    if "MEDICARE" in value:
        return "Medicare"
    elif "MEDICAID" in value:
        return "Medicaid"
    elif "PRIVATE" in value:
        return "Private"
    elif value in ["UNKNOWN", "NAN", ""] or "NO CHARGE" in value:
        return "Other / unknown"
    else:
        return "Other / unknown"
df["insurance_grouped"] = df["insurance"].apply(group_insurance)

print("Grouped discharge location distribution:")
print(df["discharge_location_grouped"].value_counts(dropna=False))
print()
print("Discharged to death or hospice indicator:")
print(df["discharged_to_death_or_hospice"].value_counts())
print()
print("Grouped admission location distribution:")
print(df["admission_location_grouped"].value_counts(dropna=False))
print()
print("Grouped insurance distribution:")
print(df["insurance_grouped"].value_counts(dropna=False))

#more granular pathway flags keep discharge and admission source information interpretable for modelling.
discharge_location_clean = df["discharge_location"].fillna("").astype(str).str.upper()
admission_location_clean = df["admission_location"].fillna("").astype(str).str.upper()
df["discharged_home_with_services_flag"] = discharge_location_clean.str.contains("HOME HEALTH|HOME WITH", regex=True).astype(int)
df["discharged_home_without_services_flag"] = (discharge_location_clean.eq("HOME")).astype(int)
df["discharged_to_rehab_flag"] = discharge_location_clean.str.contains("REHAB", regex=True).astype(int)
df["discharged_to_snf_flag"] = discharge_location_clean.str.contains("SKILLED|SNF|NURSING", regex=True).astype(int)
df["discharged_to_acute_hospital_flag"] = discharge_location_clean.str.contains("ACUTE HOSPITAL|HOSPITAL", regex=True).astype(int)
df["discharged_to_residential_care_flag"] = discharge_location_clean.str.contains("LONG TERM|CHRONIC|RESIDENTIAL|ASSISTED", regex=True).astype(int)
df["discharged_to_psych_facility_or_transfer_flag"] = discharge_location_clean.str.contains("PSYCH", regex=True).astype(int)
df["admitted_from_ed_flag"] = admission_location_clean.str.contains("EMERGENCY", regex=True).astype(int)
df["admitted_from_clinic_referral_flag"] = admission_location_clean.str.contains("CLINIC|PHYSICIAN", regex=True).astype(int)
df["admitted_from_hospital_transfer_flag"] = admission_location_clean.str.contains("TRANSFER FROM HOSPITAL|ACUTE HOSPITAL", regex=True).astype(int)
df["admitted_from_facility_flag"] = admission_location_clean.str.contains("SKILLED|FACILITY|NURSING", regex=True).astype(int)
df["admitted_from_emergency_transfer_flag"] = ((df["admitted_from_ed_flag"] == 1) | (df["admitted_from_hospital_transfer_flag"] == 1)).astype(int)
df["admission_source_high_acuity_flag"] = ((df["admitted_from_ed_flag"] == 1) | (df["admitted_from_hospital_transfer_flag"] == 1) |
    (df.get("had_ed_transfer_record", 0) == 1)).astype(int)

pathway_refinement_cols = ["discharged_home_with_services_flag", "discharged_home_without_services_flag",
    "discharged_to_rehab_flag", "discharged_to_snf_flag", "discharged_to_acute_hospital_flag",
    "discharged_to_residential_care_flag", "discharged_to_psych_facility_or_transfer_flag",
    "admitted_from_ed_flag", "admitted_from_clinic_referral_flag", "admitted_from_hospital_transfer_flag",
    "admitted_from_facility_flag", "admitted_from_emergency_transfer_flag", "admission_source_high_acuity_flag"]


Grouped discharge location distribution:
discharge_location_grouped
Home                                 72493
Unknown                              69731
Home health care                     41194
Facility / rehab / long-term care    38534
Hospice / died                        8038
Other                                 5568
Psych facility                        2933
Name: count, dtype: int64

Discharged to death or hospice indicator:
discharged_to_death_or_hospice
0    230453
1      8038
Name: count, dtype: int64

Grouped admission location distribution:
admission_location_grouped
Emergency room                            113811
Physician / clinic referral                63572
Transfer from hospital                     26721
Other                                      24519
Internal transfer involving psych           5825
Transfer from skilled nursing facility      3830
Unknown                                      213
Name: count, dtype: int64

Grouped insurance distribution:
insurance_

#### **T2.3.2 - Clinical burden and severity features**



**Create psychiatric diagnosis burden features**

Psychiatric diagnosis subtype indicators were combined to describe diagnostic complexity. Diagnosis count variables were also grouped into clinically interpretable burden categories.



In [8]:
#define diagnosis flag columns
diagnosis_flag_cols = ['has_depression', 'has_bipolar_disorder', 'has_psychotic_disorder',
    'has_anxiety_ptsd', 'has_substance_use', 'has_cognitive_delirium', 'has_personality_disorder',
    'has_eating_disorder', 'has_neurodevelopmental_disorder', 'has_other_psych_diagnosis']

diagnosis_flag_cols = [col for col in diagnosis_flag_cols if col in df.columns]

#diagnosis burden features
df['psych_diagnosis_group_count'] = df[diagnosis_flag_cols].sum(axis=1)
df['has_multiple_psych_diagnosis_groups'] = (df['psych_diagnosis_group_count'] >= 2).astype(int)
df['psych_diagnosis_burden_category'] = pd.cut(df['num_psych_diagnoses'], bins=[-0.01, 1, 2, 4, np.inf],
    labels=['1 diagnosis', '2 diagnoses', '3-4 diagnoses', '5+ diagnoses'], include_lowest=True)

df['has_severe_mental_illness'] = ((df.get('has_psychotic_disorder', 0) == 1)
    | (df.get('has_bipolar_disorder', 0) == 1)).astype(int)

print('Diagnosis flag columns:')
print(diagnosis_flag_cols)
print()
print('Psychiatric diagnosis group count summary:')
print(df['psych_diagnosis_group_count'].describe())
print()
print('Psychiatric diagnosis burden category:')
print(df['psych_diagnosis_burden_category'].value_counts(dropna=False).sort_index())

Diagnosis flag columns:
['has_depression', 'has_bipolar_disorder', 'has_psychotic_disorder', 'has_anxiety_ptsd', 'has_substance_use', 'has_cognitive_delirium', 'has_personality_disorder', 'has_eating_disorder', 'has_neurodevelopmental_disorder', 'has_other_psych_diagnosis']

Psychiatric diagnosis group count summary:
count    238491.000000
mean          1.548088
std           0.796127
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max           8.000000
Name: psych_diagnosis_group_count, dtype: float64

Psychiatric diagnosis burden category:
psych_diagnosis_burden_category
1 diagnosis      132132
2 diagnoses       61393
3-4 diagnoses     36590
5+ diagnoses       8376
Name: count, dtype: int64


**Summarise diagnosis flag groups and readmission rates**

Each psychiatric diagnosis flag is summarised by admission count, cohort percentage, readmission count, and 30-day readmission rate. The printed output is also saved as a text file for reporting and audit purposes.



In [9]:
#summarise diagnosis flag groups and readmission rates
diagnosis_flag_labels = {'has_depression': 'Depression', 'has_bipolar_disorder': 'Bipolar disorder',
    'has_psychotic_disorder': 'Psychotic disorder', 'has_anxiety_ptsd': 'Anxiety / PTSD',
    'has_substance_use': 'Substance use', 'has_cognitive_delirium': 'Cognitive disorder / delirium',
    'has_personality_disorder': 'Personality disorder', 'has_eating_disorder': 'Eating disorder',
    'has_neurodevelopmental_disorder': 'Neurodevelopmental disorder',
    'has_other_psych_diagnosis': 'Other psychiatric diagnosis'}

diagnosis_flag_summary_rows = []
total_admissions = len(df)

for col in diagnosis_flag_cols:
    admissions_with_flag = df[df[col] == 1]
    admission_count = len(admissions_with_flag)
    readmission_count = int(admissions_with_flag['readmitted_30d'].sum())
    readmission_rate = admissions_with_flag['readmitted_30d'].mean() * 100 if admission_count > 0 else np.nan

    diagnosis_flag_summary_rows.append({'Diagnosis flag group': diagnosis_flag_labels.get(col, col),
        'Flag column': col, 'Admissions': admission_count,
        'Cohort percentage': (admission_count / total_admissions) * 100,
        'Readmitted admissions': readmission_count, 'Readmission rate (%)': readmission_rate})

diagnosis_flag_readmission_summary = pd.DataFrame(diagnosis_flag_summary_rows)
diagnosis_flag_readmission_summary = diagnosis_flag_readmission_summary.sort_values(
    by='Admissions', ascending=False).reset_index(drop=True)

diagnosis_flag_readmission_summary['Cohort percentage'] = diagnosis_flag_readmission_summary['Cohort percentage'].round(2)
diagnosis_flag_readmission_summary['Readmission rate (%)'] = diagnosis_flag_readmission_summary['Readmission rate (%)'].round(2)

diagnosis_flag_table = StringIO()
diagnosis_flag_table.write('Diagnosis flag group counts and 30-day readmission rates\n')
diagnosis_flag_table.write('=' * 65 + '\n')
diagnosis_flag_table.write('Total admissions in engineered dataset: ' + str(total_admissions) + '\n\n')
diagnosis_flag_table.write(diagnosis_flag_readmission_summary.to_string(index=False))
diagnosis_flag_table.write('\n')

summary_text = diagnosis_flag_table.getvalue()
print(summary_text)

Diagnosis flag group counts and 30-day readmission rates
Total admissions in engineered dataset: 238491

         Diagnosis flag group                     Flag column  Admissions  Cohort percentage  Readmitted admissions  Readmission rate (%)
                Substance use               has_substance_use      100434              42.11                  21283                 21.19
                   Depression                  has_depression       96745              40.57                  21132                 21.84
               Anxiety / PTSD                has_anxiety_ptsd       75894              31.82                  15612                 20.57
Cognitive disorder / delirium          has_cognitive_delirium       31285              13.12                   4483                 14.33
             Bipolar disorder            has_bipolar_disorder       19353               8.11                   5001                 25.84
  Other psychiatric diagnosis       has_other_psych_diagnosis      

**Create non-psychiatric diagnosis and DRG severity features**

Non-psychiatric diagnosis counts, total diagnosis burden, DRG severity, and DRG mortality were grouped to capture broader clinical complexity and severity outside psychiatric diagnosis alone.



In [10]:
#non-psychiatric and total diagnosis burden categories
df['nonpsych_diagnosis_burden_category'] = pd.cut(df['num_nonpsych_diagnoses'],
    bins=[-0.01, 0, 5, 10, np.inf], labels=['0', '1-5', '6-10', '11+'], include_lowest=True)

df['total_diagnosis_burden_category'] = pd.cut(df['num_total_diagnoses'],
    bins=[-0.01, 5, 10, 20, np.inf], labels=['0-5', '6-10', '11-20', '21+'], include_lowest=True)

#DRG categories; 0 indicates no APR-DRG record from T2.2
drg_label_map = {0: 'No DRG record', 1: 'Minor', 2: 'Moderate', 3: 'Major', 4: 'Extreme'}
df['drg_severity_category'] = df['drg_severity'].map(drg_label_map).fillna('No DRG record')
df['drg_mortality_category'] = df['drg_mortality'].map(drg_label_map).fillna('No DRG record')
df['high_drg_severity'] = (df['drg_severity'] >= 3).astype(int)
df['high_drg_mortality'] = (df['drg_mortality'] >= 3).astype(int)

print('Non-psychiatric diagnosis burden category:')
print(df['nonpsych_diagnosis_burden_category'].value_counts(dropna=False).sort_index())
print()
print('DRG severity category:')
print(df['drg_severity_category'].value_counts(dropna=False))
print()
print('DRG mortality category:')
print(df['drg_mortality_category'].value_counts(dropna=False))
print()
print("\nReadmission rate by DRG severity category:")
print((df.groupby("drg_severity_category", observed=False)["readmitted_30d"].mean() * 100).round(2))
print()
print("\nReadmission rate by DRG mortality category:")
print((df.groupby("drg_mortality_category", observed=False)["readmitted_30d"].mean() * 100).round(2))

Non-psychiatric diagnosis burden category:
nonpsych_diagnosis_burden_category
0         7481
1-5      55204
6-10     61533
11+     114273
Name: count, dtype: int64

DRG severity category:
drg_severity_category
No DRG record    79838
Major            57893
Moderate         53318
Extreme          26154
Minor            21288
Name: count, dtype: int64

DRG mortality category:
drg_mortality_category
No DRG record    79838
Minor            54140
Moderate         44944
Major            38870
Extreme          20699
Name: count, dtype: int64


Readmission rate by DRG severity category:
drg_severity_category
Extreme          17.71
Major            20.47
Minor            11.49
Moderate         16.33
No DRG record    24.71
Name: readmitted_30d, dtype: float64


Readmission rate by DRG mortality category:
drg_mortality_category
Extreme          15.02
Major            19.87
Minor            14.49
Moderate         19.92
No DRG record    24.71
Name: readmitted_30d, dtype: float64


**Create care pathway complexity features**

Service transfer, physical transfer, and procedure burden variables were converted into categorical and binary indicators to describe care pathway complexity and intervention burden during the admission.



In [11]:
#service transfer burden features
df["service_transfer_category"] = pd.cut(df["num_service_transfers"], bins=[-0.01, 0, 1, np.inf],
    labels=["0 transfers", "1 transfer", "2+ transfers"], include_lowest=True)

df["high_service_transfer_burden"] = (df["num_service_transfers"] >= 2).astype(int)
df["multiple_services"] = (df["num_unique_services"] >= 2).astype(int)

print("Service transfer category:")
print(df["service_transfer_category"].value_counts(dropna=False).sort_index())
print()
print("High service transfer burden:")
print(df["high_service_transfer_burden"].value_counts())
print()
print("Multiple services indicator:")
print(df["multiple_services"].value_counts())

#physical transfer and careunit complexity features
if "num_transfer_events" in df.columns:
    df["physical_transfer_category"] = pd.cut(df["num_transfer_events"], bins=[-0.01, 2, 4, np.inf],
        labels=["0-2 transfer events", "3-4 transfer events", "5+ transfer events"], include_lowest=True)

    df["high_physical_transfer_burden"] = (df["num_transfer_events"] >= 5).astype(int)
    df["multiple_careunits"] = (df["num_unique_careunits"] >= 3).astype(int)
    df["had_careunit_transfer"] = (df["num_careunit_transfers"] > 0).astype(int)

    print()
    print("Physical transfer category:")
    print(df["physical_transfer_category"].value_counts(dropna=False).sort_index())
    print()
    print("High physical transfer burden:")
    print(df["high_physical_transfer_burden"].value_counts())
    print()
    print("Multiple careunits indicator:")
    print(df["multiple_careunits"].value_counts())
    print()
    print("Careunit transfer indicator:")
    print(df["had_careunit_transfer"].value_counts())

#procedure burden features
if "num_procedures" in df.columns:
    df["procedure_burden_category"] = pd.cut(df["num_procedures"], bins=[-0.01, 0, 2, np.inf],
        labels=["0 procedures", "1-2 procedures", "3+ procedures"], include_lowest=True)

    df["high_procedure_burden"] = (df["num_procedures"] >= 3).astype(int)
    df["multiple_procedure_codes"] = (df["num_unique_procedure_codes"] >= 2).astype(int)

    print()
    print("Procedure burden category:")
    print(df["procedure_burden_category"].value_counts(dropna=False).sort_index())
    print()
    print("High procedure burden:")
    print(df["high_procedure_burden"].value_counts())
    print()
    print("Multiple procedure codes indicator:")
    print(df["multiple_procedure_codes"].value_counts())

print()
print("Readmission rate by service transfer category:")
print((df.groupby("service_transfer_category", observed=False)["readmitted_30d"].mean() * 100).round(2))

if "physical_transfer_category" in df.columns:
    print()
    print("Readmission rate by physical transfer category:")
    print((df.groupby("physical_transfer_category", observed=False)["readmitted_30d"].mean() * 100).round(2))

if "procedure_burden_category" in df.columns:
    print()
    print("Readmission rate by procedure burden category:")
    print((df.groupby("procedure_burden_category", observed=False)["readmitted_30d"].mean() * 100).round(2))

Service transfer category:
service_transfer_category
0 transfers     221640
1 transfer       13646
2+ transfers      3205
Name: count, dtype: int64

High service transfer burden:
high_service_transfer_burden
0    235286
1      3205
Name: count, dtype: int64

Multiple services indicator:
multiple_services
0    221717
1     16774
Name: count, dtype: int64

Physical transfer category:
physical_transfer_category
0-2 transfer events     22649
3-4 transfer events    166309
5+ transfer events      49533
Name: count, dtype: int64

High physical transfer burden:
high_physical_transfer_burden
0    188958
1     49533
Name: count, dtype: int64

Multiple careunits indicator:
multiple_careunits
0    178015
1     60476
Name: count, dtype: int64

Careunit transfer indicator:
had_careunit_transfer
0    126012
1    112479
Name: count, dtype: int64

Procedure burden category:
procedure_burden_category
0 procedures      129332
1-2 procedures     62966
3+ procedures      46193
Name: count, dtype: int64

Hi

#### **T2.3.3 - Medication and laboratory burden features**



**Create medication exposure and polypharmacy features**

Psychiatric medication class indicators were combined to describe psychiatric medication exposure during the index admission. Overall medication burden and polypharmacy indicators were also created using the number of unique drugs and prescription rows.



In [12]:
#define psychiatric medication class columns
medication_flag_cols = ['had_antidepressant', 'had_antipsychotic', 'had_mood_stabiliser',
    'had_benzodiazepine', 'had_stimulant', 'had_sedative_hypnotic']
medication_flag_cols = [col for col in medication_flag_cols if col in df.columns]

#psychiatric medication burden features
df['has_any_psych_medication'] = (df[medication_flag_cols].sum(axis=1) > 0).astype(int)
df['psych_medication_burden_category'] = pd.cut(df['num_psych_med_classes'], bins=[-0.01, 0, 1, 2, np.inf],
    labels=['0 classes', '1 class', '2 classes', '3+ classes'], include_lowest=True)

#overall medication burden and polypharmacy features
df['has_any_prescription'] = (df['num_prescription_rows'] > 0).astype(int)
df['polypharmacy_5plus'] = (df['num_unique_drugs'] >= 5).astype(int)
df['polypharmacy_10plus'] = (df['num_unique_drugs'] >= 10).astype(int)
df['unique_drug_burden_category'] = pd.cut(df['num_unique_drugs'], bins=[-0.01, 0, 4, 9, np.inf],
    labels=['0 drugs', '1-4 drugs', '5-9 drugs', '10+ drugs'], include_lowest=True)

#selected psychiatric medication combination indicators
df['antidepressant_antipsychotic_combination'] = ((df.get('had_antidepressant', 0) == 1)
    & (df.get('had_antipsychotic', 0) == 1)).astype(int)

df['mood_stabiliser_antipsychotic_combination'] = ((df.get('had_mood_stabiliser', 0) == 1)
    & (df.get('had_antipsychotic', 0) == 1)).astype(int)

for upstream_col in ['antipsychotic_polypharmacy_2plus', 'antipsychotic_polypharmacy_3plus',
        'antipsychotic_plus_benzodiazepine', 'antipsychotic_plus_mood_stabiliser',
        'had_long_acting_injectable_antipsychotic']:
    if upstream_col in df.columns:
        df[upstream_col] = df[upstream_col].fillna(0).astype(int)

print('Medication flag columns:')
print(medication_flag_cols)
print()
print('Psychiatric medication burden category:')
print(df['psych_medication_burden_category'].value_counts(dropna=False).sort_index())
print()
print('Unique drug burden category:')
print(df['unique_drug_burden_category'].value_counts(dropna=False).sort_index())

Medication flag columns:
['had_antidepressant', 'had_antipsychotic', 'had_mood_stabiliser', 'had_benzodiazepine', 'had_stimulant', 'had_sedative_hypnotic']

Psychiatric medication burden category:
psych_medication_burden_category
0 classes     79739
1 class       64385
2 classes     55982
3+ classes    38385
Name: count, dtype: int64

Unique drug burden category:
unique_drug_burden_category
0 drugs       44585
1-4 drugs      1909
5-9 drugs     14058
10+ drugs    177939
Name: count, dtype: int64


**Create laboratory measurement and abnormality burden features**

Laboratory measurement and abnormality indicators were combined to describe biochemical testing intensity and abnormality burden across the selected laboratory panel.



In [13]:
#define laboratory feature groups
lab_measured_cols = [col for col in df.columns if col.endswith('_lab_measured')]
lab_abnormal_cols = [col for col in df.columns if col.endswith('_lab_abnormal')]
lab_missing_cols = [col for col in df.columns if '_lab_' in col and col.endswith('_missing')]

#laboratory measurement and abnormality burden
df['num_labs_measured'] = df[lab_measured_cols].sum(axis=1)
df['num_abnormal_labs'] = df[lab_abnormal_cols].sum(axis=1)
df['any_abnormal_lab'] = (df['num_abnormal_labs'] > 0).astype(int)
df['lab_abnormal_burden_category'] = pd.cut(df['num_abnormal_labs'], bins=[-0.01, 0, 1, 2, np.inf],
    labels=['0 abnormal labs', '1 abnormal lab', '2 abnormal labs', '3+ abnormal labs'], include_lowest=True)
df['num_lab_value_missing_indicators'] = df[lab_missing_cols].sum(axis=1) if lab_missing_cols else 0

print('Number of selected labs measured:')
print(df['num_labs_measured'].describe())
print()
print('Number of abnormal selected labs:')
print(df['num_abnormal_labs'].describe())
print()
print('Lab abnormality burden category:')
print(df['lab_abnormal_burden_category'].value_counts(dropna=False).sort_index())

Number of selected labs measured:
count    238491.000000
mean          5.828727
std           3.480394
min           0.000000
25%           0.000000
50%           8.000000
75%           8.000000
max           8.000000
Name: num_labs_measured, dtype: float64

Number of abnormal selected labs:
count    238491.000000
mean          2.843273
std           2.456179
min           0.000000
25%           0.000000
50%           3.000000
75%           5.000000
max           8.000000
Name: num_abnormal_labs, dtype: float64

Lab abnormality burden category:
lab_abnormal_burden_category
0 abnormal labs      68017
1 abnormal lab       19262
2 abnormal labs      26886
3+ abnormal labs    124326
Name: count, dtype: int64


**Create biochemical domain abnormality features**

Selected laboratory abnormality indicators were grouped into clinically interpretable biochemical domains, including electrolyte, renal, and haematology abnormality features.



In [14]:
#specific biochemical burden features
electrolyte_abnormal_cols = [col for col in ['sodium_lab_abnormal', 'potassium_lab_abnormal']
    if col in df.columns]

df['electrolyte_abnormality_count'] = df[electrolyte_abnormal_cols].sum(axis=1) if electrolyte_abnormal_cols else 0
df['any_electrolyte_abnormality'] = (df['electrolyte_abnormality_count'] > 0).astype(int)
df['renal_lab_abnormality'] = ((df.get('creatinine_lab_abnormal', 0) == 1)
    | (df.get('urea_nitrogen_lab_abnormal', 0) == 1)).astype(int)

df['hematology_lab_abnormality'] = ((df.get('hemoglobin_lab_abnormal', 0) == 1)
    | (df.get('wbc_lab_abnormal', 0) == 1) | (df.get('platelet_lab_abnormal', 0) == 1)).astype(int)

print('Electrolyte abnormality count:')
print(df['electrolyte_abnormality_count'].value_counts(dropna=False).sort_index())
print()
print('Renal lab abnormality:')
print(df['renal_lab_abnormality'].value_counts())
print()
print('Haematology lab abnormality:')
print(df['hematology_lab_abnormality'].value_counts())
print("\nReadmission rate by unique drug burden category:")
print((df.groupby("unique_drug_burden_category", observed=False)["readmitted_30d"].mean() * 100).round(2))
print()
print("\nReadmission rate by renal lab abnormality:")
print((df.groupby("renal_lab_abnormality")["readmitted_30d"].mean() * 100).round(2))
print()
print("\nReadmission rate by haematology lab abnormality:")
print((df.groupby("hematology_lab_abnormality")["readmitted_30d"].mean() * 100).round(2))

Electrolyte abnormality count:
electrolyte_abnormality_count
0    169112
1     46567
2     22812
Name: count, dtype: int64

Renal lab abnormality:
renal_lab_abnormality
0    141860
1     96631
Name: count, dtype: int64

Haematology lab abnormality:
hematology_lab_abnormality
1    156236
0     82255
Name: count, dtype: int64

Readmission rate by unique drug burden category:
unique_drug_burden_category
0 drugs      30.76
1-4 drugs    10.01
5-9 drugs    11.69
10+ drugs    17.88
Name: readmitted_30d, dtype: float64


Readmission rate by renal lab abnormality:
renal_lab_abnormality
0    20.54
1    18.86
Name: readmitted_30d, dtype: float64


Readmission rate by haematology lab abnormality:
hematology_lab_abnormality
0    22.54
1    18.44
Name: readmitted_30d, dtype: float64


#### **T2.3.4 - Physiological and ICU utilisation features**



**Create ICU utilisation features**

ICU utilisation features were expanded into interpretable categories describing ICU exposure, multiple ICU stays, and prolonged ICU length of stay.



In [15]:
#ICU utilisation categories
df['icu_utilisation_category'] = pd.cut(df['icu_stay_count'], bins=[-0.1, 0, 1, np.inf],
    labels=['No ICU', 'Single ICU stay', 'Multiple ICU stays'], include_lowest=True)
df['multiple_icu_stays'] = (df['icu_stay_count'] >= 2).astype(int)
df['prolonged_icu_stay_3plus_days'] = (df['total_icu_los_days'] >= 3).astype(int)
df['prolonged_icu_stay_7plus_days'] = (df['total_icu_los_days'] >= 7).astype(int)

print('ICU utilisation category distribution:')
print(df['icu_utilisation_category'].value_counts(dropna=False))
print()
print('Prolonged ICU stay indicators:')
print(df[['prolonged_icu_stay_3plus_days', 'prolonged_icu_stay_7plus_days']].sum())

ICU utilisation category distribution:
icu_utilisation_category
No ICU                198005
Single ICU stay        36499
Multiple ICU stays      3987
Name: count, dtype: int64

Prolonged ICU stay indicators:
prolonged_icu_stay_3plus_days    15332
prolonged_icu_stay_7plus_days     6295
dtype: int64


**Create vital sign measurement burden features**

Vital-sign measurement indicators were combined to describe how many selected ICU vital sign types were available for each admission. Missingness indicators from T2.2 were also summarised to retain information about measurement availability.



In [16]:
#define vital sign measurement and missingness groups
vital_measured_cols = [col for col in df.columns if col.endswith('_vital_measured')]
vital_missing_cols = [col for col in df.columns if '_vital_' in col and col.endswith('_missing')]
df['num_vital_types_measured'] = df[vital_measured_cols].sum(axis=1) if vital_measured_cols else 0
df['num_vital_value_missing_indicators'] = df[vital_missing_cols].sum(axis=1) if vital_missing_cols else 0

print('Vital measured columns:')
print(vital_measured_cols)
print()
print('Number of vital types measured:')
print(df['num_vital_types_measured'].describe())

Vital measured columns:
['heart_rate_vital_measured', 'mean_bp_arterial_vital_measured', 'mean_bp_noninvasive_vital_measured', 'respiratory_rate_vital_measured', 'spo2_vital_measured', 'systolic_bp_arterial_vital_measured', 'systolic_bp_noninvasive_vital_measured']

Number of vital types measured:
count    238491.000000
mean          0.967701
std           2.176994
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           7.000000
Name: num_vital_types_measured, dtype: float64


**Create physiological instability indicators**

Vital-sign summary features were converted into simple physiological instability indicators. These indicators describe tachycardia, tachypnoea, hypoxia, and hypotension using clinically interpretable thresholds.



In [17]:
#physiological instability indicators based only on admissions with measured vital signs
heart_rate_measured = df.get("heart_rate_vital_measured", 0) == 1
respiratory_rate_measured = df.get("respiratory_rate_vital_measured", 0) == 1
spo2_measured = df.get("spo2_vital_measured", 0) == 1

bp_measured = ((df.get("systolic_bp_noninvasive_vital_measured", 0) == 1)
    | (df.get("systolic_bp_arterial_vital_measured", 0) == 1)
    | (df.get("mean_bp_noninvasive_vital_measured", 0) == 1)
    | (df.get("mean_bp_arterial_vital_measured", 0) == 1))

df["tachycardia_indicator"] = (
    heart_rate_measured & (df["heart_rate_vital_max_value"] >= 100)).astype(int)

df["severe_tachycardia_indicator"] = (
    heart_rate_measured & (df["heart_rate_vital_max_value"] >= 120)).astype(int)

df["tachypnoea_indicator"] = (
    respiratory_rate_measured & (df["respiratory_rate_vital_max_value"] >= 22)).astype(int)

df["hypoxia_indicator"] = (
    spo2_measured & (df["spo2_vital_min_value"] < 92)).astype(int)

sbp_min_cols = [col for col in ["systolic_bp_noninvasive_vital_min_value", 
    "systolic_bp_arterial_vital_min_value"] if col in df.columns]

map_min_cols = [col for col in ["mean_bp_noninvasive_vital_min_value", "mean_bp_arterial_vital_min_value"]
    if col in df.columns]

df["minimum_systolic_bp"] = df[sbp_min_cols].min(axis=1) if sbp_min_cols else np.nan
df["minimum_mean_bp"] = df[map_min_cols].min(axis=1) if map_min_cols else np.nan
df["hypotension_indicator"] = (bp_measured & ((df["minimum_systolic_bp"] < 90)
        | (df["minimum_mean_bp"] < 65))).astype(int)

physiology_instability_cols = ["tachycardia_indicator", "severe_tachycardia_indicator",
    "tachypnoea_indicator", "hypoxia_indicator", "hypotension_indicator"]

df["physiological_instability_score"] = df[physiology_instability_cols].sum(axis=1)
df["physiological_instability_category"] = pd.cut(df["physiological_instability_score"],
    bins=[-0.01, 0, 1, 2, np.inf],
    labels=["0 indicators", "1 indicator", "2 indicators", "3+ indicators"], include_lowest=True)

print("Physiological instability indicators:")
print(df[physiology_instability_cols].sum().sort_values(ascending=False))
print()
print("Physiological instability category:")
print(df["physiological_instability_category"].value_counts(dropna=False).sort_index())
print("\nReadmission rate by ICU utilisation category:")
print((df.groupby("icu_utilisation_category", observed=False)["readmitted_30d"].mean() * 100).round(2))
print()
print("\nReadmission rate by physiological instability category:")
print((df.groupby("physiological_instability_category", observed=False)["readmitted_30d"].mean() * 100).round(2))
print()
print("\nPhysiological instability indicator counts:")
print(df[["tachycardia_indicator", "severe_tachycardia_indicator", "tachypnoea_indicator",
    "hypoxia_indicator", "hypotension_indicator"]].sum())

Physiological instability indicators:
hypotension_indicator           38955
tachypnoea_indicator            37638
tachycardia_indicator           27251
hypoxia_indicator               24149
severe_tachycardia_indicator    13604
dtype: int64

Physiological instability category:
physiological_instability_category
0 indicators     198111
1 indicator        1658
2 indicators       6940
3+ indicators     31782
Name: count, dtype: int64

Readmission rate by ICU utilisation category:
icu_utilisation_category
No ICU                20.90
Single ICU stay       14.69
Multiple ICU stays    15.25
Name: readmitted_30d, dtype: float64


Readmission rate by physiological instability category:
physiological_instability_category
0 indicators     20.90
1 indicator      14.41
2 indicators     14.35
3+ indicators    14.86
Name: readmitted_30d, dtype: float64


Physiological instability indicator counts:
tachycardia_indicator           27251
severe_tachycardia_indicator    13604
tachypnoea_indicator        

#### **T2.3.5 - Time-series feature preparation for sequential modelling**



**Document cleaned time-series dataset availability**

The cleaned long-format vital-sign file from T2.2.2 is retained separately from the admission-level dataset. The full file has already been cleaned and audited in T2.2.2, so this section avoids reprocessing the full 21-million-row file during routine feature engineering.



In [18]:
#record the cleaned time-series file and full-file audit values from T2.2.2
#these values document the sequential modelling resource without rereading the full file
timeseries_file = output_path / 't2_2_cleaned_vital_sign_timeseries.csv'
timeseries_audit_summary = pd.DataFrame({'check': ['Rows in cleaned time-series file',
        'Unique patients in cleaned time-series file', 'Unique admissions in cleaned time-series file',
        'Number of columns in cleaned time-series file'],
    'value': [21549225, 32185, 40466, 12]})

print('Cleaned time-series audit summary from T2.2.2:')
print(timeseries_audit_summary)
print()
print('Cleaned time-series file:')
print(timeseries_file)

Cleaned time-series audit summary from T2.2.2:
                                           check     value
0               Rows in cleaned time-series file  21549225
1    Unique patients in cleaned time-series file     32185
2  Unique admissions in cleaned time-series file     40466
3  Number of columns in cleaned time-series file        12

Cleaned time-series file:
/Users/ahthini/Desktop/DissProject/outputs/t2_2_cleaned_vital_sign_timeseries.csv


**Create hourly time-window sample**

A small sample of the cleaned time-series file is converted into hourly summary windows to demonstrate the structure needed for future LSTM or transformer-based modelling. This is a sample only; the full windowed sequence dataset should be generated later once the modelling strategy is finalised.



In [19]:
#create full observed hourly vital-sign window dataset in chunks
timeseries_file = output_path / "t2_2_cleaned_vital_sign_timeseries.csv"
hourly_window_file = output_path / "t2_3_hourly_vital_window_dataset.csv"
chunk_size = 1_000_000
hourly_window_chunks = []
total_timeseries_rows = 0

print("Starting full hourly time-window feature engineering...")
for chunk_number, chunk in enumerate(pd.read_csv(timeseries_file, chunksize=chunk_size)):
    total_timeseries_rows += len(chunk)

    #assign each observation to an integer hour since admission
    chunk["hour_window"] = np.floor(chunk["hours_since_admission"]).astype(int)

    #aggregate repeated observations within each admission-hour-vital group
    chunk_hourly = (chunk.groupby(["subject_id", "hadm_id", "readmitted_30d", "hour_window", "vital_name"],
            as_index=False)["valuenum"].mean())

    hourly_window_chunks.append(chunk_hourly)
    print("Processed chunk:", chunk_number, "| raw rows:", len(chunk), "| grouped rows:", len(chunk_hourly), flush=True)

#combine chunk-level grouped results
hourly_long = pd.concat(hourly_window_chunks, ignore_index=True)

#regroup after concatenating because the same admission-hour-vital can appear across chunk boundaries
hourly_long = (hourly_long.groupby(["subject_id", "hadm_id", "readmitted_30d", "hour_window", "vital_name"],
        as_index=False)["valuenum"].mean())

#pivot to one row per admission-hour
hourly_vital_windows = hourly_long.pivot_table(index=["subject_id", "hadm_id", "readmitted_30d", "hour_window"],
    columns="vital_name", values="valuenum", aggfunc="mean").reset_index()

hourly_vital_windows.columns.name = None

#save full observed hourly window dataset
hourly_vital_windows.to_csv(hourly_window_file, index=False)

print("Full hourly time-window feature engineering complete.")
print("Raw time-series rows processed:", total_timeseries_rows)
print("Hourly window dataset shape:", hourly_vital_windows.shape)
print("Unique patients:", hourly_vital_windows["subject_id"].nunique())
print("Unique admissions:", hourly_vital_windows["hadm_id"].nunique())
print("Hour window range:")
print(hourly_vital_windows["hour_window"].describe())
print()
print("Saved full hourly vital window dataset to:")
print(hourly_window_file)

Starting full hourly time-window feature engineering...
Processed chunk: 0 | raw rows: 1000000 | grouped rows: 899008
Processed chunk: 1 | raw rows: 1000000 | grouped rows: 896942
Processed chunk: 2 | raw rows: 1000000 | grouped rows: 899854
Processed chunk: 3 | raw rows: 1000000 | grouped rows: 901506
Processed chunk: 4 | raw rows: 1000000 | grouped rows: 899251
Processed chunk: 5 | raw rows: 1000000 | grouped rows: 903735
Processed chunk: 6 | raw rows: 1000000 | grouped rows: 896208
Processed chunk: 7 | raw rows: 1000000 | grouped rows: 897023
Processed chunk: 8 | raw rows: 1000000 | grouped rows: 899204
Processed chunk: 9 | raw rows: 1000000 | grouped rows: 897802
Processed chunk: 10 | raw rows: 1000000 | grouped rows: 899471
Processed chunk: 11 | raw rows: 1000000 | grouped rows: 902294
Processed chunk: 12 | raw rows: 1000000 | grouped rows: 901449
Processed chunk: 13 | raw rows: 1000000 | grouped rows: 901908
Processed chunk: 14 | raw rows: 1000000 | grouped rows: 897834
Processed

In [20]:
print("Hourly vital window dataset columns:")
print()
print("Missing values per hourly vital column:")
print(hourly_vital_windows.isna().sum().sort_values(ascending=False))
print()
print("Readmission distribution in hourly window dataset:")
print(hourly_vital_windows["readmitted_30d"].value_counts())
print()
print("Vital window non-missing counts:")
vital_window_cols = [col for col in hourly_vital_windows.columns
    if col not in ["subject_id", "hadm_id", "readmitted_30d", "hour_window"]]
print(hourly_vital_windows[vital_window_cols].notna().sum().sort_values(ascending=False))
print()
print("Hourly window preview:")
print(hourly_vital_windows.head())
print()
hourly_summary = pd.DataFrame({"metric": ["Rows", "Columns", "Unique patients",
        "Unique admissions", "Minimum hour window", "Maximum hour window"],
    "value": [hourly_vital_windows.shape[0], hourly_vital_windows.shape[1],
        hourly_vital_windows["subject_id"].nunique(), hourly_vital_windows["hadm_id"].nunique(),
        hourly_vital_windows["hour_window"].min(), hourly_vital_windows["hour_window"].max()]})
print()
print("\nHourly vital-sign time-window dataset summary:")
print(hourly_summary)

Hourly vital window dataset columns:

Missing values per hourly vital column:
systolic_bp_arterial       2741034
mean_bp_arterial           2740707
systolic_bp_noninvasive    1424954
mean_bp_noninvasive        1422976
spo2                        106588
respiratory_rate             77088
heart_rate                   24733
readmitted_30d                   0
hadm_id                          0
subject_id                       0
hour_window                      0
dtype: int64

Readmission distribution in hourly window dataset:
readmitted_30d
0    3442294
1     547709
Name: count, dtype: int64

Vital window non-missing counts:
heart_rate                 3965270
respiratory_rate           3912915
spo2                       3883415
mean_bp_noninvasive        2567027
systolic_bp_noninvasive    2565049
mean_bp_arterial           1249296
systolic_bp_arterial       1248969
dtype: int64

Hourly window preview:
   subject_id   hadm_id  readmitted_30d  hour_window  heart_rate  \
0    10000032  290790

#### **T2.3.6 - Flattened temporal vital-sign summaries**




**Create 72-hour temporal summary features**

The hourly vital-sign dataset is flattened into admission-level temporal summary features so that early dynamic physiology can be included in standard machine learning models. For each available vital sign, the mean, minimum, maximum, standard deviation, and number of observed hourly windows are calculated across the first 72 hours of admission.




In [21]:
#create admission-level 72-hour temporal summaries from the hourly vital-sign dataset
#these summaries flatten dynamic physiological information for standard machine learning models
hourly_window_file = output_path / "t2_3_hourly_vital_window_dataset.csv"
if "hourly_vital_windows" not in globals():
    hourly_vital_windows = pd.read_csv(hourly_window_file)

hour_window_limit = 72
id_cols = ["subject_id", "hadm_id"]
outcome_col = "readmitted_30d"
time_col = "hour_window"

vital_window_cols = [col for col in hourly_vital_windows.columns
    if col not in id_cols + [outcome_col, time_col]]

first_72h_vitals = hourly_vital_windows[(hourly_vital_windows[time_col] >= 0)
    & (hourly_vital_windows[time_col] < hour_window_limit)].copy()

summary_frames = []
temporal_summary_engineered_cols = []

for vital_col in vital_window_cols:
    vital_summary = (first_72h_vitals.groupby(id_cols)[vital_col]
        .agg(['mean', 'min', 'max', 'std', 'count']).reset_index())

    vital_summary = vital_summary.rename(columns={'mean': f'{vital_col}_72h_mean',
        'min': f'{vital_col}_72h_min', 'max': f'{vital_col}_72h_max',
        'std': f'{vital_col}_72h_std', 'count': f'{vital_col}_72h_observed_hours'})

    temporal_summary_engineered_cols.extend([f'{vital_col}_72h_mean', f'{vital_col}_72h_min', 
        f'{vital_col}_72h_max', f'{vital_col}_72h_std', f'{vital_col}_72h_observed_hours'])

    summary_frames.append(vital_summary)

temporal_vital_summaries = None
for summary in summary_frames:
    if temporal_vital_summaries is None:
        temporal_vital_summaries = summary
    else:
        temporal_vital_summaries = temporal_vital_summaries.merge(summary, on=id_cols, how='outer')

if temporal_vital_summaries is None:
    temporal_vital_summaries = df[id_cols].drop_duplicates().copy()

print('First 72-hour temporal summary feature table shape:')
print(temporal_vital_summaries.shape)
print()
print('Vital signs summarised:')
print(vital_window_cols)
print()
print('Number of 72-hour temporal summary features created:')
print(len(temporal_summary_engineered_cols))
print()
print('Temporal summary preview:')
print(temporal_vital_summaries.head())

First 72-hour temporal summary feature table shape:
(36443, 37)

Vital signs summarised:
['heart_rate', 'mean_bp_arterial', 'mean_bp_noninvasive', 'respiratory_rate', 'spo2', 'systolic_bp_arterial', 'systolic_bp_noninvasive']

Number of 72-hour temporal summary features created:
35

Temporal summary preview:
   subject_id   hadm_id  heart_rate_72h_mean  heart_rate_72h_min  \
0    10000032  29079034            97.000000                92.0   
1    10000690  25860671            84.935714                61.0   
2    10001217  24597018            93.296296                78.0   
3    10001217  27703517            79.729167                66.0   
4    10001725  25563031            79.277778                65.0   

   heart_rate_72h_max  heart_rate_72h_std  heart_rate_72h_observed_hours  \
0               105.0            4.092676                              9   
1               136.0           16.246721                             70   
2               106.0            7.363187            

**Merge temporal summaries into the admission-level dataset**

The 72-hour temporal summaries are merged into the existing admission-level engineered dataset. Admissions without hourly vital-sign observations are retained, with observed-hour counts set to zero and summary value columns imputed using the median among admissions with available hourly data.




In [22]:
#merge flattened temporal vital summaries into the admission-level dataset
rows_before_temporal_merge = df.shape[0]
cols_before_temporal_merge = df.shape[1]

df = df.merge(temporal_vital_summaries, on=id_cols, how='left')

observed_hour_cols = [col for col in temporal_summary_engineered_cols if col.endswith('_observed_hours')]
temporal_value_cols = [col for col in temporal_summary_engineered_cols if col not in observed_hour_cols]

#observed-hour counts are zero where an admission has no hourly vital observations for that vital sign
df[observed_hour_cols] = df[observed_hour_cols].fillna(0).astype(int)

#median-impute temporal summary value columns so the final admission-level modelling dataset has usable numeric inputs
for col in temporal_value_cols:
    if col.endswith('_72h_std'):
        df[col] = df[col].fillna(0)
    else:
        median_value = df[col].median(skipna=True)
        df[col] = df[col].fillna(median_value)

print('Shape before merging 72-hour temporal summaries:')
print((rows_before_temporal_merge, cols_before_temporal_merge))
print()
print('Shape after merging 72-hour temporal summaries:')
print(df.shape)
print()
print('Duplicate subject_id + hadm_id rows after temporal summary merge:')
print(df.duplicated(subset=id_cols).sum())
print()
print('Missing values in 72-hour temporal summary features after handling:')
print(df[temporal_summary_engineered_cols].isna().sum().sort_values(ascending=False).head(20))
print()
print('Observed-hour summary for 72-hour temporal features:')
print(df[observed_hour_cols].describe().T[['mean', 'min', 'max']])

Shape before merging 72-hour temporal summaries:
(238491, 1008)

Shape after merging 72-hour temporal summaries:
(238491, 1043)

Duplicate subject_id + hadm_id rows after temporal summary merge:
0

Missing values in 72-hour temporal summary features after handling:
heart_rate_72h_mean                       0
heart_rate_72h_min                        0
heart_rate_72h_max                        0
heart_rate_72h_std                        0
heart_rate_72h_observed_hours             0
mean_bp_arterial_72h_mean                 0
mean_bp_arterial_72h_min                  0
mean_bp_arterial_72h_max                  0
mean_bp_arterial_72h_std                  0
mean_bp_arterial_72h_observed_hours       0
mean_bp_noninvasive_72h_mean              0
mean_bp_noninvasive_72h_min               0
mean_bp_noninvasive_72h_max               0
mean_bp_noninvasive_72h_std               0
mean_bp_noninvasive_72h_observed_hours    0
respiratory_rate_72h_mean                 0
respiratory_rate_72h_min      

In [23]:
print("Admissions with any first 72-hour hourly vital data:")
print(temporal_vital_summaries["hadm_id"].nunique())
print("Percentage of admission-level dataset with first 72-hour hourly vital data:")
print(round(temporal_vital_summaries["hadm_id"].nunique() / df["hadm_id"].nunique() * 100, 2))

Admissions with any first 72-hour hourly vital data:
36443
Percentage of admission-level dataset with first 72-hour hourly vital data:
15.28


#### **T2.3.7 - Additional derived feature groups**

This section avoids reloading raw MIMIC-IV tables. Raw-table extraction features are created in T2.1, cleaned in T2.2, and then converted here into analysis-ready derived variables, missingness indicators, temporal burden summaries, and interaction features.

**Confirm incoming extracted feature groups**

These checks confirm that the additional raw-table extraction features from T2.1 are present after T2.2 cleaning. This keeps T2.3 focused on deriving modelling features from already-extracted columns rather than repeating raw data extraction.

In [24]:
#confirm additional extracted features created upstream in T2.1
utilisation_engineered_cols = [col for col in [
    "previous_total_admissions_30d", "previous_total_admissions_90d", "previous_total_admissions_365d",
    "previous_psych_admissions_30d", "previous_psych_admissions_90d", "previous_psych_admissions_365d",
    "previous_nonpsych_admissions_30d", "previous_nonpsych_admissions_90d", "previous_nonpsych_admissions_365d",
    "days_since_previous_hospital_admission", "days_since_previous_hospital_admission_filled",
    "high_utiliser_previous_year", "previous_psych_to_total_admission_ratio", "previous_icu_admissions",
    "has_previous_icu_admission", "days_since_previous_icu_admission", "days_since_previous_icu_admission_filled",
    "previous_2plus_psych_admissions", "previous_3plus_psych_admissions",
    "previous_2plus_total_admissions_365d", "previous_3plus_total_admissions_365d",
    "frequent_psych_admitter_flag", "prior_near_30d_readmission_flag", "previous_admission_within_27_33d_flag",
    "days_since_previous_admission_near_30d_flag", "previous_discharge_against_advice_flag",
    "previous_discharge_to_psych_facility_flag", "previous_discharge_to_facility_flag",
    "previous_short_los_flag", "previous_long_los_flag", "previous_admission_los_days",
    "previous_admission_los_days_filled", "mean_previous_los_days", "mean_previous_los_days_filled",
    "max_previous_los_days", "max_previous_los_days_filled", "previous_admission_had_late_orders",
    "previous_admission_had_safety_order", "previous_admission_had_discharge_planning_order",
    "previous_admission_order_activity_count", "prior_instability_score"] if col in df.columns]

ed_timing_engineered_cols = [col for col in ["has_ed_timing", "ed_length_of_stay_hours", "ed_to_admission_hours", "ed_to_ward_delay_hours"] if col in df.columns]

admission_social_los_engineered_cols = [col for col in ["discharged_against_advice", "not_married_flag",
    "single_or_divorced_or_widowed", "non_english_language_flag", "public_insurance_flag", "medicaid_flag",
    "medicare_flag", "insurance_missing_flag", "admitted_from_facility_flag", "admitted_from_hospital_transfer_flag",
    "emergency_room_admission_flag", "transfer_from_hospital_flag", "transfer_from_snf_flag",
    "internal_transfer_from_psych_flag", "discharged_home_flag", "discharged_to_facility_flag",
    "discharged_to_psych_facility_flag", "los_under_2_days", "los_under_7_days", "los_7_to_30_days",
    "los_30plus_days", "los_60plus_days", "discharged_home_with_services_flag",
    "discharged_home_without_services_flag", "discharged_to_rehab_flag", "discharged_to_snf_flag",
    "discharged_to_acute_hospital_flag", "discharged_to_residential_care_flag",
    "discharged_to_psych_facility_or_transfer_flag", "admitted_from_ed_flag",
    "admitted_from_clinic_referral_flag", "admitted_from_hospital_transfer_flag",
    "admitted_from_facility_flag", "admitted_from_emergency_transfer_flag",
    "admission_source_high_acuity_flag"] if col in df.columns]

comorbidity_psych_icd_engineered_cols = [col for col in [
    "has_self_harm_or_suicidal_ideation", "has_alcohol_related_disorder", "has_opioid_related_disorder",
    "has_stimulant_related_disorder", "has_diabetes", "has_chronic_kidney_disease", "has_copd",
    "has_heart_failure", "has_liver_disease", "has_cancer", "has_dementia",
    "physical_comorbidity_count", "charlson_comorbidity_index_simplified",
    "elixhauser_comorbidity_group_count_simplified"] if col in df.columns]

additional_icd_engineered_cols = [col for col in df.columns if col.startswith("elixhauser_")]
additional_icd_engineered_cols += [col for col in [
    "has_cannabis_related_disorder", "has_tobacco_or_nicotine_related_disorder",
    "has_sedative_hypnotic_related_disorder", "has_polysubstance_related_disorder",
    "finer_substance_use_category_count"] if col in df.columns]
additional_icd_engineered_cols = list(dict.fromkeys(additional_icd_engineered_cols))

infection_medication_engineered_cols = [col for col in [
    "had_antibiotic_exposure", "had_opioid_exposure", "had_steroid_exposure", "had_iv_prescription",
    "num_unique_antipsychotics", "antipsychotic_prescription_count", "antipsychotic_polypharmacy_2plus",
    "antipsychotic_polypharmacy_3plus", "antipsychotic_plus_benzodiazepine", "antipsychotic_plus_mood_stabiliser",
    "had_long_acting_injectable_antipsychotic", "medication_order_count_first_24h", "medication_order_count_first_72h", "microbiology_test_count",
    "positive_culture_flag", "blood_culture_positive", "urine_culture_positive", "respiratory_culture_positive",
    "num_positive_blood_culture", "num_positive_urine_culture", "num_positive_respiratory_culture",
    "num_distinct_specimen_types", "num_distinct_organisms", "polymicrobial_flag", "culture_positive_first_72h",
    "distinct_organism_count", "abnormal_microbiology_interpretation", "suspected_infection_flag"] if col in df.columns]

poe_order_engineered_cols = [col for col in [
    "num_total_poe_orders", "num_orders_first_6h", "num_orders_first_12h", "num_orders_first_24h",
    "num_orders_first_72h", "orders_24_to_72h", "orders_after_72h_count", "order_activity_slope_24_to_72h",
    "first_order_hours_from_admission", "last_order_hours_from_admission", "num_discontinued_orders",
    "num_psych_related_poe_orders", "num_consult_orders", "num_social_work_case_management_orders",
    "num_observation_safety_orders", "num_discharge_planning_orders", "num_followup_referral_orders",
    "had_sitter_or_constant_observation_order", "had_suicide_precaution_order", "had_elopement_precaution_order",
    "had_restraint_order", "had_psych_consult_order", "had_social_work_case_management_order",
    "had_followup_or_outpatient_referral_order", "had_discharge_planning_order",
    "num_unique_order_types", "num_unique_order_subtypes", "num_unique_poe_order_categories", "first_discharge_planning_order_hours",
    "last_discharge_planning_order_hours", "discharge_planning_first_72h_flag",
    "late_discharge_planning_after_72h_flag", "had_psych_followup_order",
    "had_outpatient_followup_order", "had_clinic_referral_order", "had_case_management_order",
    "num_followup_related_orders", "followup_order_first_72h_flag", "late_followup_order_flag",
    "had_sitter_order", "had_constant_observation_order", "had_behavioral_observation_order",
    "orders_last_24h_before_discharge", "orders_last_48h_before_discharge",
    "orders_last_12h_before_discharge", "orders_last_6h_before_discharge",
    "new_orders_last_24h_before_discharge_flag", "new_orders_last_12h_before_discharge_flag",
    "order_activity_last_24h_to_total_ratio", "order_activity_last_48h_to_total_ratio",
    "order_activity_last_12h_to_total_ratio", "order_activity_last_6h_to_total_ratio",
    "last_order_within_6h_of_discharge_flag", "last_order_within_12h_of_discharge_flag",
    "last_order_within_24h_of_discharge_flag", "last_order_close_to_discharge_hours", "order_activity_duration_hours",
    "safety_orders_first_24h", "safety_orders_first_72h", "safety_orders_after_72h",
    "safety_orders_last_24h_before_discharge", "safety_order_density_per_day", "late_safety_order_flag",
    "safety_order_near_discharge_flag", "self_harm_and_late_safety_order_flag",
    "care_team_touchpoints_after_72h", "care_team_touchpoints_last_24h_before_discharge",
    "care_team_touchpoint_density_per_day", "late_care_team_activity_flag",
    "high_care_team_touchpoints_near_discharge_flag",
    "had_ciWA_protocol_order", "had_alcohol_withdrawal_protocol_order",
    "num_code_status_orders", "had_code_status_change_order", "num_discharge_when_orders",
    "had_discharge_now_order", "num_transfer_to_orders", "num_level_of_urgency_orders",
    "urgent_order_flag", "routine_order_flag"] if col in df.columns]

pharmacy_intensity_engineered_cols = [col for col in [
    "num_pharmacy_records", "num_unique_pharmacy_medications", "num_iv_pharmacy_orders",
    "num_infusion_pharmacy_orders", "num_prn_pharmacy_orders", "num_scheduled_pharmacy_orders",
    "num_high_frequency_med_orders", "num_med_orders_with_duration", "num_discontinued_pharmacy_orders",
    "num_verified_pharmacy_orders", "num_oral_pharmacy_orders", "num_iv_route_pharmacy_orders",
    "num_intramuscular_pharmacy_orders", "num_subcutaneous_pharmacy_orders", "num_prn_psychotropic_orders",
    "num_scheduled_psychotropic_orders", "num_medication_route_types", "num_medication_frequency_types",
    "psychotropic_order_density_per_day", "had_prn_antipsychotic", "had_prn_benzodiazepine",
    "num_prn_benzodiazepine_orders", "num_prn_antipsychotic_orders", "had_naloxone_order",
    "had_methadone_or_buprenorphine", "had_withdrawal_treatment_order", "had_lockout_interval_order",
    "had_basal_rate_order", "had_sliding_scale_order", "had_one_hr_max_order",
    "num_unverified_pharmacy_orders", "num_floor_stock_pharmacy_orders",
    "num_patient_may_take_own_med_orders", "num_dosing_by_pharmacy_orders",
    "num_discharge_med_proc_type_orders", "pharmacy_logistics_complexity_score",
    "pharmacy_infusion_complexity_score",
    "new_antipsychotic_started_flag", "new_antidepressant_started_flag",
    "new_mood_stabiliser_started_flag", "new_benzodiazepine_started_flag",
    "psychotropic_medication_change_count", "psychotropic_started_after_72h_flag",
    "late_psychotropic_change_flag"] if col in df.columns]

care_team_engineered_cols = [col for col in [
    "num_unique_order_providers", "num_unique_enter_providers", "num_unique_caregivers_icu",
    "num_provider_order_changes", "num_care_team_touchpoints", "num_unique_order_providers_first_24h",
    "num_unique_order_providers_first_72h", "num_unique_services_during_admission",
    "service_change_count", "careunit_change_count", "service_or_careunit_change_flag",
    "high_care_team_touchpoints"] if col in df.columns]

emar_engineered_cols = [col for col in df.columns if col.startswith("emar_")]
emar_engineered_cols += [col for col in ["num_no_barcode_med_admins", "no_barcode_med_admin_rate",
    "num_nonformulary_visual_verification", "num_dose_due_records", "dose_due_to_given_gap_count",
    "num_admins_with_remaining_dose", "num_delayed_administered_events", "delayed_administered_rate",
    "num_started_emar_events", "num_stopped_emar_events", "emar_started_stopped_activity_last_24h",
    "emar_delayed_or_not_given_last_24h"] if col in df.columns]
emar_engineered_cols = list(dict.fromkeys(emar_engineered_cols))
procedure_specificity_engineered_cols = [col for col in [
    "had_mechanical_ventilation_procedure", "had_dialysis_procedure", "had_central_line_procedure",
    "had_ect_procedure", "had_restraint_related_procedure", "ect_procedure_count", "ect_within_first_72h",
    "ect_during_admission_flag"] if col in df.columns]
body_measure_engineered_cols = [col for col in [
    "latest_bmi_before_admission", "previous_bmi_before_admission", "bmi_change_recent",
    "latest_weight_kg_before_admission", "previous_weight_kg_before_admission", "weight_change_recent",
    "latest_height_cm_before_admission", "previous_height_cm_before_admission", "height_change_recent",
    "days_since_latest_body_measure", "obesity_flag", "underweight_flag", "missing_bmi_flag"] if col in df.columns]
hcpcs_engineered_cols = [col for col in [
    "num_hcpcs_events", "num_unique_hcpcs_codes", "had_hcpcs_ambulance_event",
    "had_hcpcs_emergency_event", "had_hcpcs_psych_therapy_event", "had_hcpcs_rehab_event",
    "had_hcpcs_transport_event", "num_hospital_observation_hcpcs_events", "num_emergency_hcpcs_events",
    "num_therapy_hcpcs_events", "num_rehab_hcpcs_events", "hcpcs_observation_to_total_ratio"] if col in df.columns]

lab_activity_engineered_cols = [col for col in ["num_lab_events_first_24h", "num_lab_events_first_72h",
    "num_lab_events_last_24h_before_discharge", "num_lab_events_last_48h_before_discharge",
    "num_abnormal_lab_flags", "num_abnormal_lab_flags_last_48h", "num_critical_or_priority_labs",
    "lab_activity_last_24h_flag", "lab_activity_near_discharge_flag", "abnormal_lab_burden_score"] if col in df.columns]

icu_event_engineered_cols = [col for col in ["num_icu_input_events", "num_icu_output_events", "num_icu_datetime_events",
    "num_icu_procedure_events", "num_chart_warnings", "had_chart_warning", "num_inputevent_rate_changes",
    "num_continuous_infusion_events", "num_paused_inputevents", "num_stopped_inputevents",
    "num_change_dose_rate_inputevents", "num_paused_procedureevents", "num_stopped_procedureevents",
    "icu_event_interruption_score", "icu_event_density_per_day"] if col in df.columns]

transfer_timing_engineered_cols = [col for col in [
    "medical_to_psych_service_transfer", "psych_to_medical_service_transfer", "same_first_last_service",
    "time_to_first_transfer_hours", "last_careunit_before_discharge", "psych_service_involved_anytime",
    "medicine_service_involved_anytime", "medicine_and_psych_services_both_flag"] if col in df.columns]

#extend incoming feature lists for the newest trajectory and process-quality fields
utilisation_trend_source_cols = [col for col in df.columns if (col.startswith("previous_ed_visits_") or col in [
    "admissions_per_30d", "admissions_per_90d", "admissions_per_365d", "admission_frequency_acceleration",
    "increasing_admission_frequency_flag", "decreasing_admission_frequency_flag", "frequent_ed_user_flag",
    "time_between_last_two_admissions_days", "time_between_last_two_admissions_days_filled",
    "mean_previous_admission_gap_days", "mean_previous_admission_gap_days_filled",
    "min_previous_admission_gap_days", "min_previous_admission_gap_days_filled",
    "std_previous_admission_gap_days", "std_previous_admission_gap_days_filled"])]
utilisation_engineered_cols = list(dict.fromkeys(utilisation_engineered_cols + utilisation_trend_source_cols))

pharmacy_intensity_engineered_cols = list(dict.fromkeys(globals().get("pharmacy_intensity_engineered_cols", []) + [col for col in [
    "psychotropic_orders_last_24h_before_discharge", "psychotropic_orders_last_48h_before_discharge",
    "prn_psychotropic_orders_last_24h_before_discharge", "num_floor_stock_pharmacy_orders",
    "num_patient_may_take_own_med_orders", "num_dosing_by_pharmacy_orders",
    "num_discharge_med_proc_type_orders", "pharmacy_logistics_complexity_score"] if col in df.columns]))

emar_engineered_cols = list(dict.fromkeys(globals().get("emar_engineered_cols", []) + [col for col in [
    "emar_refused_event_count", "emar_refusal_rate", "emar_psychotropic_refused_count",
    "emar_psychotropic_refusal_rate", "emar_psychotropic_not_given_count", "emar_nonpsychotropic_not_given_count",
    "emar_delayed_administration_count", "num_delayed_administered_events", "delayed_administered_rate",
    "num_started_emar_events", "num_stopped_emar_events", "emar_started_stopped_activity_last_24h",
    "emar_delayed_or_not_given_last_24h", "emar_total_delay_hours", "emar_mean_delay_hours"] if col in df.columns]))

upstream_feature_groups = { "Prior utilisation and prior ICU": utilisation_engineered_cols,
    "ED timing": ed_timing_engineered_cols,
    "ICD comorbidity and psychiatric subgroups": comorbidity_psych_icd_engineered_cols,
    "Additional ICD substance and Elixhauser flags": additional_icd_engineered_cols,
    "Admission/social/LOS proxies": admission_social_los_engineered_cols,
    "Medication/infection proxies": infection_medication_engineered_cols,
    "POE order acuity and timing": poe_order_engineered_cols,
    "Pharmacy medication intensity": pharmacy_intensity_engineered_cols,
    "Care-team complexity": care_team_engineered_cols,
    "EMAR medication administration": emar_engineered_cols,
    "Procedure specificity": procedure_specificity_engineered_cols,
    "HCPCS event burden": hcpcs_engineered_cols,
    "Laboratory ordering intensity": lab_activity_engineered_cols,
    "ICU event burden": icu_event_engineered_cols,
    "OMR body measures": body_measure_engineered_cols,
    "Service/transfer timing": transfer_timing_engineered_cols}

print("Incoming T2.1 extracted feature groups present in cleaned dataset:")
for group_name, cols in upstream_feature_groups.items():
    print(f"{group_name}: {len(cols)} columns")

Incoming T2.1 extracted feature groups present in cleaned dataset:
Prior utilisation and prior ICU: 63 columns
ED timing: 4 columns
ICD comorbidity and psychiatric subgroups: 14 columns
Additional ICD substance and Elixhauser flags: 36 columns
Admission/social/LOS proxies: 35 columns
Medication/infection proxies: 28 columns
POE order acuity and timing: 72 columns
Pharmacy medication intensity: 47 columns
Care-team complexity: 7 columns
EMAR medication administration: 41 columns
Procedure specificity: 8 columns
HCPCS event burden: 12 columns
Laboratory ordering intensity: 10 columns
ICU event burden: 15 columns
OMR body measures: 13 columns
Service/transfer timing: 8 columns


**Create analysis-ready sparse timing and body-measure columns**

Some upstream features are missing because the event was not observed or not applicable, such as no ED record, no prior ICU stay, or no recent OMR body measurement. This block creates missingness indicators and filled numeric copies so models can use both the value and whether it was observed.

In [25]:
#create missingness indicators and filled copies for sparse upstream numeric features
sparse_upstream_numeric_cols = [col for col in [
    "ed_length_of_stay_hours", "ed_to_admission_hours", "ed_to_ward_delay_hours",
    "days_since_previous_hospital_admission", "days_since_previous_icu_admission",
    "time_to_first_transfer_hours", "first_order_hours_from_admission", "last_order_hours_from_admission",
    "days_since_latest_body_measure", "latest_bmi_before_admission", "previous_bmi_before_admission", "bmi_change_recent",
    "latest_weight_kg_before_admission", "previous_weight_kg_before_admission", "weight_change_recent",
    "latest_height_cm_before_admission", "previous_height_cm_before_admission", "height_change_recent"] if col in df.columns]

zero_fill_sparse_cols = [col for col in [
    "ed_length_of_stay_hours", "ed_to_admission_hours", "ed_to_ward_delay_hours",
    "days_since_previous_hospital_admission", "days_since_previous_icu_admission",
    "time_to_first_transfer_hours", "first_order_hours_from_admission", "last_order_hours_from_admission"] if col in sparse_upstream_numeric_cols]
median_fill_sparse_cols = [col for col in sparse_upstream_numeric_cols if col not in zero_fill_sparse_cols]

sparse_upstream_missingness_cols = []
sparse_upstream_filled_cols = []

for col in sparse_upstream_numeric_cols:
    missing_col = f"{col}_missing"
    filled_col = f"{col}_filled"
    df[missing_col] = df[col].isna().astype(int)
    sparse_upstream_missingness_cols.append(missing_col)

    numeric_values = pd.to_numeric(df[col], errors="coerce")
    if col in zero_fill_sparse_cols:
        fill_value = 0
    else:
        fill_value = numeric_values.median()
        if pd.isna(fill_value):
            fill_value = 0
    df[filled_col] = numeric_values.fillna(fill_value)
    sparse_upstream_filled_cols.append(filled_col)

utilisation_engineered_cols = list(dict.fromkeys(utilisation_engineered_cols + [col for col in sparse_upstream_missingness_cols + sparse_upstream_filled_cols if "previous" in col]))
ed_timing_engineered_cols = list(dict.fromkeys(ed_timing_engineered_cols + [col for col in sparse_upstream_missingness_cols + sparse_upstream_filled_cols if col.startswith("ed_")]))
body_measure_engineered_cols = list(dict.fromkeys(body_measure_engineered_cols + [col for col in sparse_upstream_missingness_cols + sparse_upstream_filled_cols if any(term in col for term in ["bmi", "weight", "height"])]))
transfer_timing_engineered_cols = list(dict.fromkeys(transfer_timing_engineered_cols + [col for col in sparse_upstream_missingness_cols + sparse_upstream_filled_cols if "time_to_first_transfer" in col]))
poe_order_engineered_cols = list(dict.fromkeys(globals().get("poe_order_engineered_cols", []) + [col for col in sparse_upstream_missingness_cols + sparse_upstream_filled_cols if "order_hours_from_admission" in col]))

print("Sparse upstream numeric columns reviewed:")
print(sparse_upstream_numeric_cols)
print("Missingness indicator columns created:", len(sparse_upstream_missingness_cols))
print("Filled numeric companion columns created:", len(sparse_upstream_filled_cols))
print("Remaining missing values in new companion columns:")
print(df[sparse_upstream_missingness_cols + sparse_upstream_filled_cols].isna().sum().sort_values(ascending=False).head(20))

Sparse upstream numeric columns reviewed:
['ed_length_of_stay_hours', 'ed_to_admission_hours', 'ed_to_ward_delay_hours', 'days_since_previous_hospital_admission', 'days_since_previous_icu_admission', 'time_to_first_transfer_hours', 'first_order_hours_from_admission', 'last_order_hours_from_admission', 'days_since_latest_body_measure', 'latest_bmi_before_admission', 'previous_bmi_before_admission', 'bmi_change_recent', 'latest_weight_kg_before_admission', 'previous_weight_kg_before_admission', 'weight_change_recent', 'latest_height_cm_before_admission', 'previous_height_cm_before_admission', 'height_change_recent']
Missingness indicator columns created: 18
Filled numeric companion columns created: 18
Remaining missing values in new companion columns:
ed_length_of_stay_hours_missing                   0
ed_to_admission_hours_missing                     0
ed_to_ward_delay_hours_missing                    0
days_since_previous_hospital_admission_missing    0
days_since_previous_icu_admissio

/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/1189008698.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[missing_col] = df[col].isna().astype(int)
/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/1189008698.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[filled_col] = numeric_values.fillna(fill_value)
/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/1189008698.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame

**Trajectory, intensity and dynamic features**

This section turns the extracted admission-level signals into compact rates, proportions and short trend summaries. The aim is to capture whether care was concentrated early, continued late, or showed signs of unresolved medical/medication complexity near discharge.

In [26]:
#create compact rate, proportion, lab-dynamic and vital-dynamic features after all upstream blocks have been merged
rate_denominator_days = pd.to_numeric(df["hospital_los_days"], errors="coerce").clip(lower=1).fillna(1)

healthcare_intensity_engineered_cols = []
rate_sources = {
    "orders_per_hospital_day": "num_total_poe_orders",
    "new_orders_last_24h_per_hospital_day": "orders_last_24h_before_discharge",
    "medications_per_hospital_day": "num_pharmacy_records",
    "prescriptions_per_hospital_day": "num_prescription_rows",
    "procedures_per_hospital_day": "num_procedures",
    "care_team_touchpoints_per_hospital_day": "num_care_team_touchpoints",
    "discontinued_orders_per_hospital_day": "num_discontinued_orders",
    "discontinued_pharmacy_orders_per_hospital_day": "num_discontinued_pharmacy_orders"}
for new_col, source_col in rate_sources.items():
    if source_col in df.columns:
        df[new_col] = pd.to_numeric(df[source_col], errors="coerce").fillna(0) / rate_denominator_days
        healthcare_intensity_engineered_cols.append(new_col)

lab_count_cols = [col for col in df.columns if col.endswith("_lab_count")]
if lab_count_cols:
    df["total_selected_lab_event_count"] = df[lab_count_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
    df["lab_tests_per_hospital_day"] = df["total_selected_lab_event_count"] / rate_denominator_days
    healthcare_intensity_engineered_cols.extend(["total_selected_lab_event_count", "lab_tests_per_hospital_day"])

order_proportion_engineered_cols = []
if "num_total_poe_orders" in df.columns:
    order_total = pd.to_numeric(df["num_total_poe_orders"], errors="coerce").replace(0, np.nan)
else:
    order_total = pd.Series(np.nan, index=df.index)
order_proportion_sources = {
    "discontinued_order_proportion": "num_discontinued_orders",
    "psych_related_order_proportion": "num_psych_related_poe_orders",
    "consult_order_proportion": "num_consult_orders",
    "safety_order_proportion": "num_observation_safety_orders",
    "followup_order_proportion": "num_followup_referral_orders",
    "discharge_planning_order_proportion": "num_discharge_planning_orders",
    "orders_first_72h_to_total_ratio": "num_orders_first_72h",
    "orders_after_72h_to_total_ratio": "orders_after_72h_count"}
for new_col, source_col in order_proportion_sources.items():
    if source_col in df.columns:
        df[new_col] = (pd.to_numeric(df[source_col], errors="coerce").fillna(0) / order_total).replace([np.inf, -np.inf], 0).fillna(0)
        order_proportion_engineered_cols.append(new_col)

lab_vital_dynamics_engineered_cols = []
lab_prefixes = sorted({col.split("_lab_mean_value")[0] for col in df.columns if col.endswith("_lab_mean_value")})
for lab in lab_prefixes:
    mean_col = f"{lab}_lab_mean_value"
    std_col = f"{lab}_lab_std_value"
    first_col = f"{lab}_lab_first_value"
    last_col = f"{lab}_lab_last_value"
    change_col = f"{lab}_lab_value_change"
    if std_col in df.columns:
        new_col = f"{lab}_lab_coefficient_of_variation"
        df[new_col] = (pd.to_numeric(df[std_col], errors="coerce") / pd.to_numeric(df[mean_col], errors="coerce").abs().replace(0, np.nan)).replace([np.inf, -np.inf], 0).fillna(0)
        lab_vital_dynamics_engineered_cols.append(new_col)
    if change_col in df.columns:
        new_col = f"{lab}_lab_rate_of_change_per_day"
        df[new_col] = pd.to_numeric(df[change_col], errors="coerce").fillna(0) / rate_denominator_days
        lab_vital_dynamics_engineered_cols.append(new_col)
    if first_col in df.columns and last_col in df.columns:
        new_col = f"{lab}_lab_last_to_first_ratio"
        df[new_col] = (pd.to_numeric(df[last_col], errors="coerce") / pd.to_numeric(df[first_col], errors="coerce").replace(0, np.nan)).replace([np.inf, -np.inf], 0).fillna(0)
        lab_vital_dynamics_engineered_cols.append(new_col)

vital_prefixes = sorted({col.split("_vital_mean_value")[0] for col in df.columns if col.endswith("_vital_mean_value")})
vital_abnormal_flags = []
for vital in vital_prefixes:
    mean_col = f"{vital}_vital_mean_value"
    min_col = f"{vital}_vital_min_value"
    max_col = f"{vital}_vital_max_value"
    change_col = f"{vital}_vital_value_change"
    if min_col in df.columns and max_col in df.columns:
        range_col = f"{vital}_vital_value_range"
        df[range_col] = (pd.to_numeric(df[max_col], errors="coerce") - pd.to_numeric(df[min_col], errors="coerce")).fillna(0)
        lab_vital_dynamics_engineered_cols.append(range_col)
        if mean_col in df.columns:
            relative_col = f"{vital}_vital_relative_range"
            df[relative_col] = (df[range_col] / pd.to_numeric(df[mean_col], errors="coerce").abs().replace(0, np.nan)).replace([np.inf, -np.inf], 0).fillna(0)
            lab_vital_dynamics_engineered_cols.append(relative_col)
    if change_col in df.columns:
        rate_col = f"{vital}_vital_rate_of_change_per_day"
        df[rate_col] = pd.to_numeric(df[change_col], errors="coerce").fillna(0) / rate_denominator_days
        lab_vital_dynamics_engineered_cols.append(rate_col)

if {"heart_rate_vital_max_value", "heart_rate_vital_min_value"}.issubset(df.columns):
    df["heart_rate_abnormal_anytime_flag"] = ((df["heart_rate_vital_max_value"] > 110) | (df["heart_rate_vital_min_value"] < 50)).astype(int)
    vital_abnormal_flags.append("heart_rate_abnormal_anytime_flag")
if {"respiratory_rate_vital_max_value", "respiratory_rate_vital_min_value"}.issubset(df.columns):
    df["respiratory_rate_abnormal_anytime_flag"] = ((df["respiratory_rate_vital_max_value"] > 24) | (df["respiratory_rate_vital_min_value"] < 10)).astype(int)
    vital_abnormal_flags.append("respiratory_rate_abnormal_anytime_flag")
if "spo2_vital_min_value" in df.columns:
    df["spo2_low_anytime_flag"] = (df["spo2_vital_min_value"] < 92).astype(int)
    vital_abnormal_flags.append("spo2_low_anytime_flag")
for bp_prefix in ["systolic_bp_arterial", "systolic_bp_noninvasive"]:
    min_col = f"{bp_prefix}_vital_min_value"
    max_col = f"{bp_prefix}_vital_max_value"
    if min_col in df.columns and max_col in df.columns:
        flag_col = f"{bp_prefix}_abnormal_anytime_flag"
        df[flag_col] = ((df[max_col] > 180) | (df[min_col] < 90)).astype(int)
        vital_abnormal_flags.append(flag_col)
for bp_prefix in ["mean_bp_arterial", "mean_bp_noninvasive"]:
    min_col = f"{bp_prefix}_vital_min_value"
    max_col = f"{bp_prefix}_vital_max_value"
    if min_col in df.columns and max_col in df.columns:
        flag_col = f"{bp_prefix}_abnormal_anytime_flag"
        df[flag_col] = ((df[max_col] > 120) | (df[min_col] < 65)).astype(int)
        vital_abnormal_flags.append(flag_col)
if vital_abnormal_flags:
    df["vital_abnormality_burden_score"] = df[vital_abnormal_flags].sum(axis=1)
    lab_vital_dynamics_engineered_cols.extend(vital_abnormal_flags + ["vital_abnormality_burden_score"])

admission_discharge_timing_engineered_cols = [col for col in [
    "admission_year", "admission_month", "admission_dayofweek", "admission_hour", "weekend_admission",
    "night_admission", "admission_season", "discharge_hour", "discharge_month", "discharge_dayofweek",
    "weekend_discharge", "friday_discharge", "night_discharge", "discharge_season"] if col in df.columns]

print("Trajectory, intensity and dynamic feature groups created:")
for group_name, cols in {
        "Healthcare intensity rates": healthcare_intensity_engineered_cols,
        "Order proportions": order_proportion_engineered_cols,
        "Lab/vital dynamics": lab_vital_dynamics_engineered_cols,
        "Admission/discharge timing": admission_discharge_timing_engineered_cols}.items():
    print(group_name + ":", len(cols), "columns")

/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/465766916.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = pd.to_numeric(df[source_col], errors="coerce").fillna(0) / rate_denominator_days
/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/465766916.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = pd.to_numeric(df[source_col], errors="coerce").fillna(0) / rate_denominator_days
/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/465766916.py:16: Perfor

Trajectory, intensity and dynamic feature groups created:
Healthcare intensity rates: 10 columns
Order proportions: 8 columns
Lab/vital dynamics: 53 columns
Admission/discharge timing: 14 columns


/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/465766916.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = (pd.to_numeric(df[std_col], errors="coerce") / pd.to_numeric(df[mean_col], errors="coerce").abs().replace(0, np.nan)).replace([np.inf, -np.inf], 0).fillna(0)
/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/465766916.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = pd.to_numeric(df[change_col], errors="coerce").fillna(0) / rate_denominator_days
/var/folder

**Create first 72-hour vital burden features**

Hourly vital-sign windows are converted into abnormal-window and physiological-instability burden features. These features summarise how often instability occurred in the first 72 hours, rather than using only first, last, minimum, maximum, or mean vital-sign values.

In [27]:
#first-72-hour vital-sign burden features from the hourly vital-sign dataset
hourly_window_file = output_path / "t2_3_hourly_vital_window_dataset.csv"
temporal_burden_engineered_cols = []
if hourly_window_file.exists():
    hourly_vitals = pd.read_csv(hourly_window_file)
    first_72h_vitals = hourly_vitals[hourly_vitals["hour_window"].between(0, 72, inclusive="both")].copy()
    if not first_72h_vitals.empty:
        systolic_bp_combined = first_72h_vitals[["systolic_bp_arterial", "systolic_bp_noninvasive"]].min(axis=1, skipna=True)
        mean_bp_combined = first_72h_vitals[["mean_bp_arterial", "mean_bp_noninvasive"]].min(axis=1, skipna=True)
        first_72h_vitals["tachycardia_hour"] = (first_72h_vitals["heart_rate"] >= 100).fillna(False).astype(int)
        first_72h_vitals["severe_tachycardia_hour"] = (first_72h_vitals["heart_rate"] >= 120).fillna(False).astype(int)
        first_72h_vitals["tachypnoea_hour"] = (first_72h_vitals["respiratory_rate"] >= 22).fillna(False).astype(int)
        first_72h_vitals["hypoxia_hour"] = (first_72h_vitals["spo2"] < 92).fillna(False).astype(int)
        first_72h_vitals["hypotension_hour"] = ((systolic_bp_combined < 90) | (mean_bp_combined < 65)).fillna(False).astype(int)
        instability_components = ["tachycardia_hour", "tachypnoea_hour", "hypoxia_hour", "hypotension_hour"]
        first_72h_vitals["hourly_instability_score"] = first_72h_vitals[instability_components].sum(axis=1)
        first_72h_vitals["any_abnormal_vital_hour"] = (first_72h_vitals["hourly_instability_score"] > 0).astype(int)
        first_72h_vitals["hours_with_2plus_instability_indicators"] = (first_72h_vitals["hourly_instability_score"] >= 2).astype(int)
        first_72h_vitals["hours_with_3plus_instability_indicators"] = (first_72h_vitals["hourly_instability_score"] >= 3).astype(int)

        def longest_abnormal_run(values):
            longest = 0
            current = 0
            for value in values:
                if value == 1:
                    current += 1
                    longest = max(longest, current)
                else:
                    current = 0
            return longest

        vital_burden_features = first_72h_vitals.sort_values(id_cols + ["hour_window"]).groupby(id_cols, as_index=False).agg(
            vital_72h_hourly_rows=("hour_window", "count"),
            vital_72h_abnormal_window_count=("any_abnormal_vital_hour", "sum"),
            vital_72h_abnormal_window_percentage=("any_abnormal_vital_hour", "mean"),
            vital_72h_max_instability_score=("hourly_instability_score", "max"),
            vital_72h_mean_instability_score=("hourly_instability_score", "mean"),
            vital_72h_hours_with_2plus_instability_indicators=("hours_with_2plus_instability_indicators", "sum"),
            vital_72h_hours_with_3plus_instability_indicators=("hours_with_3plus_instability_indicators", "sum"),
            tachycardia_72h_hour_percentage=("tachycardia_hour", "mean"),
            severe_tachycardia_72h_hour_percentage=("severe_tachycardia_hour", "mean"),
            tachypnoea_72h_hour_percentage=("tachypnoea_hour", "mean"),
            hypoxia_72h_hour_percentage=("hypoxia_hour", "mean"),
            hypotension_72h_hour_percentage=("hypotension_hour", "mean"))
        longest_runs = first_72h_vitals.sort_values(id_cols + ["hour_window"]).groupby(id_cols)["any_abnormal_vital_hour"].apply(longest_abnormal_run).reset_index(name="vital_72h_longest_abnormal_run_hours")
        vital_burden_features = vital_burden_features.merge(longest_runs, on=id_cols, how="left")
        vital_burden_features["has_first_72h_vital_burden_data"] = 1
        df = df.drop(columns=[col for col in vital_burden_features.columns if col not in id_cols], errors="ignore")
        df = df.merge(vital_burden_features, on=id_cols, how="left")
        temporal_burden_engineered_cols = [col for col in vital_burden_features.columns if col not in id_cols]
        for col in temporal_burden_engineered_cols:
            df[col] = df[col].fillna(0)

**Create clinical interaction features**

Selected clinically meaningful interactions are created from existing admission history, psychiatric diagnosis, medication, ICU, laboratory, and age features. These columns let the models capture combinations such as recent psychiatric admission plus severe mental illness or polypharmacy plus older age.

In [28]:
#clinically motivated interaction terms
interaction_engineered_cols = []
df["age_65plus"] = (df["anchor_age"] >= 65).astype(int)

def numeric_feature(col_name):
    if col_name in df.columns:
        return pd.to_numeric(df[col_name], errors="coerce").fillna(0)
    return pd.Series(0, index=df.index)

interaction_definitions = {
    "recent_psych_admission_90d_x_severe_mental_illness": ("recent_psych_admission_90d", "has_severe_mental_illness"),
    "recent_psych_admission_90d_x_substance_use": ("recent_psych_admission_90d", "has_substance_use"),
    "prior_psych_365d_x_self_harm": ("previous_psych_admissions_365d", "has_self_harm_or_suicidal_ideation"),
    "prior_psych_90d_x_self_harm": ("previous_psych_admissions_90d", "has_self_harm_or_suicidal_ideation"),
    "prior_psych_365d_x_smi": ("previous_psych_admissions_365d", "has_severe_mental_illness"),
    "prior_psych_90d_x_substance_use": ("previous_psych_admissions_90d", "has_substance_use"),
    "self_harm_x_discharged_to_psych_facility": ("has_self_harm_or_suicidal_ideation", "discharged_to_psych_facility_flag"),
    "smi_x_antipsychotic_polypharmacy": ("has_severe_mental_illness", "antipsychotic_polypharmacy_2plus"),
    "severe_mental_illness_x_antipsychotic": ("has_severe_mental_illness", "had_antipsychotic"),
    "substance_use_x_benzodiazepine": ("has_substance_use", "had_benzodiazepine"),
    "frequent_psych_admitter_x_discharged_against_advice": ("frequent_psych_admitter_flag", "discharged_against_advice"),
    "polypharmacy_10plus_x_age_65plus": ("polypharmacy_10plus", "age_65plus"),
    "icu_utilisation_x_physiological_instability": ("had_icu_stay", "physiological_instability_score"),
    "lab_abnormality_burden_x_icu_use": ("num_abnormal_labs", "had_icu_stay"),
    "safety_order_and_self_harm_flag": ("num_observation_safety_orders", "has_self_harm_or_suicidal_ideation"),
    "prn_psychotropic_and_self_harm_flag": ("num_prn_psychotropic_orders", "has_self_harm_or_suicidal_ideation"),
    "substance_use_and_withdrawal_treatment_flag": ("has_substance_use", "had_withdrawal_treatment_order"),
    "psychosis_and_previous_psych_365d": ("has_psychotic_disorder", "previous_psych_admissions_365d"),
    "depression_and_previous_psych_365d": ("has_depression", "previous_psych_admissions_365d"),
    "late_order_activity_and_discharge_planning": ("late_order_activity_flag", "had_discharge_planning_order"),
    "frequent_psych_admitter_and_discharge_planning": ("frequent_psych_admitter_flag", "had_discharge_planning_order"),
    "ed_los_high_and_discharge_planning": ("high_ed_los_flag", "had_discharge_planning_order"),
    "care_team_touchpoints_high_and_followup_order": ("high_care_team_touchpoints", "had_followup_or_outpatient_referral_order")}

for new_col, (left_col, right_col) in interaction_definitions.items():
    if left_col in df.columns and right_col in df.columns:
        df[new_col] = numeric_feature(left_col) * numeric_feature(right_col)
        interaction_engineered_cols.append(new_col)

print("Clinical interaction features created:", len(interaction_engineered_cols))
print(interaction_engineered_cols)


Clinical interaction features created: 20
['recent_psych_admission_90d_x_severe_mental_illness', 'recent_psych_admission_90d_x_substance_use', 'prior_psych_365d_x_self_harm', 'prior_psych_90d_x_self_harm', 'prior_psych_365d_x_smi', 'prior_psych_90d_x_substance_use', 'self_harm_x_discharged_to_psych_facility', 'smi_x_antipsychotic_polypharmacy', 'severe_mental_illness_x_antipsychotic', 'substance_use_x_benzodiazepine', 'frequent_psych_admitter_x_discharged_against_advice', 'polypharmacy_10plus_x_age_65plus', 'icu_utilisation_x_physiological_instability', 'lab_abnormality_burden_x_icu_use', 'safety_order_and_self_harm_flag', 'prn_psychotropic_and_self_harm_flag', 'substance_use_and_withdrawal_treatment_flag', 'psychosis_and_previous_psych_365d', 'depression_and_previous_psych_365d', 'frequent_psych_admitter_and_discharge_planning']


/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/624388887.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["age_65plus"] = (df["anchor_age"] >= 65).astype(int)
/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/624388887.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = numeric_feature(left_col) * numeric_feature(right_col)
/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/624388887.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the

**Create SHAP-guided pathway, aftercare, and interaction features**

This section combines the extracted WP2.1/WP2.2 variables into compact clinical patterns. The goal is to help the model distinguish cases that can look superficially similar: late orders because someone is medically unstable, late orders because discharge planning is being finalised, low order activity despite psychiatric risk, and home discharge with or without follow-up support. These are derived from information available during the index admission and from previous admissions only.

### Late-order specificity and quiet-risk refinements

The rapid-trial SHAP/error review suggested that late care activity is useful, but a plain count can blur together routine discharge administration, psychiatric safety management, unresolved medical workup, and protective aftercare. This section derives compact scores and flags from the extracted POE, aftercare, medication, and psychiatric-history features so the model can separate meaningful late activity from safer planned discharge activity. It also adds "quiet risk" features for patients who have little late activity but still carry psychiatric or medical risk signals.

### Discharge continuity and mixed-risk phenotype

This section also derives compact planning and continuity indicators from existing medication, safety, aftercare, prior ED-only use, and medical-complexity variables. These features are intended to distinguish unmanaged risk from risk that appears to have a structured transition plan.

In [29]:
#derive compact order pathway, aftercare, pharmacy-density, and medical-complexity features
order_pattern_engineered_cols = []
pharmacy_route_engineered_cols = []
aftercare_engineered_cols = []
medical_complexity_interaction_cols = []
body_measure_trend_engineered_cols = []
pathway_refinement_engineered_cols = globals().get("pathway_refinement_cols", [])
care_fragmentation_engineered_cols = []
diagnosis_persistence_engineered_cols = []
hcpcs_model_engineered_cols = globals().get("hcpcs_engineered_cols", [])

# --- shared helpers and denominator columns ---
hospital_los_for_rates = numeric_feature("hospital_los_days").clip(lower=1)

#order-timing columns are kept numeric; if a specific order type is absent, timing is set to 0
zero_fill_order_timing_cols = ["first_discharge_planning_order_hours", "last_discharge_planning_order_hours",
    "last_order_close_to_discharge_hours", "order_activity_duration_hours"]
for col in zero_fill_order_timing_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# --- order trajectory and late-order activity features ---
if {"orders_24_to_72h", "num_orders_first_24h"}.issubset(df.columns):
    df["order_activity_ratio_24_to_72h"] = numeric_feature("orders_24_to_72h") / (numeric_feature("num_orders_first_24h") + 1)
    order_pattern_engineered_cols.append("order_activity_ratio_24_to_72h")
if "last_order_hours_from_admission" in df.columns:
    df["last_order_after_72h_flag"] = (numeric_feature("last_order_hours_from_admission") > 72).astype(int)
    df["late_order_activity_flag"] = (numeric_feature("last_order_hours_from_admission") > 48).astype(int)
    order_pattern_engineered_cols += ["last_order_after_72h_flag", "late_order_activity_flag"]
if {"last_order_hours_from_admission", "first_order_hours_from_admission"}.issubset(df.columns):
    df["order_activity_duration_hours"] = (numeric_feature("last_order_hours_from_admission") -
        numeric_feature("first_order_hours_from_admission")).clip(lower=0)
    order_pattern_engineered_cols.append("order_activity_duration_hours")
if "last_order_close_to_discharge_hours" in df.columns:
    df["last_order_close_to_discharge_hours_filled"] = numeric_feature("last_order_close_to_discharge_hours")
    order_pattern_engineered_cols.append("last_order_close_to_discharge_hours_filled")
if "num_total_poe_orders" in df.columns:
    df["orders_per_hospital_day"] = numeric_feature("num_total_poe_orders") / hospital_los_for_rates
    nonzero_orders = numeric_feature("num_total_poe_orders")
    high_order_threshold = nonzero_orders[nonzero_orders > 0].quantile(0.75) if (nonzero_orders > 0).any() else 0
    df["high_poe_order_activity"] = (nonzero_orders >= high_order_threshold).astype(int)
    order_pattern_engineered_cols += ["orders_per_hospital_day", "high_poe_order_activity"]
if "num_orders_first_72h" in df.columns:
    df["poe_order_density_first_72h"] = numeric_feature("num_orders_first_72h") / 3
    order_pattern_engineered_cols.append("poe_order_density_first_72h")
if "orders_after_72h_count" in df.columns:
    df["orders_after_72h_per_hospital_day"] = numeric_feature("orders_after_72h_count") / hospital_los_for_rates
    order_pattern_engineered_cols.append("orders_after_72h_per_hospital_day")
if {"late_order_activity_flag", "high_poe_order_activity"}.issubset(df.columns):
    df["late_order_activity_x_high_poe_orders"] = numeric_feature("late_order_activity_flag") * numeric_feature("high_poe_order_activity")
    df["high_order_activity_and_late_last_order"] = df["late_order_activity_x_high_poe_orders"]
    order_pattern_engineered_cols += ["late_order_activity_x_high_poe_orders", "high_order_activity_and_late_last_order"]
if {"ed_length_of_stay_hours", "high_poe_order_activity"}.issubset(df.columns):
    ed_los_threshold = numeric_feature("ed_length_of_stay_hours")[numeric_feature("ed_length_of_stay_hours") > 0].quantile(0.75)
    if pd.isna(ed_los_threshold):
        ed_los_threshold = 0
    df["high_ed_los_flag"] = (numeric_feature("ed_length_of_stay_hours") >= ed_los_threshold).astype(int)
    df["ed_los_high_x_high_order_activity"] = numeric_feature("high_ed_los_flag") * numeric_feature("high_poe_order_activity")
    order_pattern_engineered_cols += ["high_ed_los_flag", "ed_los_high_x_high_order_activity"]
if {"orders_last_12h_before_discharge", "has_self_harm_or_suicidal_ideation"}.issubset(df.columns):
    df["late_orders_without_self_harm_flag"] = ((numeric_feature("orders_last_12h_before_discharge") > 0) &
        (numeric_feature("has_self_harm_or_suicidal_ideation") == 0)).astype(int)
    df["late_orders_and_self_harm_flag"] = ((numeric_feature("orders_last_12h_before_discharge") > 0) &
        (numeric_feature("has_self_harm_or_suicidal_ideation") == 1)).astype(int)
    order_pattern_engineered_cols += ["late_orders_without_self_harm_flag", "late_orders_and_self_harm_flag"]
if {"orders_last_12h_before_discharge", "previous_psych_admissions_365d"}.issubset(df.columns):
    df["late_orders_without_prior_psych_history_flag"] = ((numeric_feature("orders_last_12h_before_discharge") > 0) &
        (numeric_feature("previous_psych_admissions_365d") == 0)).astype(int)
    order_pattern_engineered_cols.append("late_orders_without_prior_psych_history_flag")
if {"orders_last_12h_before_discharge", "discharged_to_psych_facility_flag"}.issubset(df.columns):
    df["late_orders_and_psych_facility_discharge_flag"] = ((numeric_feature("orders_last_12h_before_discharge") > 0) &
        (numeric_feature("discharged_to_psych_facility_flag") == 1)).astype(int)
    order_pattern_engineered_cols.append("late_orders_and_psych_facility_discharge_flag")
if {"late_psych_safety_orders_last_12h", "has_self_harm_or_suicidal_ideation"}.issubset(df.columns):
    df["late_safety_orders_and_self_harm_flag"] = ((numeric_feature("late_psych_safety_orders_last_12h") > 0) &
        (numeric_feature("has_self_harm_or_suicidal_ideation") == 1)).astype(int)
    df["safety_orders_without_self_harm_flag"] = ((numeric_feature("late_psych_safety_orders_last_12h") > 0) &
        (numeric_feature("has_self_harm_or_suicidal_ideation") == 0)).astype(int)
    order_pattern_engineered_cols += ["late_safety_orders_and_self_harm_flag", "safety_orders_without_self_harm_flag"]
if {"late_psych_safety_orders_last_12h", "previous_psych_admissions_365d"}.issubset(df.columns):
    df["safety_orders_without_prior_readmission_history_flag"] = ((numeric_feature("late_psych_safety_orders_last_12h") > 0) &
        (numeric_feature("previous_psych_admissions_365d") == 0)).astype(int)
    order_pattern_engineered_cols.append("safety_orders_without_prior_readmission_history_flag")
if {"late_lab_orders_last_12h", "late_psych_safety_orders_last_12h"}.issubset(df.columns):
    df["late_lab_orders_without_psych_safety_orders_flag"] = ((numeric_feature("late_lab_orders_last_12h") > 0) &
        (numeric_feature("late_psych_safety_orders_last_12h") == 0)).astype(int)
    order_pattern_engineered_cols.append("late_lab_orders_without_psych_safety_orders_flag")
if {"late_medication_orders_last_12h", "late_psychotropic_change_flag"}.issubset(df.columns):
    df["late_medication_orders_and_psychotropic_change_flag"] = ((numeric_feature("late_medication_orders_last_12h") > 0) &
        (numeric_feature("late_psychotropic_change_flag") == 1)).astype(int)
    order_pattern_engineered_cols.append("late_medication_orders_and_psychotropic_change_flag")


# --- late-order meaning and order-type share features ---
#late-order meaning features describe the type mix of care activity close to discharge
if "orders_last_12h_before_discharge" in df.columns:
    late_order_total_12h = numeric_feature("orders_last_12h_before_discharge").replace(0, np.nan)
    late_order_share_sources = {
        "late_medication_order_share_last_12h": "late_medication_orders_last_12h",
        "late_lab_order_share_last_12h": "late_lab_orders_last_12h",
        "late_imaging_order_share_last_12h": "late_imaging_orders_last_12h",
        "late_consult_order_share_last_12h": "late_consult_orders_last_12h",
        "late_discharge_admin_order_share_last_12h": "late_discharge_admin_orders_last_12h",
        "late_routine_care_order_share_last_12h": "late_routine_care_orders_last_12h",
        "late_psych_safety_order_share_last_12h": "late_psych_safety_orders_last_12h",
        "late_followup_aftercare_order_share_last_12h": "late_followup_or_aftercare_orders_last_12h"}
    for new_col, source_col in late_order_share_sources.items():
        if source_col in df.columns:
            df[new_col] = (numeric_feature(source_col) / late_order_total_12h).replace([np.inf, -np.inf], 0).fillna(0)
            order_pattern_engineered_cols.append(new_col)

    if {"late_lab_orders_last_12h", "late_imaging_orders_last_12h", "late_medication_orders_last_12h", "late_psych_safety_orders_last_12h"}.issubset(df.columns):
        late_medical_activity = (numeric_feature("late_lab_orders_last_12h") + numeric_feature("late_imaging_orders_last_12h") +
            numeric_feature("late_medication_orders_last_12h"))
        df["late_medical_to_psych_order_ratio_last_12h"] = (late_medical_activity /
            (numeric_feature("late_psych_safety_orders_last_12h") + 1)).replace([np.inf, -np.inf], 0).fillna(0)
        order_pattern_engineered_cols.append("late_medical_to_psych_order_ratio_last_12h")
    if "late_discharge_admin_orders_last_12h" in df.columns:
        df["late_admin_to_total_late_order_ratio"] = (numeric_feature("late_discharge_admin_orders_last_12h") /
            late_order_total_12h).replace([np.inf, -np.inf], 0).fillna(0)
        order_pattern_engineered_cols.append("late_admin_to_total_late_order_ratio")
    if "late_psych_safety_orders_last_12h" in df.columns:
        df["late_safety_to_total_late_order_ratio"] = (numeric_feature("late_psych_safety_orders_last_12h") /
            late_order_total_12h).replace([np.inf, -np.inf], 0).fillna(0)
        order_pattern_engineered_cols.append("late_safety_to_total_late_order_ratio")
    if "late_followup_or_aftercare_orders_last_12h" in df.columns:
        df["late_followup_to_total_late_order_ratio"] = (numeric_feature("late_followup_or_aftercare_orders_last_12h") /
            late_order_total_12h).replace([np.inf, -np.inf], 0).fillna(0)
        order_pattern_engineered_cols.append("late_followup_to_total_late_order_ratio")
    late_routine_admin = numeric_feature("late_discharge_admin_orders_last_12h") + numeric_feature("late_routine_care_orders_last_12h")
    df["late_nonroutine_order_share_last_12h"] = ((numeric_feature("orders_last_12h_before_discharge") - late_routine_admin).clip(lower=0) /
        late_order_total_12h).replace([np.inf, -np.inf], 0).fillna(0)
    order_pattern_engineered_cols.append("late_nonroutine_order_share_last_12h")

#near-discharge lab-workup summary features use existing lab-event and abnormality counts
if "late_lab_orders_last_6h" in df.columns and "orders_last_6h_before_discharge" in df.columns:
    df["late_lab_order_share_last_6h"] = (numeric_feature("late_lab_orders_last_6h") /
        numeric_feature("orders_last_6h_before_discharge").replace(0, np.nan)).replace([np.inf, -np.inf], 0).fillna(0)
    order_pattern_engineered_cols.append("late_lab_order_share_last_6h")
if {"num_abnormal_lab_flags_last_48h", "num_lab_events_last_24h_before_discharge"}.issubset(df.columns):
    df["abnormal_labs_last_24h_before_discharge"] = np.minimum(numeric_feature("num_abnormal_lab_flags_last_48h"),
        numeric_feature("num_lab_events_last_24h_before_discharge"))
    df["new_abnormal_lab_last_24h_flag"] = (df["abnormal_labs_last_24h_before_discharge"] > 0).astype(int)
    order_pattern_engineered_cols += ["abnormal_labs_last_24h_before_discharge", "new_abnormal_lab_last_24h_flag"]
if {"num_abnormal_lab_flags_last_48h", "late_lab_orders_last_12h"}.issubset(df.columns):
    df["abnormal_labs_last_12h_before_discharge"] = np.minimum(numeric_feature("num_abnormal_lab_flags_last_48h"),
        numeric_feature("late_lab_orders_last_12h"))
    order_pattern_engineered_cols.append("abnormal_labs_last_12h_before_discharge")


for col in ["had_followup_or_outpatient_referral_order", "had_discharge_planning_order", "num_followup_referral_orders",
        "num_discharge_planning_orders", "first_discharge_planning_order_hours", "last_discharge_planning_order_hours",
        "discharge_planning_first_72h_flag", "late_discharge_planning_after_72h_flag",
        "had_psych_followup_order", "had_outpatient_followup_order", "had_clinic_referral_order",
        "had_case_management_order", "num_followup_related_orders", "followup_order_first_72h_flag",
        "late_followup_order_flag", "had_sitter_order", "had_constant_observation_order",
        "had_behavioral_observation_order", "had_ciWA_protocol_order", "had_alcohol_withdrawal_protocol_order"]:
    if col in df.columns:
        aftercare_engineered_cols.append(col)
if {"had_followup_or_outpatient_referral_order", "had_discharge_planning_order"}.issubset(df.columns):
    df["had_followup_or_discharge_planning_order"] = ((numeric_feature("had_followup_or_outpatient_referral_order") == 1) |
        (numeric_feature("had_discharge_planning_order") == 1)).astype(int)
    aftercare_engineered_cols.append("had_followup_or_discharge_planning_order")
if {"num_discharge_planning_orders", "hospital_los_days"}.issubset(df.columns):
    df["discharge_planning_order_density_per_day"] = numeric_feature("num_discharge_planning_orders") / hospital_los_for_rates
    aftercare_engineered_cols.append("discharge_planning_order_density_per_day")
if {"had_discharge_planning_order", "had_psych_followup_order"}.issubset(df.columns):
    df["discharge_planning_and_psych_followup_flag"] = ((numeric_feature("had_discharge_planning_order") == 1) &
        (numeric_feature("had_psych_followup_order") == 1)).astype(int)
    aftercare_engineered_cols.append("discharge_planning_and_psych_followup_flag")
if {"had_discharge_planning_order", "had_social_work_case_management_order"}.issubset(df.columns):
    df["discharge_planning_and_social_work_flag"] = ((numeric_feature("had_discharge_planning_order") == 1) &
        (numeric_feature("had_social_work_case_management_order") == 1)).astype(int)
    aftercare_engineered_cols.append("discharge_planning_and_social_work_flag")
if {"discharged_home_flag", "had_followup_or_outpatient_referral_order"}.issubset(df.columns):
    df["home_discharge_without_followup_order_flag"] = ((numeric_feature("discharged_home_flag") == 1) &
        (numeric_feature("had_followup_or_outpatient_referral_order") == 0)).astype(int)
    aftercare_engineered_cols.append("home_discharge_without_followup_order_flag")
if {"discharged_home_flag", "had_discharge_planning_order"}.issubset(df.columns):
    df["home_discharge_without_discharge_planning_order_flag"] = ((numeric_feature("discharged_home_flag") == 1) &
        (numeric_feature("had_discharge_planning_order") == 0)).astype(int)
    aftercare_engineered_cols.append("home_discharge_without_discharge_planning_order_flag")
if {"discharged_home_flag", "orders_last_12h_before_discharge"}.issubset(df.columns):
    df["home_discharge_and_late_orders_flag"] = ((numeric_feature("discharged_home_flag") == 1) &
        (numeric_feature("orders_last_12h_before_discharge") > 0)).astype(int)
    aftercare_engineered_cols.append("home_discharge_and_late_orders_flag")
if {"discharged_to_facility_flag", "orders_last_12h_before_discharge"}.issubset(df.columns):
    df["facility_discharge_and_late_orders_flag"] = ((numeric_feature("discharged_to_facility_flag") == 1) &
        (numeric_feature("orders_last_12h_before_discharge") > 0)).astype(int)
    aftercare_engineered_cols.append("facility_discharge_and_late_orders_flag")
if {"discharged_to_psych_facility_flag", "has_self_harm_or_suicidal_ideation"}.issubset(df.columns):
    df["psych_facility_discharge_and_self_harm_flag"] = ((numeric_feature("discharged_to_psych_facility_flag") == 1) &
        (numeric_feature("has_self_harm_or_suicidal_ideation") == 1)).astype(int)
    aftercare_engineered_cols.append("psych_facility_discharge_and_self_harm_flag")
if {"discharged_to_psych_facility_flag", "late_psych_safety_orders_last_12h"}.issubset(df.columns):
    df["psych_facility_discharge_and_safety_orders_flag"] = ((numeric_feature("discharged_to_psych_facility_flag") == 1) &
        (numeric_feature("late_psych_safety_orders_last_12h") > 0)).astype(int)
    aftercare_engineered_cols.append("psych_facility_discharge_and_safety_orders_flag")


#safe-discharge buffer features help distinguish late activity with aftercare/supervision from unmanaged late activity
if {"orders_last_12h_before_discharge", "had_followup_or_outpatient_referral_order"}.issubset(df.columns):
    df["late_orders_but_followup_present_flag"] = ((numeric_feature("orders_last_12h_before_discharge") > 0) &
        (numeric_feature("had_followup_or_outpatient_referral_order") == 1)).astype(int)
    aftercare_engineered_cols.append("late_orders_but_followup_present_flag")
if {"orders_last_12h_before_discharge", "had_discharge_planning_order"}.issubset(df.columns):
    df["late_orders_but_discharge_planning_present_flag"] = ((numeric_feature("orders_last_12h_before_discharge") > 0) &
        (numeric_feature("had_discharge_planning_order") == 1)).astype(int)
    aftercare_engineered_cols.append("late_orders_but_discharge_planning_present_flag")
if {"orders_last_12h_before_discharge", "discharged_to_psych_facility_flag", "discharged_to_facility_flag"}.issubset(df.columns):
    df["late_orders_but_supervised_discharge_flag"] = ((numeric_feature("orders_last_12h_before_discharge") > 0) &
        ((numeric_feature("discharged_to_psych_facility_flag") == 1) | (numeric_feature("discharged_to_facility_flag") == 1))).astype(int)
    df["late_orders_but_psych_facility_discharge_flag"] = ((numeric_feature("orders_last_12h_before_discharge") > 0) &
        (numeric_feature("discharged_to_psych_facility_flag") == 1)).astype(int)
    df["late_orders_but_facility_discharge_flag"] = ((numeric_feature("orders_last_12h_before_discharge") > 0) &
        (numeric_feature("discharged_to_facility_flag") == 1)).astype(int)
    aftercare_engineered_cols += ["late_orders_but_supervised_discharge_flag", "late_orders_but_psych_facility_discharge_flag",
        "late_orders_but_facility_discharge_flag"]
if {"discharged_home_flag", "had_followup_or_outpatient_referral_order"}.issubset(df.columns):
    df["home_discharge_with_followup_order_flag"] = ((numeric_feature("discharged_home_flag") == 1) &
        (numeric_feature("had_followup_or_outpatient_referral_order") == 1)).astype(int)
    aftercare_engineered_cols.append("home_discharge_with_followup_order_flag")
if {"discharged_home_flag", "had_discharge_planning_order"}.issubset(df.columns):
    df["home_discharge_with_discharge_planning_order_flag"] = ((numeric_feature("discharged_home_flag") == 1) &
        (numeric_feature("had_discharge_planning_order") == 1)).astype(int)
    aftercare_engineered_cols.append("home_discharge_with_discharge_planning_order_flag")
if {"discharged_to_psych_facility_flag", "had_followup_or_outpatient_referral_order"}.issubset(df.columns):
    df["psych_facility_discharge_with_followup_order_flag"] = ((numeric_feature("discharged_to_psych_facility_flag") == 1) &
        (numeric_feature("had_followup_or_outpatient_referral_order") == 1)).astype(int)
    aftercare_engineered_cols.append("psych_facility_discharge_with_followup_order_flag")
if {"discharged_to_facility_flag", "had_discharge_planning_order"}.issubset(df.columns):
    df["facility_discharge_with_discharge_planning_flag"] = ((numeric_feature("discharged_to_facility_flag") == 1) &
        (numeric_feature("had_discharge_planning_order") == 1)).astype(int)
    aftercare_engineered_cols.append("facility_discharge_with_discharge_planning_flag")

#quiet-but-risky features flag admissions with little late order activity but important psychiatric risk context
if "orders_last_12h_before_discharge" in df.columns:
    df["low_late_order_activity_flag"] = (numeric_feature("orders_last_12h_before_discharge") == 0).astype(int)
    aftercare_engineered_cols.append("low_late_order_activity_flag")
    quiet_risk_pairs = {
        "low_order_activity_with_self_harm_flag": "has_self_harm_or_suicidal_ideation",
        "low_order_activity_with_smi_flag": "has_severe_mental_illness",
        "low_order_activity_with_psychosis_flag": "has_psychotic_disorder",
        "low_order_activity_with_recent_psych_admission_flag": "recent_psych_admission_90d",
        "low_order_activity_with_prior_psych_365d_flag": "previous_psych_admissions_365d",
        "no_late_orders_but_self_harm_flag": "has_self_harm_or_suicidal_ideation",
        "no_late_orders_but_recent_psych_admission_flag": "recent_psych_admission_90d",
        "no_late_orders_but_psychosis_flag": "has_psychotic_disorder"}
    for new_col, risk_col in quiet_risk_pairs.items():
        if risk_col in df.columns:
            df[new_col] = ((numeric_feature("orders_last_12h_before_discharge") == 0) &
                (numeric_feature(risk_col) > 0)).astype(int)
            aftercare_engineered_cols.append(new_col)
if {"discharged_home_flag", "orders_last_12h_before_discharge", "previous_psych_admissions_365d"}.issubset(df.columns):
    df["home_discharge_low_activity_prior_psych_flag"] = ((numeric_feature("discharged_home_flag") == 1) &
        (numeric_feature("orders_last_12h_before_discharge") == 0) & (numeric_feature("previous_psych_admissions_365d") > 0)).astype(int)
    aftercare_engineered_cols.append("home_discharge_low_activity_prior_psych_flag")

#psychiatric-acuity specificity features distinguish actively managed versus unmanaged psychiatric risk
if {"has_self_harm_or_suicidal_ideation", "num_observation_safety_orders"}.issubset(df.columns):
    df["self_harm_without_safety_order_flag"] = ((numeric_feature("has_self_harm_or_suicidal_ideation") == 1) &
        (numeric_feature("num_observation_safety_orders") == 0)).astype(int)
    df["self_harm_with_late_safety_order_flag"] = ((numeric_feature("has_self_harm_or_suicidal_ideation") == 1) &
        (numeric_feature("late_psych_safety_orders_last_12h") > 0)).astype(int)
    aftercare_engineered_cols += ["self_harm_without_safety_order_flag", "self_harm_with_late_safety_order_flag"]
if {"has_self_harm_or_suicidal_ideation", "had_followup_or_outpatient_referral_order"}.issubset(df.columns):
    df["self_harm_without_followup_order_flag"] = ((numeric_feature("has_self_harm_or_suicidal_ideation") == 1) &
        (numeric_feature("had_followup_or_outpatient_referral_order") == 0)).astype(int)
    aftercare_engineered_cols.append("self_harm_without_followup_order_flag")
if {"has_severe_mental_illness", "had_followup_or_outpatient_referral_order"}.issubset(df.columns):
    df["smi_without_followup_order_flag"] = ((numeric_feature("has_severe_mental_illness") == 1) &
        (numeric_feature("had_followup_or_outpatient_referral_order") == 0)).astype(int)
    aftercare_engineered_cols.append("smi_without_followup_order_flag")
if {"has_psychotic_disorder", "had_followup_or_outpatient_referral_order"}.issubset(df.columns):
    df["psychosis_without_followup_order_flag"] = ((numeric_feature("has_psychotic_disorder") == 1) &
        (numeric_feature("had_followup_or_outpatient_referral_order") == 0)).astype(int)
    aftercare_engineered_cols.append("psychosis_without_followup_order_flag")
if {"has_psychotic_disorder", "late_psych_safety_orders_last_12h"}.issubset(df.columns):
    df["psychosis_with_late_safety_order_flag"] = ((numeric_feature("has_psychotic_disorder") == 1) &
        (numeric_feature("late_psych_safety_orders_last_12h") > 0)).astype(int)
    aftercare_engineered_cols.append("psychosis_with_late_safety_order_flag")
if {"has_severe_mental_illness", "discharged_to_psych_facility_flag"}.issubset(df.columns):
    df["smi_with_psych_facility_discharge_flag"] = ((numeric_feature("has_severe_mental_illness") == 1) &
        (numeric_feature("discharged_to_psych_facility_flag") == 1)).astype(int)
    aftercare_engineered_cols.append("smi_with_psych_facility_discharge_flag")

#medical-workup near discharge features separate psychiatric-looking risk from unresolved medical activity
if {"late_lab_orders_last_12h", "had_followup_or_outpatient_referral_order"}.issubset(df.columns):
    df["late_lab_activity_without_followup_flag"] = ((numeric_feature("late_lab_orders_last_12h") > 0) &
        (numeric_feature("had_followup_or_outpatient_referral_order") == 0)).astype(int)
    aftercare_engineered_cols.append("late_lab_activity_without_followup_flag")
if {"late_lab_orders_last_12h", "discharged_to_facility_flag", "discharged_to_psych_facility_flag"}.issubset(df.columns):
    df["late_lab_activity_without_facility_discharge_flag"] = ((numeric_feature("late_lab_orders_last_12h") > 0) &
        (numeric_feature("discharged_to_facility_flag") == 0) & (numeric_feature("discharged_to_psych_facility_flag") == 0)).astype(int)
    aftercare_engineered_cols.append("late_lab_activity_without_facility_discharge_flag")
if {"late_lab_orders_last_12h", "num_abnormal_lab_flags_last_48h"}.issubset(df.columns):
    df["medical_workup_near_discharge_score"] = (numeric_feature("late_lab_orders_last_12h") +
        numeric_feature("num_abnormal_lab_flags_last_48h") + numeric_feature("lab_activity_near_discharge_flag"))
    aftercare_engineered_cols.append("medical_workup_near_discharge_score")

#late care-fragmentation features use existing care-team/order timing signals without storing raw staff identifiers
if {"num_provider_order_changes", "order_activity_last_24h_to_total_ratio"}.issubset(df.columns):
    df["provider_changes_last_24h_before_discharge"] = (numeric_feature("num_provider_order_changes") *
        numeric_feature("order_activity_last_24h_to_total_ratio"))
    care_fragmentation_engineered_cols.append("provider_changes_last_24h_before_discharge")
if {"care_team_touchpoints_last_24h_before_discharge", "orders_last_12h_before_discharge", "orders_last_24h_before_discharge"}.issubset(df.columns):
    df["care_team_touchpoints_last_12h_before_discharge"] = (numeric_feature("care_team_touchpoints_last_24h_before_discharge") *
        (numeric_feature("orders_last_12h_before_discharge") / numeric_feature("orders_last_24h_before_discharge").replace(0, np.nan))).replace([np.inf, -np.inf], 0).fillna(0)
    care_fragmentation_engineered_cols.append("care_team_touchpoints_last_12h_before_discharge")
if {"care_team_touchpoints_last_24h_before_discharge", "orders_last_6h_before_discharge", "orders_last_24h_before_discharge"}.issubset(df.columns):
    df["care_team_touchpoints_last_6h_before_discharge"] = (numeric_feature("care_team_touchpoints_last_24h_before_discharge") *
        (numeric_feature("orders_last_6h_before_discharge") / numeric_feature("orders_last_24h_before_discharge").replace(0, np.nan))).replace([np.inf, -np.inf], 0).fillna(0)
    care_fragmentation_engineered_cols.append("care_team_touchpoints_last_6h_before_discharge")
if {"care_team_touchpoints_last_24h_before_discharge", "num_care_team_touchpoints"}.issubset(df.columns):
    df["late_care_team_touchpoint_share"] = (numeric_feature("care_team_touchpoints_last_24h_before_discharge") /
        numeric_feature("num_care_team_touchpoints").replace(0, np.nan)).replace([np.inf, -np.inf], 0).fillna(0)
    care_fragmentation_engineered_cols.append("late_care_team_touchpoint_share")
if {"multiple_services", "orders_last_24h_before_discharge"}.issubset(df.columns):
    df["service_change_last_24h_before_discharge_flag"] = ((numeric_feature("multiple_services") > 0) &
        (numeric_feature("orders_last_24h_before_discharge") > 0)).astype(int)
    care_fragmentation_engineered_cols.append("service_change_last_24h_before_discharge_flag")
if {"multiple_careunits", "orders_last_24h_before_discharge"}.issubset(df.columns):
    df["careunit_change_last_24h_before_discharge_flag"] = ((numeric_feature("multiple_careunits") > 0) &
        (numeric_feature("orders_last_24h_before_discharge") > 0)).astype(int)
    care_fragmentation_engineered_cols.append("careunit_change_last_24h_before_discharge_flag")
late_fragmentation_components = [col for col in ["provider_changes_last_24h_before_discharge",
    "care_team_touchpoints_last_12h_before_discharge", "care_team_touchpoints_last_6h_before_discharge",
    "late_care_team_touchpoint_share", "service_change_last_24h_before_discharge_flag",
    "careunit_change_last_24h_before_discharge_flag"] if col in df.columns]
if late_fragmentation_components:
    df["late_care_fragmentation_score"] = df[late_fragmentation_components].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
    care_fragmentation_engineered_cols.append("late_care_fragmentation_score")


for col in ["num_oral_pharmacy_orders", "num_iv_route_pharmacy_orders", "num_intramuscular_pharmacy_orders",
        "num_subcutaneous_pharmacy_orders", "num_medication_route_types", "num_medication_frequency_types",
        "num_prn_psychotropic_orders", "num_scheduled_psychotropic_orders", "psychotropic_order_density_per_day"]:
    if col in df.columns:
        pharmacy_route_engineered_cols.append(col)
if {"num_prn_psychotropic_orders", "num_scheduled_psychotropic_orders"}.issubset(df.columns):
    df["num_psychotropic_pharmacy_orders"] = numeric_feature("num_prn_psychotropic_orders") + numeric_feature("num_scheduled_psychotropic_orders")
    df["psychotropic_pharmacy_orders_per_day"] = df["num_psychotropic_pharmacy_orders"] / hospital_los_for_rates
    pharmacy_route_engineered_cols += ["num_psychotropic_pharmacy_orders", "psychotropic_pharmacy_orders_per_day"]
for col in ["had_prn_antipsychotic", "had_prn_benzodiazepine", "num_prn_benzodiazepine_orders",
        "num_prn_antipsychotic_orders", "had_naloxone_order", "had_methadone_or_buprenorphine",
        "had_withdrawal_treatment_order", "psychotropic_medication_change_count",
        "psychotropic_started_after_72h_flag", "late_psychotropic_change_flag",
        "new_antipsychotic_started_flag", "new_antidepressant_started_flag",
        "new_mood_stabiliser_started_flag", "new_benzodiazepine_started_flag"]:
    if col in df.columns:
        pharmacy_route_engineered_cols.append(col)

if {"charlson_comorbidity_index_simplified", "elixhauser_comorbidity_group_count_simplified"}.issubset(df.columns):
    df["medical_complexity_score"] = (numeric_feature("charlson_comorbidity_index_simplified") +
        numeric_feature("elixhauser_comorbidity_group_count_simplified") + numeric_feature("had_icu_stay") +
        numeric_feature("high_procedure_burden"))
    df["older_medical_complexity_score"] = numeric_feature("age_65plus") * df["medical_complexity_score"]
    medical_complexity_interaction_cols += ["medical_complexity_score", "older_medical_complexity_score"]
for new_col, left_col, right_col in [
        ("age_65plus_x_charlson", "age_65plus", "charlson_comorbidity_index_simplified"),
        ("age_65plus_x_elixhauser_group_count", "age_65plus", "elixhauser_comorbidity_group_count_simplified"),
        ("age_65plus_x_discharge_to_facility", "age_65plus", "discharged_to_facility_flag"),
        ("age_65plus_x_hospital_los_7plus", "age_65plus", "los_7_to_30_days"),
        ("transfer_from_hospital_x_high_comorbidity", "transfer_from_hospital_flag", "medical_complexity_score"),
        ("infection_or_positive_culture_x_older_age", "age_65plus", "suspected_infection_flag")]:
    if left_col in df.columns and right_col in df.columns:
        df[new_col] = numeric_feature(left_col) * numeric_feature(right_col)
        medical_complexity_interaction_cols.append(new_col)
for new_col, source_col in [("num_unique_services_during_admission", "num_unique_services"),
        ("service_change_count", "num_service_transfers"), ("careunit_change_count", "num_careunit_transfers")]:
    if source_col in df.columns:
        df[new_col] = numeric_feature(source_col)
        care_fragmentation_engineered_cols.append(new_col)
if {"service_change_count", "careunit_change_count"}.issubset(df.columns):
    df["service_or_careunit_change_flag"] = ((numeric_feature("service_change_count") > 0) |
        (numeric_feature("careunit_change_count") > 0)).astype(int)
    care_fragmentation_engineered_cols.append("service_or_careunit_change_flag")
for col in ["num_unique_order_providers", "num_provider_order_changes", "num_care_team_touchpoints"]:
    if col in df.columns:
        care_fragmentation_engineered_cols.append(col)
# --- care-team fragmentation and late care-team activity features ---
if "num_care_team_touchpoints" in df.columns:
    care_touchpoint_threshold = numeric_feature("num_care_team_touchpoints")[numeric_feature("num_care_team_touchpoints") > 0].quantile(0.75)
    if pd.isna(care_touchpoint_threshold):
        care_touchpoint_threshold = 0
    df["high_care_team_touchpoints"] = (numeric_feature("num_care_team_touchpoints") >= care_touchpoint_threshold).astype(int)
    care_fragmentation_engineered_cols.append("high_care_team_touchpoints")

for col in ["psychosis_on_current_and_previous_admission_flag", "depression_on_current_and_previous_admission_flag",
        "substance_use_on_current_and_previous_admission_flag", "self_harm_recurrent_flag",
        "num_recurrent_psych_diagnosis_groups"]:
    if col in df.columns:
        diagnosis_persistence_engineered_cols.append(col)

if {"had_icu_stay", "high_procedure_burden"}.issubset(df.columns):
    df["icu_or_procedure_complexity_flag"] = ((numeric_feature("had_icu_stay") == 1) | (numeric_feature("high_procedure_burden") == 1)).astype(int)
    medical_complexity_interaction_cols.append("icu_or_procedure_complexity_flag")
if {"age_65plus", "previous_psych_admissions_365d"}.issubset(df.columns):
    df["older_age_x_no_recent_psych_history"] = ((numeric_feature("age_65plus") == 1) &
        (numeric_feature("previous_psych_admissions_365d") == 0)).astype(int)
    medical_complexity_interaction_cols.append("older_age_x_no_recent_psych_history")
if {"previous_psych_admissions_365d", "medical_complexity_score"}.issubset(df.columns):
    df["medical_complexity_without_prior_psych_flag"] = ((numeric_feature("previous_psych_admissions_365d") == 0) &
        (numeric_feature("medical_complexity_score") >= numeric_feature("medical_complexity_score").quantile(0.75))).astype(int)
    medical_complexity_interaction_cols.append("medical_complexity_without_prior_psych_flag")
if {"medical_complexity_score", "previous_psych_admissions_365d"}.issubset(df.columns):
    df["medical_complexity_without_psych_history_flag"] = ((numeric_feature("medical_complexity_score") >= numeric_feature("medical_complexity_score").quantile(0.75)) &
        (numeric_feature("previous_psych_admissions_365d") == 0)).astype(int)
    medical_complexity_interaction_cols.append("medical_complexity_without_psych_history_flag")
if {"had_icu_stay", "previous_psych_admissions_365d"}.issubset(df.columns):
    df["icu_complexity_without_psych_history_flag"] = ((numeric_feature("had_icu_stay") == 1) &
        (numeric_feature("previous_psych_admissions_365d") == 0)).astype(int)
    medical_complexity_interaction_cols.append("icu_complexity_without_psych_history_flag")
if {"num_abnormal_lab_flags_last_48h", "previous_psych_admissions_365d"}.issubset(df.columns):
    df["abnormal_labs_without_psych_history_flag"] = ((numeric_feature("num_abnormal_lab_flags_last_48h") > 0) &
        (numeric_feature("previous_psych_admissions_365d") == 0)).astype(int)
    medical_complexity_interaction_cols.append("abnormal_labs_without_psych_history_flag")
if {"medical_complexity_score", "num_psych_related_poe_orders"}.issubset(df.columns):
    df["high_medical_activity_low_psych_activity_flag"] = ((numeric_feature("medical_complexity_score") >= numeric_feature("medical_complexity_score").quantile(0.75)) &
        (numeric_feature("num_psych_related_poe_orders") == 0)).astype(int)
    medical_complexity_interaction_cols.append("high_medical_activity_low_psych_activity_flag")

for new_col, left_col, right_col in [
        ("no_recent_psych_history_and_high_charlson", "previous_psych_admissions_365d", "charlson_comorbidity_index_simplified"),
        ("no_recent_psych_history_and_long_ed_los", "previous_psych_admissions_365d", "high_ed_los_flag"),
        ("no_recent_psych_history_and_discharge_to_facility", "previous_psych_admissions_365d", "discharged_to_facility_flag")]:
    if left_col in df.columns and right_col in df.columns:
        if new_col == "no_recent_psych_history_and_high_charlson":
            df[new_col] = ((numeric_feature(left_col) == 0) & (numeric_feature(right_col) >= 2)).astype(int)
        else:
            df[new_col] = ((numeric_feature(left_col) == 0) & (numeric_feature(right_col) == 1)).astype(int)
        medical_complexity_interaction_cols.append(new_col)


#align duplicated naming from the SHAP review so the exported dataset is easy to audit
shap_theme_alias_cols = []
for new_col, source_col in [
        ("num_safety_observation_orders", "num_observation_safety_orders"),
        ("num_order_provider_changes", "num_provider_order_changes"),
        ("late_order_activity_and_high_poe_orders", "late_order_activity_x_high_poe_orders"),
        ("self_harm_and_previous_psych_365d", "prior_psych_365d_x_self_harm"),
        ("self_harm_and_previous_psych_90d", "prior_psych_90d_x_self_harm"),
        ("self_harm_and_psych_facility_discharge", "self_harm_x_discharged_to_psych_facility"),
        ("smi_and_previous_psych_365d", "prior_psych_365d_x_smi"),
        ("high_order_activity_and_self_harm", "high_order_activity_x_self_harm")]:
    if source_col in df.columns:
        df[new_col] = numeric_feature(source_col)
        shap_theme_alias_cols.append(new_col)
if "older_age_x_no_recent_psych_history" in df.columns:
    df["older_age_and_no_recent_psych_history"] = numeric_feature("older_age_x_no_recent_psych_history")
    shap_theme_alias_cols.append("older_age_and_no_recent_psych_history")

for col in ["bmi_change_recent", "weight_change_recent", "height_change_recent", "days_since_latest_body_measure"]:
    if col in df.columns:
        body_measure_trend_engineered_cols.append(col)
if {"underweight_flag", "elixhauser_weight_loss"}.issubset(df.columns):
    df["underweight_or_weight_loss_flag"] = ((numeric_feature("underweight_flag") == 1) |
        (numeric_feature("elixhauser_weight_loss") == 1)).astype(int)
    body_measure_trend_engineered_cols.append("underweight_or_weight_loss_flag")

pathway_refinement_engineered_cols = list(dict.fromkeys([col for col in pathway_refinement_engineered_cols if col in df.columns]))
care_fragmentation_engineered_cols = list(dict.fromkeys([col for col in care_fragmentation_engineered_cols if col in df.columns]))
diagnosis_persistence_engineered_cols = list(dict.fromkeys([col for col in diagnosis_persistence_engineered_cols if col in df.columns]))

# --- rapid-trial error-review additions: bounce-back, escalation, and unmanaged-risk features ---
#additional SHAP-guided features from the rapid-trial error review
#These are deliberately compact: they describe ED bounce-back, psychiatric/medical mixture,
#diagnosis escalation, discharge-friction, medication-readiness, prior adherence friction,
#and discharge timing context without adding raw IDs or high-cardinality text.
advanced_pathway_engineered_cols = []

def binary_feature(col_name):
    return (numeric_feature(col_name) > 0).astype(int)

#ED-only use is a proxy because MIMIC ED contact can overlap with admissions; subtracting admissions gives a conservative bounce-back estimate.
if {"previous_ed_visits_30d", "previous_total_admissions_30d"}.issubset(df.columns):
    df["previous_ed_visits_without_admission_30d"] = (numeric_feature("previous_ed_visits_30d") -
        numeric_feature("previous_total_admissions_30d")).clip(lower=0)
    advanced_pathway_engineered_cols.append("previous_ed_visits_without_admission_30d")
if {"previous_ed_visits_90d", "previous_total_admissions_90d"}.issubset(df.columns):
    df["previous_ed_visits_without_admission_90d"] = (numeric_feature("previous_ed_visits_90d") -
        numeric_feature("previous_total_admissions_90d")).clip(lower=0)
    df["prior_ed_only_high_use_flag"] = ((df["previous_ed_visits_without_admission_90d"] >= 2) &
        (numeric_feature("previous_total_admissions_90d") == 0)).astype(int)
    advanced_pathway_engineered_cols += ["previous_ed_visits_without_admission_90d", "prior_ed_only_high_use_flag"]
if {"previous_ed_visits_365d", "previous_total_admissions_365d"}.issubset(df.columns):
    df["ed_visit_to_admission_ratio_365d"] = (numeric_feature("previous_ed_visits_365d") /
        (numeric_feature("previous_total_admissions_365d") + 1)).replace([np.inf, -np.inf], 0).fillna(0)
    advanced_pathway_engineered_cols.append("ed_visit_to_admission_ratio_365d")

ed_admission_flag = binary_feature("emergency_room_admission_flag")
if "admission_location_grouped" in df.columns:
    ed_admission_flag = ((ed_admission_flag == 1) |
        df["admission_location_grouped"].astype(str).str.contains("emergency|ed", case=False, na=False)).astype(int)
transfer_admission_flag = ((binary_feature("transfer_from_hospital_flag") == 1) |
    (binary_feature("admitted_from_hospital_transfer_flag") == 1) |
    (binary_feature("admitted_from_facility_flag") == 1)).astype(int)
recent_psych_history_flag = ((numeric_feature("previous_psych_admissions_90d") > 0) |
    (numeric_feature("previous_psych_admissions_365d") > 0)).astype(int)
psych_history_flag = ((numeric_feature("previous_psych_admissions") > 0) |
    (numeric_feature("previous_psych_admissions_365d") > 0)).astype(int)
if "has_self_harm_or_suicidal_ideation" in df.columns:
    df["ed_admission_with_self_harm_flag"] = ed_admission_flag * binary_feature("has_self_harm_or_suicidal_ideation")
    advanced_pathway_engineered_cols.append("ed_admission_with_self_harm_flag")
if "has_psychotic_disorder" in df.columns:
    df["ed_admission_with_psychosis_flag"] = ed_admission_flag * binary_feature("has_psychotic_disorder")
    advanced_pathway_engineered_cols.append("ed_admission_with_psychosis_flag")
if "has_severe_mental_illness" in df.columns:
    df["ed_admission_with_smi_flag"] = ed_admission_flag * binary_feature("has_severe_mental_illness")
    advanced_pathway_engineered_cols.append("ed_admission_with_smi_flag")
df["emergency_admission_with_recent_psych_history_flag"] = ed_admission_flag * recent_psych_history_flag
df["transfer_admission_with_psych_history_flag"] = transfer_admission_flag * psych_history_flag
advanced_pathway_engineered_cols += ["emergency_admission_with_recent_psych_history_flag", "transfer_admission_with_psych_history_flag"]

high_charlson_flag = (numeric_feature("charlson_comorbidity_index_simplified") >= 3).astype(int)
if high_charlson_flag.sum() == 0:
    high_charlson_flag = (numeric_feature("charlson_comorbidity_index") >= 3).astype(int)
any_psych_flag = ((numeric_feature("psych_diagnosis_group_count") > 0) | (numeric_feature("num_psych_diagnoses") > 0) |
    (binary_feature("has_severe_mental_illness") == 1)).astype(int)
df["psych_and_medical_comorbidity_flag"] = any_psych_flag * high_charlson_flag
advanced_pathway_engineered_cols.append("psych_and_medical_comorbidity_flag")
for new_col, left_col, right_col in [
        ("psychosis_and_substance_use_flag", "has_psychotic_disorder", "has_substance_use"),
        ("depression_and_substance_use_flag", "has_depression", "has_substance_use"),
        ("self_harm_and_substance_use_flag", "has_self_harm_or_suicidal_ideation", "has_substance_use"),
        ("smi_and_high_charlson_flag", "has_severe_mental_illness", "charlson_comorbidity_index_simplified")]:
    if right_col == "charlson_comorbidity_index_simplified":
        df[new_col] = binary_feature(left_col) * high_charlson_flag
    else:
        df[new_col] = binary_feature(left_col) * binary_feature(right_col)
    advanced_pathway_engineered_cols.append(new_col)
if "discharged_to_psych_facility_flag" in df.columns:
    df["psych_diagnosis_without_psych_discharge_flag"] = (any_psych_flag * (1 - binary_feature("discharged_to_psych_facility_flag"))).astype(int)
    advanced_pathway_engineered_cols.append("psych_diagnosis_without_psych_discharge_flag")

#New/escalating psychiatric signal is different from long-standing recurrent psychiatric history.
df["new_self_harm_this_admission_flag"] = (binary_feature("has_self_harm_or_suicidal_ideation") *
    (1 - binary_feature("self_harm_recurrent_flag"))).astype(int)
df["new_psychosis_this_admission_flag"] = (binary_feature("has_psychotic_disorder") *
    (1 - binary_feature("psychosis_on_current_and_previous_admission_flag"))).astype(int)
df["new_substance_use_this_admission_flag"] = (binary_feature("has_substance_use") *
    (1 - binary_feature("substance_use_on_current_and_previous_admission_flag"))).astype(int)
df["new_smi_this_admission_flag"] = (binary_feature("has_severe_mental_illness") *
    (numeric_feature("previous_psych_admissions_365d") == 0).astype(int)).astype(int)
if {"psych_diagnosis_group_count", "num_recurrent_psych_diagnosis_groups"}.issubset(df.columns):
    df["psych_diagnosis_group_count_increased_from_previous_flag"] = (
        numeric_feature("psych_diagnosis_group_count") > numeric_feature("num_recurrent_psych_diagnosis_groups")).astype(int)
    df["current_psych_group_not_seen_previous_flag"] = ((numeric_feature("psych_diagnosis_group_count") > 0) &
        (numeric_feature("num_recurrent_psych_diagnosis_groups") == 0)).astype(int)
    advanced_pathway_engineered_cols += ["psych_diagnosis_group_count_increased_from_previous_flag", "current_psych_group_not_seen_previous_flag"]
advanced_pathway_engineered_cols += ["new_self_harm_this_admission_flag", "new_psychosis_this_admission_flag",
    "new_substance_use_this_admission_flag", "new_smi_this_admission_flag"]

#Late discontinuation and cancellation can represent unresolved plans, changed treatment, or discharge coordination friction.
if {"discontinued_orders_last_24h_before_discharge", "orders_last_24h_before_discharge"}.issubset(df.columns):
    df["late_order_discontinuation_share"] = (numeric_feature("discontinued_orders_last_24h_before_discharge") /
        numeric_feature("orders_last_24h_before_discharge").replace(0, np.nan)).replace([np.inf, -np.inf], 0).fillna(0)
    advanced_pathway_engineered_cols.append("late_order_discontinuation_share")

#Discharge support and medication-continuity features distinguish risky late activity from managed transitions.
followup_present_flag = ((binary_feature("had_followup_or_outpatient_referral_order") == 1) |
    (binary_feature("had_specific_psych_followup_order") == 1) | (binary_feature("had_outpatient_psychiatry_appointment_order") == 1) |
    (binary_feature("had_home_health_or_vna_order") == 1)).astype(int)
psych_followup_present_flag = ((binary_feature("had_psych_followup_order") == 1) |
    (binary_feature("had_specific_psych_followup_order") == 1) | (binary_feature("had_outpatient_psychiatry_appointment_order") == 1)).astype(int)
late_orders_flag = (numeric_feature("orders_last_12h_before_discharge") > 0).astype(int)
home_discharge_flag = binary_feature("discharged_home_flag")
psych_facility_discharge_flag = binary_feature("discharged_to_psych_facility_flag")
facility_discharge_flag = binary_feature("discharged_to_facility_flag")
df["late_orders_but_followup_present_flag"] = late_orders_flag * followup_present_flag
df["late_orders_but_discharge_planning_present_flag"] = late_orders_flag * binary_feature("had_discharge_planning_order")
df["late_orders_but_supervised_discharge_flag"] = late_orders_flag * ((psych_facility_discharge_flag == 1) | (facility_discharge_flag == 1)).astype(int)
df["home_discharge_with_followup_order_flag"] = home_discharge_flag * followup_present_flag
df["home_discharge_with_discharge_planning_order_flag"] = home_discharge_flag * binary_feature("had_discharge_planning_order")
df["psych_facility_discharge_with_followup_order_flag"] = psych_facility_discharge_flag * followup_present_flag
df["facility_discharge_with_discharge_planning_flag"] = facility_discharge_flag * binary_feature("had_discharge_planning_order")
df["self_harm_no_psych_followup_flag"] = binary_feature("has_self_harm_or_suicidal_ideation") * (1 - psych_followup_present_flag)
df["psychosis_home_discharge_no_followup_flag"] = binary_feature("has_psychotic_disorder") * home_discharge_flag * (1 - psych_followup_present_flag)
df["smi_home_discharge_no_followup_flag"] = binary_feature("has_severe_mental_illness") * home_discharge_flag * (1 - psych_followup_present_flag)
df["substance_use_home_discharge_no_followup_flag"] = binary_feature("has_substance_use") * home_discharge_flag * (1 - followup_present_flag)
df["prior_psych_admission_no_followup_flag"] = psych_history_flag * (1 - psych_followup_present_flag)
advanced_pathway_engineered_cols += ["late_orders_but_followup_present_flag", "late_orders_but_discharge_planning_present_flag",
    "late_orders_but_supervised_discharge_flag", "home_discharge_with_followup_order_flag",
    "home_discharge_with_discharge_planning_order_flag", "psych_facility_discharge_with_followup_order_flag",
    "facility_discharge_with_discharge_planning_flag", "self_harm_no_psych_followup_flag",
    "psychosis_home_discharge_no_followup_flag", "smi_home_discharge_no_followup_flag",
    "substance_use_home_discharge_no_followup_flag", "prior_psych_admission_no_followup_flag"]
if {"discharge_medication_order_flag", "has_psychotic_disorder", "had_antipsychotic"}.issubset(df.columns):
    df["no_psychotropic_discharge_med_in_psychosis_flag"] = (binary_feature("has_psychotic_disorder") *
        (1 - binary_feature("discharge_medication_order_flag")) * (1 - binary_feature("had_antipsychotic"))).astype(int)
    advanced_pathway_engineered_cols.append("no_psychotropic_discharge_med_in_psychosis_flag")

#Current-admission EMAR friction is retained only when explicitly extracted upstream.
#A true previous-admission EMAR profile would require a separate prior-hadm extraction, so it is not imputed here.
if "emar_not_given_last_24h_before_discharge_count" in df.columns:
    df["current_admission_high_emar_friction_flag"] = ((numeric_feature("emar_not_given_rate") >=
        numeric_feature("emar_not_given_rate").quantile(0.75)) & (numeric_feature("emar_not_given_rate") > 0)).astype(int)
    advanced_pathway_engineered_cols.append("current_admission_high_emar_friction_flag")

# --- readmission-history phenotype features ---
#Readmission-history phenotype separates psychiatric-cycling, medical-cycling, and rapid recurrent patterns.
prior_psych = numeric_feature("previous_psych_admissions_365d")
prior_total = numeric_feature("previous_total_admissions_365d")
prior_medical_proxy = (prior_total - prior_psych).clip(lower=0)
df["mostly_psych_readmission_pattern_flag"] = ((prior_psych >= 2) & (prior_psych >= prior_medical_proxy)).astype(int)
df["mostly_medical_readmission_pattern_flag"] = ((prior_medical_proxy >= 2) & (prior_medical_proxy > prior_psych)).astype(int)
df["mixed_psych_medical_readmission_pattern_flag"] = ((prior_psych > 0) & (prior_medical_proxy > 0)).astype(int)
df["rapid_recurrent_readmission_pattern_flag"] = ((prior_total >= 2) & (numeric_feature("mean_previous_admission_gap_days") <= 30)).astype(int)
df["long_gap_recurrent_readmission_pattern_flag"] = ((prior_total >= 2) & (numeric_feature("mean_previous_admission_gap_days") > 90)).astype(int)
advanced_pathway_engineered_cols += ["mostly_psych_readmission_pattern_flag", "mostly_medical_readmission_pattern_flag",
    "mixed_psych_medical_readmission_pattern_flag", "rapid_recurrent_readmission_pattern_flag",
    "long_gap_recurrent_readmission_pattern_flag"]

# --- calendar and discharge timing context ---
#Calendar/discharge timing features are cheap proxies for aftercare access and transition risk.
if "dischtime" in df.columns:
    discharge_time = pd.to_datetime(df["dischtime"], errors="coerce")
    df["holiday_period_discharge_flag"] = discharge_time.dt.strftime("%m-%d").isin(
        ["01-01", "12-24", "12-25", "12-26", "12-31", "07-04", "11-25", "11-26"]).fillna(False).astype(int)
    df["month_end_discharge_flag"] = (discharge_time.dt.day >= 28).fillna(False).astype(int)
    df["weekend_or_friday_discharge_with_no_followup_flag"] = ((discharge_time.dt.dayofweek.isin([4, 5, 6])).fillna(False).astype(int) *
        (1 - followup_present_flag)).astype(int)
    df["night_discharge_flag"] = discharge_time.dt.hour.isin(list(range(0, 6)) + list(range(20, 24))).fillna(False).astype(int)
    df["friday_evening_discharge_flag"] = ((discharge_time.dt.dayofweek == 4) & (discharge_time.dt.hour >= 17)).fillna(False).astype(int)
    df["late_discharge_hour_with_late_orders_flag"] = ((numeric_feature("discharge_hour") >= 17).astype(int) * late_orders_flag).astype(int)
    df["late_discharge_hour_with_no_followup_flag"] = ((numeric_feature("discharge_hour") >= 17).astype(int) * (1 - followup_present_flag)).astype(int)
    advanced_pathway_engineered_cols += ["holiday_period_discharge_flag", "month_end_discharge_flag",
        "weekend_or_friday_discharge_with_no_followup_flag", "night_discharge_flag", "friday_evening_discharge_flag",
        "late_discharge_hour_with_late_orders_flag", "late_discharge_hour_with_no_followup_flag"]

# --- near-discharge medical, vital, and EMAR instability summaries ---
#Near-discharge medical/vital/EMAR summaries are compact instability signals rather than many extra raw measurements.
if "num_abnormal_lab_flags_last_48h" in df.columns:
    df["abnormal_labs_last_24h_before_discharge"] = numeric_feature("num_abnormal_lab_flags_last_48h")
    df["abnormal_labs_last_12h_before_discharge"] = (numeric_feature("num_abnormal_lab_flags_last_48h") > 0).astype(int)
    df["new_abnormal_lab_last_24h_flag"] = (numeric_feature("num_abnormal_lab_flags_last_48h") > 0).astype(int)
    advanced_pathway_engineered_cols += ["abnormal_labs_last_24h_before_discharge", "abnormal_labs_last_12h_before_discharge",
        "new_abnormal_lab_last_24h_flag"]
if {"spo2_vital_last_value", "heart_rate_vital_last_value"}.intersection(df.columns):
    df["spo2_low_last_24h_before_discharge_flag"] = (numeric_feature("spo2_vital_last_value") < 92).astype(int)
    df["tachycardia_last_24h_before_discharge_flag"] = (numeric_feature("heart_rate_vital_last_value") > 110).astype(int)
    df["hypotension_last_24h_before_discharge_flag"] = ((numeric_feature("systolic_bp_arterial_vital_last_value") < 90) |
        (numeric_feature("mean_bp_arterial_vital_last_value") < 65)).astype(int)
    df["vital_instability_near_discharge_score"] = (df["spo2_low_last_24h_before_discharge_flag"] +
        df["tachycardia_last_24h_before_discharge_flag"] + df["hypotension_last_24h_before_discharge_flag"])
    df["vital_instability_home_discharge_flag"] = (df["vital_instability_near_discharge_score"] > 0).astype(int) * home_discharge_flag
    advanced_pathway_engineered_cols += ["spo2_low_last_24h_before_discharge_flag", "tachycardia_last_24h_before_discharge_flag",
        "hypotension_last_24h_before_discharge_flag", "vital_instability_near_discharge_score",
        "vital_instability_home_discharge_flag"]
if "emar_not_given_last_24h_before_discharge_count" in df.columns:
    df["emar_friction_near_discharge_score"] = (numeric_feature("emar_not_given_last_24h_before_discharge_count") +
        numeric_feature("emar_refusal_last_24h_before_discharge_flag") +
        numeric_feature("missed_psychotropic_last_24h_before_discharge_flag"))
    advanced_pathway_engineered_cols.append("emar_friction_near_discharge_score")

advanced_pathway_engineered_cols = list(dict.fromkeys([col for col in advanced_pathway_engineered_cols if col in df.columns]))



# --- late-order specificity and quiet-risk refinements ---
# Rapid-trial SHAP showed that true positives and false positives can both have late order activity.
# These features therefore describe what the late activity means, whether it is paired with protective
# aftercare, and whether a patient looks quiet in the record despite psychiatric or medical risk.
late_error_refinement_engineered_cols = []

def add_late_refinement_feature(col_name, values, as_flag=False):
    series = pd.Series(values, index=df.index) if not isinstance(values, pd.Series) else values.reindex(df.index)
    series = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    if as_flag:
        series = (series > 0).astype(int)
    df[col_name] = series
    late_error_refinement_engineered_cols.append(col_name)

late_category_12h_sources = {
    "medication": "late_medication_orders_last_12h",
    "lab": "late_lab_orders_last_12h",
    "imaging": "late_imaging_orders_last_12h",
    "consult": "late_consult_orders_last_12h",
    "psych_safety": "late_psych_safety_orders_last_12h",
    "discharge_admin": "late_discharge_admin_orders_last_12h",
    "followup_aftercare": "late_followup_or_aftercare_orders_last_12h",
    "social_work_case_management": "late_social_work_or_case_management_orders_last_12h",
    "routine_care": "late_routine_care_orders_last_12h"}
late_category_6h_sources = {name: col.replace("_last_12h", "_last_6h")
    for name, col in late_category_12h_sources.items()}

late_category_12h = pd.DataFrame({name: numeric_feature(col)
    for name, col in late_category_12h_sources.items() if col in df.columns}, index=df.index)
late_category_6h = pd.DataFrame({name: numeric_feature(col)
    for name, col in late_category_6h_sources.items() if col in df.columns}, index=df.index)

if not late_category_12h.empty:
    add_late_refinement_feature("late_order_type_diversity_last_12h", (late_category_12h > 0).sum(axis=1))
    clinical_12h_cols = [col for col in ["medication", "lab", "imaging", "consult", "psych_safety"] if col in late_category_12h]
    admin_12h_cols = [col for col in ["discharge_admin", "followup_aftercare", "social_work_case_management"] if col in late_category_12h]
    nonroutine_12h_cols = [col for col in clinical_12h_cols + admin_12h_cols if col in late_category_12h]
    if clinical_12h_cols:
        add_late_refinement_feature("late_clinical_order_count_last_12h", late_category_12h[clinical_12h_cols].sum(axis=1))
    if admin_12h_cols:
        add_late_refinement_feature("late_admin_aftercare_order_count_last_12h", late_category_12h[admin_12h_cols].sum(axis=1))
    if nonroutine_12h_cols:
        add_late_refinement_feature("late_nonroutine_order_count_last_12h", late_category_12h[nonroutine_12h_cols].sum(axis=1))

    late_severity_weights = {"psych_safety": 4.0, "consult": 3.0, "lab": 2.0, "imaging": 2.0,
        "medication": 2.0, "discharge_admin": 1.25, "followup_aftercare": 1.25,
        "social_work_case_management": 1.25, "routine_care": 0.5}
    late_order_severity_score = sum(late_category_12h[col] * weight
        for col, weight in late_severity_weights.items() if col in late_category_12h)
    add_late_refinement_feature("late_order_severity_score_last_12h", late_order_severity_score)
    if "orders_last_12h_before_discharge" in df.columns:
        late_order_denominator = numeric_feature("orders_last_12h_before_discharge").replace(0, np.nan)
        add_late_refinement_feature("late_order_severity_per_late_order_last_12h",
            (late_order_severity_score / late_order_denominator).fillna(0))
    intensity_components = late_order_severity_score
    if "care_team_touchpoints_last_12h_before_discharge" in df.columns:
        intensity_components = intensity_components + numeric_feature("care_team_touchpoints_last_12h_before_discharge")
    if "last_order_within_6h_of_discharge_flag" in df.columns:
        intensity_components = intensity_components + (2 * numeric_feature("last_order_within_6h_of_discharge_flag"))
    add_late_refinement_feature("late_order_intensity_score_last_12h", intensity_components)

if not late_category_6h.empty:
    add_late_refinement_feature("late_order_type_diversity_last_6h", (late_category_6h > 0).sum(axis=1))
    clinical_6h_cols = [col for col in ["medication", "lab", "imaging", "consult", "psych_safety"] if col in late_category_6h]
    if clinical_6h_cols:
        add_late_refinement_feature("late_clinical_order_count_last_6h", late_category_6h[clinical_6h_cols].sum(axis=1))

# Protective aftercare score: turns individual discharge/follow-up flags into a compact support signal.
aftercare_quality_sources = ["had_psych_followup_order", "had_outpatient_followup_order",
    "had_specific_psych_followup_order", "had_outpatient_psychiatry_appointment_order",
    "had_therapy_or_counselling_followup_order", "had_substance_use_followup_order",
    "had_case_management_followup_order", "had_social_work_aftercare_plan", "had_home_health_or_vna_order",
    "had_transport_arranged_order", "had_snf_or_rehab_placement_order", "had_discharge_planning_order",
    "discharge_planning_and_psych_followup_flag", "discharge_planning_and_social_work_flag"]
aftercare_quality_score = sum(numeric_feature(col) for col in aftercare_quality_sources if col in df.columns)
add_late_refinement_feature("aftercare_quality_score", aftercare_quality_score)
add_late_refinement_feature("protective_aftercare_present_flag", (df["aftercare_quality_score"] >= 2).astype(int))
add_late_refinement_feature("low_aftercare_quality_flag", (df["aftercare_quality_score"] == 0).astype(int))

late_activity_present = pd.Series(0, index=df.index, dtype=int)
for col in ["orders_last_12h_before_discharge", "orders_last_6h_before_discharge", "late_order_intensity_score_last_12h"]:
    if col in df.columns:
        late_activity_present = ((late_activity_present == 1) | (numeric_feature(col) > 0)).astype(int)
low_late_activity = pd.Series(1, index=df.index, dtype=int)
if "low_late_order_activity_flag" in df.columns:
    low_late_activity = numeric_feature("low_late_order_activity_flag").astype(int)
elif "orders_last_24h_before_discharge" in df.columns:
    low_late_activity = (numeric_feature("orders_last_24h_before_discharge") <= 1).astype(int)

home_discharge = numeric_feature("discharged_home_flag").astype(int) if "discharged_home_flag" in df.columns else pd.Series(0, index=df.index, dtype=int)
psych_facility_discharge = numeric_feature("discharged_to_psych_facility_flag").astype(int) if "discharged_to_psych_facility_flag" in df.columns else pd.Series(0, index=df.index, dtype=int)
followup_present = (df["aftercare_quality_score"] > 0).astype(int)
self_harm_present = numeric_feature("has_self_harm_or_suicidal_ideation").astype(int) if "has_self_harm_or_suicidal_ideation" in df.columns else pd.Series(0, index=df.index, dtype=int)
smi_present = numeric_feature("has_severe_mental_illness").astype(int) if "has_severe_mental_illness" in df.columns else pd.Series(0, index=df.index, dtype=int)
psychosis_present = pd.Series(0, index=df.index, dtype=int)
for col in ["has_psychotic_disorder", "elixhauser_psychoses"]:
    if col in df.columns:
        psychosis_present = ((psychosis_present == 1) | (numeric_feature(col) > 0)).astype(int)
prior_psych_history = pd.Series(0, index=df.index, dtype=int)
for col in ["previous_psych_admissions", "previous_psych_admissions_90d", "previous_psych_admissions_180d",
        "previous_psych_admissions_365d", "recent_psych_admission_365d"]:
    if col in df.columns:
        prior_psych_history = ((prior_psych_history == 1) | (numeric_feature(col) > 0)).astype(int)

add_late_refinement_feature("late_activity_with_low_aftercare_quality_flag",
    ((late_activity_present == 1) & (df["low_aftercare_quality_flag"] == 1)).astype(int))
add_late_refinement_feature("home_discharge_low_aftercare_quality_flag",
    ((home_discharge == 1) & (df["low_aftercare_quality_flag"] == 1)).astype(int))
add_late_refinement_feature("psychosis_or_self_harm_low_aftercare_quality_flag",
    (((psychosis_present == 1) | (self_harm_present == 1)) & (df["low_aftercare_quality_flag"] == 1)).astype(int))
add_late_refinement_feature("late_order_type_diversity_x_low_aftercare_flag",
    numeric_feature("late_order_type_diversity_last_12h") * numeric_feature("low_aftercare_quality_flag"))
add_late_refinement_feature("late_order_severity_x_low_aftercare_score",
    numeric_feature("late_order_severity_score_last_12h") * numeric_feature("low_aftercare_quality_flag"))

# Late psych-safety and medication-change features focus on unresolved acuity near discharge.
late_psych_safety_activity = pd.Series(0, index=df.index, dtype=float)
for col in ["late_psych_safety_orders_last_12h", "late_psych_safety_orders_last_6h",
        "safety_orders_last_24h_before_discharge", "new_safety_order_last_24h_before_discharge_flag",
        "self_harm_and_late_safety_order_flag"]:
    if col in df.columns:
        late_psych_safety_activity = late_psych_safety_activity + numeric_feature(col)
add_late_refinement_feature("late_psych_safety_activity_score_near_discharge", late_psych_safety_activity)
add_late_refinement_feature("late_psych_safety_without_followup_flag",
    ((late_psych_safety_activity > 0) & (followup_present == 0)).astype(int))
add_late_refinement_feature("late_psych_safety_home_discharge_flag",
    ((late_psych_safety_activity > 0) & (home_discharge == 1)).astype(int))
add_late_refinement_feature("late_psych_safety_without_psych_facility_discharge_flag",
    ((late_psych_safety_activity > 0) & (psych_facility_discharge == 0)).astype(int))

late_medication_change_activity = pd.Series(0, index=df.index, dtype=float)
for col in ["late_medication_orders_last_12h", "late_medication_orders_last_6h",
        "late_psychotropic_change_flag", "psychotropic_medication_change_count",
        "psychotropic_started_after_72h_flag", "new_psychotropic_last_24h_before_discharge_flag"]:
    if col in df.columns:
        late_medication_change_activity = late_medication_change_activity + numeric_feature(col)
add_late_refinement_feature("late_medication_change_intensity_score", late_medication_change_activity)
add_late_refinement_feature("late_medication_change_without_followup_flag",
    ((late_medication_change_activity > 0) & (followup_present == 0)).astype(int))
add_late_refinement_feature("late_medication_change_home_discharge_flag",
    ((late_medication_change_activity > 0) & (home_discharge == 1)).astype(int))

# Quiet-risk features target false negatives: patients with little late activity but clear psychiatric or medical risk.
add_late_refinement_feature("quiet_self_harm_low_activity_score",
    low_late_activity + self_harm_present + (1 - followup_present))
add_late_refinement_feature("quiet_smi_psychosis_low_activity_score",
    low_late_activity + ((smi_present == 1) | (psychosis_present == 1)).astype(int) + (1 - followup_present))
add_late_refinement_feature("quiet_prior_psych_low_activity_score",
    low_late_activity + prior_psych_history + (1 - followup_present))
medical_frailty_present = pd.Series(0, index=df.index, dtype=int)
for col in ["medical_complexity_without_prior_psych_flag", "medical_complexity_without_psych_history_flag"]:
    if col in df.columns:
        medical_frailty_present = ((medical_frailty_present == 1) | (numeric_feature(col) > 0)).astype(int)
if "charlson_comorbidity_index_simplified" in df.columns:
    medical_frailty_present = ((medical_frailty_present == 1) | (numeric_feature("charlson_comorbidity_index_simplified") >= 2)).astype(int)
if "anchor_age" in df.columns:
    medical_frailty_present = ((medical_frailty_present == 1) | (numeric_feature("anchor_age") >= 65)).astype(int)
add_late_refinement_feature("quiet_medical_frailty_low_psych_history_score",
    low_late_activity + medical_frailty_present + (1 - prior_psych_history))
quiet_risk_summary = (numeric_feature("quiet_self_harm_low_activity_score") +
    numeric_feature("quiet_smi_psychosis_low_activity_score") +
    numeric_feature("quiet_prior_psych_low_activity_score") +
    numeric_feature("quiet_medical_frailty_low_psych_history_score"))
add_late_refinement_feature("quiet_risk_summary_score", quiet_risk_summary)
add_late_refinement_feature("quiet_risk_flag", (quiet_risk_summary >= 6).astype(int))
add_late_refinement_feature("quiet_home_discharge_psych_risk_flag",
    ((home_discharge == 1) & (df["quiet_risk_flag"] == 1) &
        ((self_harm_present == 1) | (smi_present == 1) | (psychosis_present == 1) | (prior_psych_history == 1))).astype(int))

transport_or_placement_activity = pd.Series(0, index=df.index, dtype=int)
for col in ["had_transport_arranged_order", "transport_delay_order_flag", "placement_related_order_flag",
        "placement_related_orders_last_48h", "awaiting_bed_or_placement_flag"]:
    if col in df.columns:
        transport_or_placement_activity = ((transport_or_placement_activity == 1) | (numeric_feature(col) > 0)).astype(int)
add_late_refinement_feature("late_transport_or_placement_activity_flag",
    ((late_activity_present == 1) & (transport_or_placement_activity == 1)).astype(int))

late_error_refinement_engineered_cols = list(dict.fromkeys([col for col in late_error_refinement_engineered_cols if col in df.columns]))


# --- discharge-continuity and mixed-risk phenotype features ---
# These features are derived from signals already available during the index admission.
# They ask a practical clinical question: when psychiatric risk is present, is there also evidence
# of medication continuity, aftercare planning, safety management, or supervised discharge?
# The paired "without plan" flags target false negatives, while the planning scores can help separate
# high-risk-but-managed cases from high-risk-and-unmanaged cases.
discharge_continuity_engineered_cols = []
discharge_continuity_feature_data = {}

def add_discharge_continuity_feature(col_name, values, as_flag=False):
    series = pd.Series(values, index=df.index) if not isinstance(values, pd.Series) else values.reindex(df.index)
    series = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    if as_flag:
        series = (series > 0).astype(int)
    discharge_continuity_feature_data[col_name] = series
    discharge_continuity_engineered_cols.append(col_name)

psychosis_present = binary_feature("has_psychotic_disorder")
bipolar_present = binary_feature("has_bipolar_disorder")
depression_present = binary_feature("has_depression")
smi_present = binary_feature("has_severe_mental_illness")
self_harm_present = binary_feature("has_self_harm_or_suicidal_ideation")
substance_use_present = binary_feature("has_substance_use")
recent_psych_history_present = ((numeric_feature("recent_psych_admission_90d") > 0) |
    (numeric_feature("previous_psych_admissions_365d") > 0)).astype(int)

antipsychotic_present = ((binary_feature("had_antipsychotic") == 1) |
    (binary_feature("had_antipsychotic_order") == 1) |
    (binary_feature("had_long_acting_injectable_antipsychotic") == 1) |
    (binary_feature("new_antipsychotic_started_flag") == 1)).astype(int)
antidepressant_present = ((binary_feature("had_antidepressant") == 1) |
    (binary_feature("had_antidepressant_order") == 1) |
    (binary_feature("new_antidepressant_started_flag") == 1)).astype(int)
mood_stabiliser_present = ((binary_feature("had_mood_stabiliser") == 1) |
    (binary_feature("had_mood_stabiliser_order") == 1) |
    (binary_feature("new_mood_stabiliser_started_flag") == 1)).astype(int)
benzodiazepine_present = ((binary_feature("had_benzodiazepine") == 1) |
    (binary_feature("had_benzodiazepine_order") == 1)).astype(int)
psychotropic_present = ((antipsychotic_present == 1) | (antidepressant_present == 1) |
    (mood_stabiliser_present == 1) | (benzodiazepine_present == 1)).astype(int)

medication_plan_present = ((binary_feature("discharge_medication_order_flag") == 1) |
    (binary_feature("meds_to_beds_order_flag") == 1)).astype(int)
followup_plan_present = ((binary_feature("had_followup_or_outpatient_referral_order") == 1) |
    (binary_feature("had_psych_followup_order") == 1) |
    (binary_feature("had_case_management_followup_order") == 1) |
    (binary_feature("had_outpatient_psychiatry_appointment_order") == 1) |
    (binary_feature("had_substance_use_followup_order") == 1) |
    (binary_feature("protective_aftercare_present_flag") == 1)).astype(int)
safety_plan_present = ((binary_feature("num_observation_safety_orders") == 1) |
    (binary_feature("had_suicide_or_self_harm_precaution_order") == 1) |
    (binary_feature("had_sitter_order") == 1) |
    (binary_feature("had_1to1_observation_order") == 1) |
    (binary_feature("had_elopement_or_escape_risk_order") == 1)).astype(int)
supervised_discharge_present = ((binary_feature("discharged_to_psych_facility_flag") == 1) |
    (binary_feature("discharged_to_facility_flag") == 1) |
    (binary_feature("discharged_home_with_services_flag") == 1) |
    (binary_feature("had_home_health_or_vna_order") == 1) |
    (binary_feature("had_snf_or_rehab_placement_order") == 1)).astype(int)
home_discharge_present = ((binary_feature("discharged_home_flag") == 1) |
    (binary_feature("discharged_home_without_services_flag") == 1)).astype(int)
late_medication_change_present = ((binary_feature("late_psychotropic_change_flag") == 1) |
    (binary_feature("late_medication_orders_and_psychotropic_change_flag") == 1) |
    (binary_feature("new_psychotropic_last_24h_before_discharge_flag") == 1) |
    (numeric_feature("psychotropic_orders_last_24h_before_discharge") > 0)).astype(int)
withdrawal_or_substance_plan_present = ((binary_feature("had_withdrawal_treatment_order") == 1) |
    (binary_feature("had_alcohol_withdrawal_protocol_order") == 1) |
    (binary_feature("had_substance_use_followup_order") == 1) |
    (binary_feature("substance_use_and_withdrawal_treatment_flag") == 1)).astype(int)

add_discharge_continuity_feature("discharge_medication_continuity_score",
    medication_plan_present + psychotropic_present + antipsychotic_present + antidepressant_present + mood_stabiliser_present)
add_discharge_continuity_feature("psychiatric_aftercare_planning_score",
    followup_plan_present + safety_plan_present + supervised_discharge_present + medication_plan_present)
add_discharge_continuity_feature("unmanaged_psychiatric_discharge_risk_score",
    self_harm_present + smi_present + psychosis_present + recent_psych_history_present + home_discharge_present +
    (1 - followup_plan_present) + (1 - medication_plan_present) + late_medication_change_present)

add_discharge_continuity_feature("psychosis_without_antipsychotic_continuity_flag",
    ((psychosis_present == 1) & (antipsychotic_present == 0) & (medication_plan_present == 0)).astype(int), as_flag=True)
add_discharge_continuity_feature("bipolar_without_mood_stabiliser_continuity_flag",
    ((bipolar_present == 1) & (mood_stabiliser_present == 0) & (medication_plan_present == 0)).astype(int), as_flag=True)
add_discharge_continuity_feature("depression_without_antidepressant_continuity_flag",
    ((depression_present == 1) & (antidepressant_present == 0) & (medication_plan_present == 0)).astype(int), as_flag=True)
add_discharge_continuity_feature("smi_without_psychotropic_continuity_flag",
    ((smi_present == 1) & (psychotropic_present == 0) & (medication_plan_present == 0)).astype(int), as_flag=True)
add_discharge_continuity_feature("self_harm_without_aftercare_or_safety_plan_flag",
    ((self_harm_present == 1) & (followup_plan_present == 0) & (safety_plan_present == 0)).astype(int), as_flag=True)
add_discharge_continuity_feature("substance_use_without_followup_or_withdrawal_plan_flag",
    ((substance_use_present == 1) & (withdrawal_or_substance_plan_present == 0) & (followup_plan_present == 0)).astype(int), as_flag=True)
add_discharge_continuity_feature("late_medication_change_without_discharge_med_plan_flag",
    ((late_medication_change_present == 1) & (medication_plan_present == 0)).astype(int), as_flag=True)
add_discharge_continuity_feature("home_discharge_without_medication_or_followup_plan_flag",
    ((home_discharge_present == 1) & (medication_plan_present == 0) & (followup_plan_present == 0)).astype(int), as_flag=True)
add_discharge_continuity_feature("psychotropic_change_with_aftercare_plan_flag",
    ((late_medication_change_present == 1) & (followup_plan_present == 1)).astype(int), as_flag=True)
add_discharge_continuity_feature("psychotropic_change_without_aftercare_plan_flag",
    ((late_medication_change_present == 1) & (followup_plan_present == 0)).astype(int), as_flag=True)

prior_ed_only_present = ((numeric_feature("previous_ed_visits_without_admission_30d") > 0) |
    (numeric_feature("previous_ed_visits_without_admission_90d") >= 2) |
    (binary_feature("prior_ed_only_high_use_flag") == 1)).astype(int)
low_inpatient_psych_history = ((numeric_feature("previous_psych_admissions_365d") == 0) &
    (numeric_feature("recent_psych_admission_90d") == 0)).astype(int)
add_discharge_continuity_feature("prior_ed_only_use_with_current_self_harm_flag",
    ((prior_ed_only_present == 1) & (self_harm_present == 1)).astype(int), as_flag=True)
add_discharge_continuity_feature("prior_ed_only_use_with_current_smi_flag",
    ((prior_ed_only_present == 1) & (smi_present == 1)).astype(int), as_flag=True)
add_discharge_continuity_feature("prior_ed_only_use_with_current_psychosis_flag",
    ((prior_ed_only_present == 1) & (psychosis_present == 1)).astype(int), as_flag=True)
add_discharge_continuity_feature("prior_ed_only_use_with_substance_use_flag",
    ((prior_ed_only_present == 1) & (substance_use_present == 1)).astype(int), as_flag=True)
add_discharge_continuity_feature("ed_heavy_low_inpatient_psych_history_flag",
    ((prior_ed_only_present == 1) & (low_inpatient_psych_history == 1)).astype(int), as_flag=True)
add_discharge_continuity_feature("ed_heavy_home_discharge_no_followup_flag",
    ((prior_ed_only_present == 1) & (home_discharge_present == 1) & (followup_plan_present == 0)).astype(int), as_flag=True)
add_discharge_continuity_feature("ed_only_crisis_pattern_score",
    prior_ed_only_present + self_harm_present + smi_present + substance_use_present + low_inpatient_psych_history +
    (1 - followup_plan_present))

medical_instability_present = ((numeric_feature("medical_complexity_score") > 0) |
    (numeric_feature("older_medical_complexity_score") > 0) |
    (numeric_feature("vital_instability_near_discharge_score") > 0) |
    (numeric_feature("medical_workup_near_discharge_score") > 0) |
    (numeric_feature("num_lab_events_last_24h_before_discharge") > 0) |
    (binary_feature("had_icu_stay") == 1) |
    (binary_feature("icu_or_procedure_complexity_flag") == 1)).astype(int)
older_patient_present = ((numeric_feature("anchor_age") >= 65) | (binary_feature("age_65plus") == 1)).astype(int)
low_psych_marker_present = ((self_harm_present == 0) & (smi_present == 0) & (psychosis_present == 0) &
    (recent_psych_history_present == 0)).astype(int)
add_discharge_continuity_feature("older_medical_complexity_home_discharge_flag",
    ((older_patient_present == 1) & (medical_instability_present == 1) & (home_discharge_present == 1)).astype(int), as_flag=True)
add_discharge_continuity_feature("medical_complexity_no_aftercare_flag",
    ((medical_instability_present == 1) & (followup_plan_present == 0) & (supervised_discharge_present == 0)).astype(int), as_flag=True)
add_discharge_continuity_feature("medical_complexity_late_workup_home_discharge_flag",
    ((medical_instability_present == 1) & (home_discharge_present == 1) &
        ((numeric_feature("medical_workup_near_discharge_score") > 0) |
         (numeric_feature("num_lab_events_last_24h_before_discharge") > 0))).astype(int), as_flag=True)
add_discharge_continuity_feature("low_psych_marker_medical_instability_flag",
    ((low_psych_marker_present == 1) & (medical_instability_present == 1)).astype(int), as_flag=True)
add_discharge_continuity_feature("icu_or_procedure_without_psych_marker_flag",
    (((binary_feature("had_icu_stay") == 1) | (binary_feature("icu_or_procedure_complexity_flag") == 1)) &
        (low_psych_marker_present == 1)).astype(int), as_flag=True)

if discharge_continuity_feature_data:
    df = pd.concat([df, pd.DataFrame(discharge_continuity_feature_data, index=df.index)], axis=1)
discharge_continuity_engineered_cols = list(dict.fromkeys([col for col in discharge_continuity_engineered_cols if col in df.columns]))


# --- final bookkeeping: keep only columns actually present after conditional derivations ---
order_pattern_engineered_cols = list(dict.fromkeys([col for col in order_pattern_engineered_cols if col in df.columns]))
pharmacy_route_engineered_cols = list(dict.fromkeys([col for col in pharmacy_route_engineered_cols if col in df.columns]))
aftercare_engineered_cols = list(dict.fromkeys([col for col in aftercare_engineered_cols if col in df.columns]))
medical_complexity_interaction_cols = list(dict.fromkeys([col for col in medical_complexity_interaction_cols if col in df.columns]))
body_measure_trend_engineered_cols = list(dict.fromkeys([col for col in body_measure_trend_engineered_cols if col in df.columns]))
advanced_pathway_engineered_cols = list(dict.fromkeys([col for col in globals().get("advanced_pathway_engineered_cols", []) if col in df.columns]))
late_error_refinement_engineered_cols = list(dict.fromkeys([col for col in globals().get("late_error_refinement_engineered_cols", []) if col in df.columns]))
discharge_continuity_engineered_cols = list(dict.fromkeys([col for col in globals().get("discharge_continuity_engineered_cols", []) if col in df.columns]))
shap_theme_alias_cols = list(dict.fromkeys([col for col in shap_theme_alias_cols if col in df.columns]))

interaction_engineered_cols = list(dict.fromkeys(interaction_engineered_cols + shap_theme_alias_cols + pathway_refinement_engineered_cols +
    care_fragmentation_engineered_cols + diagnosis_persistence_engineered_cols + order_pattern_engineered_cols +
    aftercare_engineered_cols + pharmacy_route_engineered_cols + medical_complexity_interaction_cols +
    body_measure_trend_engineered_cols + advanced_pathway_engineered_cols + late_error_refinement_engineered_cols +
    discharge_continuity_engineered_cols))

print("Pathway refinement features available:", len(pathway_refinement_engineered_cols))
print("Care-fragmentation features available:", len(care_fragmentation_engineered_cols))
print("Diagnosis-persistence features available:", len(diagnosis_persistence_engineered_cols))
print("Order-pattern features created:", len(order_pattern_engineered_cols))
print("Aftercare features available:", len(aftercare_engineered_cols))
print("Pharmacy route/intensity features available:", len(pharmacy_route_engineered_cols))
print("Medical-complexity interaction features created:", len(medical_complexity_interaction_cols))
print("Body-measure trend features available:", len(body_measure_trend_engineered_cols))
print("Advanced pathway/aftercare/error-review features created:", len(advanced_pathway_engineered_cols))
print("Late-order specificity and quiet-risk features created:", len(late_error_refinement_engineered_cols))
print("Discharge-continuity and mixed-risk phenotype features created:", len(discharge_continuity_engineered_cols))
print("SHAP-theme alias features created:", len(shap_theme_alias_cols))


/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/2728080296.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["order_activity_ratio_24_to_72h"] = numeric_feature("orders_24_to_72h") / (numeric_feature("num_orders_first_24h") + 1)
/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_61165/2728080296.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["last_order_after_72h_flag"] = (numeric_feature("last_order_hours_from_admission") > 72).astype(int)
/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/

Pathway refinement features available: 13
Care-fragmentation features available: 15
Diagnosis-persistence features available: 5
Order-pattern features created: 39
Aftercare features available: 59
Pharmacy route/intensity features available: 25
Medical-complexity interaction features created: 18
Body-measure trend features available: 5
Advanced pathway/aftercare/error-review features created: 55
Late-order specificity and quiet-risk features created: 32
Discharge-continuity and mixed-risk phenotype features created: 25
SHAP-theme alias features created: 8


**Collect additional derived and upstream feature groups**

The upstream extracted feature groups and new T2.3-derived columns are collected so the final engineered dataset records exactly which additional variables were added. This also supports later feature-audit and reporting tables.

In [30]:
#summarise upstream extracted features plus T2.3-derived temporal burden and interaction features
additional_feature_groups = {
    "Prior utilisation and prior ICU": utilisation_engineered_cols,
    "ED timing": ed_timing_engineered_cols,
    "Admission/discharge timing": globals().get("admission_discharge_timing_engineered_cols", []),
    "ICD comorbidity and psychiatric subgroups": comorbidity_psych_icd_engineered_cols,
    "Additional ICD substance and Elixhauser flags": additional_icd_engineered_cols,
    "Admission/social/LOS proxies": globals().get("admission_social_los_engineered_cols", []),
    "Medication/infection proxies": infection_medication_engineered_cols,
    "Healthcare intensity rates": globals().get("healthcare_intensity_engineered_cols", []),
    "Order proportions": globals().get("order_proportion_engineered_cols", []),
    "POE order acuity and timing": globals().get("poe_order_engineered_cols", []),
    "Pathway refinement": globals().get("pathway_refinement_engineered_cols", []),
    "Order timing and aftercare patterns": globals().get("order_pattern_engineered_cols", []) + globals().get("aftercare_engineered_cols", []),
    "Diagnosis persistence": globals().get("diagnosis_persistence_engineered_cols", []),
    "Pharmacy medication intensity": globals().get("pharmacy_intensity_engineered_cols", []),
    "Pharmacy route and psychotropic intensity": globals().get("pharmacy_route_engineered_cols", []),
    "Care-team complexity": globals().get("care_team_engineered_cols", []) + globals().get("care_fragmentation_engineered_cols", []),
    "EMAR medication administration": emar_engineered_cols,
    "Procedure specificity": procedure_specificity_engineered_cols,
    "HCPCS event burden": globals().get("hcpcs_engineered_cols", []),
    "OMR body measures": body_measure_engineered_cols + globals().get("body_measure_trend_engineered_cols", []),
    "Medical complexity interactions": globals().get("medical_complexity_interaction_cols", []),
    "Service/transfer timing": transfer_timing_engineered_cols,
    "72-hour vital burden": globals().get("temporal_burden_engineered_cols", []),
    "Advanced pathway and aftercare interactions": globals().get("advanced_pathway_engineered_cols", []),
    "Late-order specificity and quiet-risk refinements": globals().get("late_error_refinement_engineered_cols", []),
    "Discharge continuity and mixed-risk phenotype": globals().get("discharge_continuity_engineered_cols", []),
    "Interaction terms": globals().get("interaction_engineered_cols", [])}

for group_name, cols in additional_feature_groups.items():
    present_cols = [col for col in cols if col in df.columns]
    print(group_name + ":", len(present_cols), "columns")

preview_cols = []
for cols in additional_feature_groups.values():
    preview_cols.extend([col for col in cols if col in df.columns][:4])
preview_cols = list(dict.fromkeys(preview_cols))[:30]
if preview_cols:
    print("\nAdditional feature preview:")
    print(df[id_cols + preview_cols].head())

Prior utilisation and prior ICU: 71 columns
ED timing: 10 columns
Admission/discharge timing: 14 columns
ICD comorbidity and psychiatric subgroups: 14 columns
Additional ICD substance and Elixhauser flags: 36 columns
Admission/social/LOS proxies: 35 columns
Medication/infection proxies: 28 columns
Healthcare intensity rates: 10 columns
Order proportions: 8 columns
POE order acuity and timing: 76 columns
Pathway refinement: 13 columns
Order timing and aftercare patterns: 98 columns
Diagnosis persistence: 5 columns
Pharmacy medication intensity: 47 columns
Pharmacy route and psychotropic intensity: 25 columns
Care-team complexity: 22 columns
EMAR medication administration: 41 columns
Procedure specificity: 8 columns
HCPCS event burden: 12 columns
OMR body measures: 36 columns
Medical complexity interactions: 18 columns
Service/transfer timing: 10 columns
72-hour vital burden: 14 columns
Advanced pathway and aftercare interactions: 55 columns
Late-order specificity and quiet-risk refineme

In [31]:
#collect engineered feature columns created in T2.3 by feature domain
admission_history_engineered_cols = ["previous_psych_admissions", "has_previous_psych_admission",
    "days_since_previous_psych_admission", "days_since_previous_psych_admission_filled",
    "time_since_previous_psych_category", "has_previous_hospital_admission",
    "has_previous_nonpsych_admission", "has_multiple_previous_hospital_admissions",
    "recent_psych_admission_30d", "recent_psych_admission_90d", "recent_psych_admission_365d"]

demographic_admission_engineered_cols = ["age_group", "hospital_los_category", "weekend_admission", 
    "admission_hour", "night_admission", "discharge_location_grouped", "discharged_to_death_or_hospice",
    "admission_location_grouped", "insurance_grouped"]

clinical_burden_engineered_cols = ["psych_diagnosis_group_count", "has_multiple_psych_diagnosis_groups",
    "psych_diagnosis_burden_category",  "nonpsych_diagnosis_burden_category", "total_diagnosis_burden_category",
    "has_severe_mental_illness", "drg_severity_category", "drg_mortality_category", "high_drg_severity",
    "high_drg_mortality", "service_transfer_category", "high_service_transfer_burden", "multiple_services",
    "physical_transfer_category", "high_physical_transfer_burden", "multiple_careunits", "had_careunit_transfer",
    "procedure_burden_category", "high_procedure_burden", "multiple_procedure_codes"]

medication_engineered_cols = ["has_any_psych_medication", "psych_medication_burden_category",
    "has_any_prescription", "polypharmacy_5plus", "polypharmacy_10plus", "unique_drug_burden_category",
    "antidepressant_antipsychotic_combination", "mood_stabiliser_antipsychotic_combination"]

laboratory_engineered_cols = ["num_labs_measured", "num_abnormal_labs", "any_abnormal_lab",
    "lab_abnormal_burden_category", "electrolyte_abnormality_count", "any_electrolyte_abnormality",
    "renal_lab_abnormality", "hematology_lab_abnormality", "num_lab_value_missing_indicators"]

#include dynamic laboratory features retained from T2.1/T2.2 if the lab extraction has been rerun
laboratory_dynamic_engineered_cols = [col for col in df.columns if "_lab_" in col and col.endswith((
    "_lab_last_value", "_lab_min_value", "_lab_max_value", "_lab_std_value", "_lab_count",
    "_lab_value_change", "_lab_value_range", "_lab_first_24h_mean", "_lab_first_24h_min",
    "_lab_first_24h_max", "_lab_first_24h_count", "_lab_first_72h_mean", "_lab_first_72h_min",
    "_lab_first_72h_max", "_lab_first_72h_count", "_lab_severe_abnormal"))]
laboratory_engineered_cols = list(dict.fromkeys(laboratory_engineered_cols + laboratory_dynamic_engineered_cols))

physiology_icu_engineered_cols = ["icu_utilisation_category", "multiple_icu_stays",
    "prolonged_icu_stay_3plus_days", "prolonged_icu_stay_7plus_days", "num_vital_types_measured",
    "num_vital_value_missing_indicators", "minimum_systolic_bp", "minimum_mean_bp",
    "tachycardia_indicator", "severe_tachycardia_indicator", "tachypnoea_indicator", "hypoxia_indicator",
    "hypotension_indicator", "physiological_instability_score", "physiological_instability_category"]

#temporal summary columns are created in T2.3.6 from the first 72 hours of hourly vital-sign data
if "temporal_summary_engineered_cols" not in globals():
    temporal_summary_engineered_cols = [col for col in df.columns
        if "_72h_" in col and (col.endswith("_mean") or col.endswith("_min")
            or col.endswith("_max") or col.endswith("_std") or col.endswith("_observed_hours"))]

#combine engineered feature groups into one list
engineered_feature_groups = {"Admission history": admission_history_engineered_cols,
    "Demographic and admission": demographic_admission_engineered_cols,
    "Clinical burden and severity": clinical_burden_engineered_cols,
    "Medication": medication_engineered_cols, "Laboratory": laboratory_engineered_cols,
    "Physiology and ICU": physiology_icu_engineered_cols,
    "72-hour temporal vital summaries": temporal_summary_engineered_cols}

#add extended feature groups when the additional T2.3 feature block has been run
optional_engineered_groups = {
    "Extended prior utilisation and prior ICU": globals().get("utilisation_engineered_cols", []),
    "ED timing": globals().get("ed_timing_engineered_cols", []),
    "Admission/discharge timing": globals().get("admission_discharge_timing_engineered_cols", []),
    "ICD comorbidity and psychiatric subgroups": globals().get("comorbidity_psych_icd_engineered_cols", []),
    "Additional ICD substance and Elixhauser flags": globals().get("additional_icd_engineered_cols", []),
    "Medication and infection proxies": globals().get("infection_medication_engineered_cols", []),
    "Healthcare intensity rates": globals().get("healthcare_intensity_engineered_cols", []),
    "Order proportions": globals().get("order_proportion_engineered_cols", []),
    "POE order acuity and timing": globals().get("poe_order_engineered_cols", []),
    "Pathway refinement": globals().get("pathway_refinement_engineered_cols", []),
    "Order timing and aftercare patterns": globals().get("order_pattern_engineered_cols", []) + globals().get("aftercare_engineered_cols", []),
    "Diagnosis persistence": globals().get("diagnosis_persistence_engineered_cols", []),
    "Pharmacy medication intensity": globals().get("pharmacy_intensity_engineered_cols", []),
    "Pharmacy route and psychotropic intensity": globals().get("pharmacy_route_engineered_cols", []),
    "Care-team complexity": globals().get("care_team_engineered_cols", []) + globals().get("care_fragmentation_engineered_cols", []),
    "EMAR medication administration": globals().get("emar_engineered_cols", []),
    "Procedure specificity": globals().get("procedure_specificity_engineered_cols", []),
    "HCPCS event burden": globals().get("hcpcs_engineered_cols", []),
    "OMR body measures": globals().get("body_measure_engineered_cols", []) + globals().get("body_measure_trend_engineered_cols", []),
    "Medical complexity interactions": globals().get("medical_complexity_interaction_cols", []),
    "Service and transfer timing": globals().get("transfer_timing_engineered_cols", []),
    "72-hour vital instability burden": globals().get("temporal_burden_engineered_cols", []),
    "Laboratory/vital dynamics": globals().get("lab_vital_dynamics_engineered_cols", []),
    "Advanced pathway and aftercare interactions": globals().get("advanced_pathway_engineered_cols", []),
    "Late-order specificity and quiet-risk refinements": globals().get("late_error_refinement_engineered_cols", []),
    "Discharge continuity and mixed-risk phenotype": globals().get("discharge_continuity_engineered_cols", []),
    "Clinical interaction terms": globals().get("interaction_engineered_cols", [])}

for group_name, cols in optional_engineered_groups.items():
    if len(cols) > 0:
        engineered_feature_groups[group_name] = cols

engineered_cols = []

for group_name, cols in engineered_feature_groups.items():
    present_cols = [col for col in cols if col in df.columns]
    engineered_feature_groups[group_name] = present_cols
    engineered_cols.extend(present_cols)

#remove accidental duplicates while preserving order
engineered_cols = list(dict.fromkeys(engineered_cols))
print("Engineered feature counts by domain:")
for group_name, cols in engineered_feature_groups.items():
    print(group_name + ":", len(cols))

print()
print("Total engineered features created:")
print(len(engineered_cols))
print()
print("Engineered feature columns:")
print(engineered_cols)

Engineered feature counts by domain:
Admission history: 11
Demographic and admission: 9
Clinical burden and severity: 20
Medication: 8
Laboratory: 137
Physiology and ICU: 15
72-hour temporal vital summaries: 35
Extended prior utilisation and prior ICU: 71
ED timing: 10
Admission/discharge timing: 14
ICD comorbidity and psychiatric subgroups: 14
Additional ICD substance and Elixhauser flags: 36
Medication and infection proxies: 28
Healthcare intensity rates: 10
Order proportions: 8
POE order acuity and timing: 76
Pathway refinement: 13
Order timing and aftercare patterns: 98
Diagnosis persistence: 5
Pharmacy medication intensity: 47
Pharmacy route and psychotropic intensity: 25
Care-team complexity: 22
EMAR medication administration: 41
Procedure specificity: 8
HCPCS event burden: 12
OMR body measures: 36
Medical complexity interactions: 18
Service and transfer timing: 10
72-hour vital instability burden: 14
Laboratory/vital dynamics: 53
Advanced pathway and aftercare interactions: 55
L

**Check final engineered dataset integrity**

The final engineered admission-level dataset is checked for duplicate admissions, remaining missing values in engineered features, and outcome label distribution before saving.



In [32]:
print('Dataset shape after feature engineering:',df.shape)
print('Duplicate subject_id + hadm_id rows:',df.duplicated(subset=['subject_id', 'hadm_id']).sum())
engineered_missing = df[engineered_cols].isna().sum().sort_values(ascending=False)
print('Missing values in engineered features:',engineered_missing[engineered_missing > 0])
print()
print('Readmission label distribution:')
print(df['readmitted_30d'].value_counts())
print()
print('Readmission label distribution (%):')
print((df['readmitted_30d'].value_counts(normalize=True) * 100).round(2))

Dataset shape after feature engineering: (238491, 1402)


Duplicate subject_id + hadm_id rows: 0
Missing values in engineered features: ed_to_ward_delay_hours                    234979
days_since_previous_icu_admission         189921
time_to_first_transfer_hours              186036
height_change_recent                      160681
previous_height_cm_before_admission       160681
bmi_change_recent                         134143
previous_bmi_before_admission             134143
time_between_last_two_admissions_days     128201
mean_previous_admission_gap_days          128201
min_previous_admission_gap_days           128201
std_previous_admission_gap_days           128201
weight_change_recent                      123210
previous_weight_kg_before_admission       123210
latest_height_cm_before_admission         111956
days_since_previous_psych_admission       107926
num_stopped_emar_events                   106633
num_started_emar_events                   106633
num_delayed_administered_events           106633
emar_total_delay_hours                  

**Preview engineered features**

A small subset of engineered variables is displayed to confirm that the final dataset contains the expected features before saving.



In [33]:
preview_cols = ['subject_id', 'hadm_id', 'anchor_age', 'age_group', 'hospital_los_days',
    'hospital_los_category', 'previous_psych_admissions', 'time_since_previous_psych_category',
    'psych_diagnosis_group_count', 'has_any_psych_medication', 'num_abnormal_labs',
    'physiological_instability_score', 'has_cannabis_related_disorder', 'elixhauser_individual_group_count', 'had_mechanical_ventilation_procedure', 'emar_administration_record_count', 'emar_missed_or_not_given_flag', 'readmitted_30d']

preview_cols = [col for col in preview_cols if col in df.columns]
print('Engineered feature preview:')
print(df[preview_cols].head(10))

Engineered feature preview:
   subject_id   hadm_id  anchor_age age_group  hospital_los_days  \
0    10000032  22595853          52     50-59           0.786111   
1    10000032  22841357          52     50-59           1.015278   
2    10000032  29079034          52     50-59           2.222222   
3    10000032  25742920          52     50-59           1.754167   
4    10000068  25022803          19     18-29           0.298611   
5    10000084  23052089          72     70-79           4.538889   
6    10000084  29888819          72     70-79           0.455556   
7    10000117  22927623          48     40-49           0.532639   
8    10000117  27988844          48     40-49           2.930556   
9    10000690  23280645          86       80+           7.751389   

  hospital_los_category  previous_psych_admissions  \
0              0-1 days                          0   
1              1-3 days                          1   
2              1-3 days                          2   
3      

**Save engineered admission-level dataset**

The final engineered admission-level dataset is saved for T2.4 exploratory analysis and WP3 model development. The save operation is kept separate from the final print checks to reduce notebook memory pressure.



In [34]:
#remove any partial output file before trying again
output_file = output_path / 't2_3_engineered_dataset.csv'
if output_file.exists():
    output_file.unlink()
    print("Removed partial output file:", output_file)

Removed partial output file: /Users/ahthini/Desktop/DissProject/outputs/t2_3_engineered_dataset.csv


In [35]:
#save engineered admission-level dataset for later tasks
df.to_csv(output_file, index=False, chunksize=50000)
print('Saved engineered admission-level dataset to:')
print(output_file)
print('Final engineered admission-level dataset shape:')
print(df.shape)
print()
print("Saved time-series hourly window dataset to:")
print(hourly_window_file)
print("Final hourly vital window dataset shape:", hourly_vital_windows.shape)
print("Unique admissions in hourly window dataset:", hourly_vital_windows["hadm_id"].nunique())

Saved engineered admission-level dataset to:
/Users/ahthini/Desktop/DissProject/outputs/t2_3_engineered_dataset.csv
Final engineered admission-level dataset shape:
(238491, 1402)

Saved time-series hourly window dataset to:
/Users/ahthini/Desktop/DissProject/outputs/t2_3_hourly_vital_window_dataset.csv
Final hourly vital window dataset shape: (3990003, 11)
Unique admissions in hourly window dataset: 40466


In [36]:
#summarise final T2.3 outputs for dissertation reporting
t2_3_output_summary = pd.DataFrame({"output_dataset": ["Admission-level engineered dataset",
        "Hourly vital-sign time-series dataset"],
    "file_name": ["t2_3_engineered_dataset.csv", "t2_3_hourly_vital_window_dataset.csv"],
    "unit_of_analysis": ["One row per hospital admission", "One row per admission-hour"],
    "rows": [df.shape[0], hourly_vital_windows.shape[0]],
    "columns": [df.shape[1], hourly_vital_windows.shape[1]],
    "unique_patients": [df["subject_id"].nunique(), hourly_vital_windows["subject_id"].nunique()],
    "unique_admissions": [df["hadm_id"].nunique(), hourly_vital_windows["hadm_id"].nunique()],
    "intended_use": ["Main admission-level machine learning models with flattened temporal summaries",
        "Future sequential or hybrid time-series modelling"]})

print("Final T2.3 output summary:")
print(t2_3_output_summary)

Final T2.3 output summary:
                          output_dataset  \
0     Admission-level engineered dataset   
1  Hourly vital-sign time-series dataset   

                              file_name                unit_of_analysis  \
0           t2_3_engineered_dataset.csv  One row per hospital admission   
1  t2_3_hourly_vital_window_dataset.csv      One row per admission-hour   

      rows  columns  unique_patients  unique_admissions  \
0   238491     1402           107926             238491   
1  3990003       11            32185              40466   

                                        intended_use  
0  Main admission-level machine learning models w...  
1  Future sequential or hybrid time-series modelling  
